# Step 1: Entry Legitimacy Assessment

Classifies each row from OCR-segmented ASCC catalog input as one of:

- **listing** -- a real catalog entry; proceed to field extraction
- **cross_reference** -- a real catalog row that redirects to another section; no extractable postmark data
- **non_entry** -- annotation, fragment, or page artifact mistakenly included; flag for review

Assessment is purely text-structure-based. Section-context columns (Default Shape, etc.) are ignored at this stage.

### Signals

Evaluated in order:

1. **Relationship indicator prefix** -- starts with Same, (L), (E), or asterisked variants. Immediate listing.
2. **Cross-reference pattern** -- contains "See [Place/Section]" without a semicolon-parenthetical. Cross-reference.
3. **Fragment detection** -- starts with unmatched closing paren or lowercase mid-sentence content. Non-entry.
4. **Trailing valuation** -- ends with a number (e.g. `1500.00`, `1,500`), `--`, or `---`. Nearly all legitimate entries have this.
5. **Core structural anatomy** -- has at least one of: semicolon-parenthetical, four-digit year (1700-1869), decade ref (e.g. `1850's`), or c-prefixed year.

Final classification:
- Signal 1 hit -> listing (auto)
- Signal 2 hit -> cross_reference
- Signal 3 hit -> non_entry
- Signal 4 AND Signal 5 -> listing
- Signal 4 only OR Signal 5 only -> listing (low confidence, flagged for review)
- Neither Signal 4 nor Signal 5 -> non_entry

## 0. Setup

In [1]:
import pandas as pd
import re
import os

INPUT_CSV = './wip/in/VA_ASCC_CTLG.csv'  # <-- Change per state
INPUT_DIR = './wip/in/'
OUT_DIR   = './wip/out/'

# Region abbrev is derived from the first two characters of the input
# filename (e.g. VA_ASCC_CTLG.csv -> VA), then cross-checked against the
# regions seed CSV. The check fails loudly if the prefix is unknown.
REGION_ABBREV = os.path.basename(INPUT_CSV)[:2].upper()

# Reference work seed: single-row CSV with id, code, etc. The bundle's
# RW_CODE drives the marking and cover code prefixes.
_rw_seed = pd.read_csv(os.path.join(INPUT_DIR, 'reference_works.csv'))
if len(_rw_seed) != 1:
    raise ValueError(
        f"reference_works.csv must contain exactly 1 row (got {len(_rw_seed)})."
    )
RW_ID   = int(_rw_seed.iloc[0]['id'])
RW_CODE = str(_rw_seed.iloc[0]['code']).strip()

# Region seed: full Django-shape regions table. Used to resolve REGION_ABBREV
# to its REGION_ID for FK columns in markings/post_offices/etc.
_region_seed = pd.read_csv(os.path.join(INPUT_DIR, 'regions.csv'))
_match = _region_seed[_region_seed['abbrev'].astype(str).str.upper() == REGION_ABBREV]
if len(_match) != 1:
    raise ValueError(
        f"REGION_ABBREV={REGION_ABBREV!r} must match exactly 1 row in regions.csv "
        f"(matched {len(_match)}). Adjust INPUT_CSV or regions.csv."
    )
REGION_ID = int(_match.iloc[0]['id'])

print(f"rw_code={RW_CODE} (id={RW_ID})  region={REGION_ABBREV} (id={REGION_ID})")

df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df)} rows from {INPUT_CSV}')
print(f'Columns: {list(df.columns)}')

# Required columns from the new 5-column OCR CSV format produced by
# tools/ascc_page_extract.py. Chunk and Type are new; the notebook
# previously inferred Type via its own heuristic classifier (Step 1),
# but the upstream extractor now provides it for the coarse META/LISTING
# split. Step 1 still sub-classifies within LISTING rows.
REQUIRED_COLS = ['Listing', 'Page', 'Chunk', 'Images Above', 'Type']
# Optional columns: consumed if present, ignored otherwise.
OPTIONAL_COLS = ['Manuscript', 'Default Shape', 'Institutional Ownership']

missing_required = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_required:
    raise ValueError(
        f'Required columns missing from {INPUT_CSV}: {missing_required}'
    )

# Split on the upstream Type column. META rows (section headers, column
# headers, editorial notes, state running-heads) are passed to the stub
# below and excluded from all downstream processing.
meta_df = df[df['Type'] == 'META'].reset_index(drop=True)
listings_df = df[df['Type'] == 'LISTING'].reset_index(drop=True)


def process_meta_rows(meta_df):
    # TODO: parse META rows for section-heading / state-heading context,
    # column headers, and cross-reference targets. Inputs: meta_df with
    # columns Listing, Page, Chunk, Images Above, Type. Currently a no-op.
    return None


process_meta_rows(meta_df)

# All downstream cells operate on LISTING rows only.
df = listings_df

print(f'  META rows:    {len(meta_df)}')
print(f'  LISTING rows: {len(df)} (downstream)')

present_optional = [c for c in OPTIONAL_COLS if c in df.columns]
absent_optional = [c for c in OPTIONAL_COLS if c not in df.columns]
print(f'Optional columns present: {present_optional}')
if absent_optional:
    print(f'Optional columns absent (fallback behavior): {absent_optional}')


rw_code=APMC (id=1)  region=VA (id=51)
Loaded 1595 rows from ./wip/in/VA_ASCC_CTLG.csv
Columns: ['Listing', 'Page', 'Chunk', 'Images Above', 'Type', 'Manuscript', 'Default Shape']
  META rows:    56
  LISTING rows: 1539 (downstream)
Optional columns present: ['Manuscript', 'Default Shape']
Optional columns absent (fallback behavior): ['Institutional Ownership']


## 1. Preprocessing

Strip dot leaders before signal detection. These are OCR artifacts from the catalog's
visual formatting (dots connecting the entry text to the price column) and carry no
semantic content.

In [2]:
def strip_dot_leaders(text):
    """Remove runs of 2+ dots (dot leaders) and collapse resulting whitespace."""
    t = re.sub(r'\.{2,}', ' ', str(text))
    return re.sub(r'  +', ' ', t).strip()

df['clean_text'] = df['Listing'].apply(strip_dot_leaders)

# Quick check
print('Sample cleaned entries:')
for t in df['clean_text'].head(5):
    print(f'  {t}')

Sample cleaned entries:
  Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) 1500.00
  (L)(Sept.15,1774) 1000.00
  Colchester(June 30,1774;Ms;Black) 1500.00
  Fredericksburg(May 10,1755;Ms;Black) 2000.00
  Fredg.(Aug.2,1772;Ms;Black) 1500.00


## 2. Signal Detection

### Signal 1: Relationship Indicator Prefix

Entry starts with Same, (L), (E), or their asterisked/plus-prefixed variants.
Leading `*` on its own (marking first-of-town) is NOT a relationship indicator --
it only counts when combined with `(L)` or `(E)`.

In [3]:
RELATIONSHIP_PATTERN = re.compile(
    r'^\+?'
    r'(?:'
    r'Same'
    r'|\*?[(\[{]L[)\]}]\*?'
    r'|\*?[(\[{]E[)\]}]\*?'
    r')',
    re.IGNORECASE
)

df['s1_relationship'] = df['clean_text'].apply(
    lambda t: bool(RELATIONSHIP_PATTERN.match(t))
)

print(f'Signal 1 hits: {df["s1_relationship"].sum()}')
print()
print('Examples:')
for t in df.loc[df['s1_relationship'], 'clean_text'].head(8):
    print(f'  {t[:100]}')

Signal 1 hits: 260

Examples:
  (L)(Sept.15,1774) 1000.00
  (L)(June 27,1775) 1000.00
  (L)(Dec.6,1765) 1200.00
  Same(--,1773;Ms;Black) 1000.00
  (L)(Oct.2,1775) 1200.00
  Same(May 6,1775;MDD same line) 1500.00
  Same(July 25,1775;MDD below;Red) 1500.00
  Same(July 25,1775;Black) 1200.00


### Signal 2: Cross-Reference

Contains "See" followed by a section/place name, AND lacks a semicolon-parenthetical
(which would indicate it is a real listing that happens to also mention a cross-reference).

In [4]:
def detect_cross_reference(text):
    """True if the entry is a pure cross-reference (See X, no semicolon-parenthetical)."""
    has_see = bool(re.search(r'\bSee\b', text))
    has_semicolon_paren = bool(re.search(r'\([^)]*;[^)]*\)', text))
    return has_see and not has_semicolon_paren

df['s2_cross_ref'] = df['clean_text'].apply(detect_cross_reference)

print(f'Signal 2 hits: {df["s2_cross_ref"].sum()}')
print()
for t in df.loc[df['s2_cross_ref'], 'clean_text']:
    print(f'  {t[:120]}')

Signal 2 hits: 7

  (L) See State --
  FREDERICKSBURG(backstamp)(E)See Colonial listing --
  (L) See State --
  (L) See State --
  (L) See State --
  (L) See State --
  ROCKBRIDGE,ALUM SPRINGS,VA.(See Alum Springs) --


### Signal 3: Fragment Detection

Two sub-checks:
- Starts with an unmatched closing paren fragment (`)` appears before any `(`)
- Starts with lowercase mid-sentence content (segmentation split mid-entry)

In [5]:
def detect_fragment(text):
    """True if the entry looks like a segmentation fragment."""
    t = text.strip()
    if not t:
        return True
    # Unmatched closing paren: ')' appears before any '('
    first_open = t.find('(')
    first_close = t.find(')')
    if first_close != -1 and (first_open == -1 or first_close < first_open):
        return True
    # Starts with lowercase (mid-sentence fragment)
    # But exclude known patterns: 'c1850' (c-date), 'la.' (abbreviation)
    if t[0].islower() and not re.match(r'^c\d{4}', t):
        return True
    return False

df['s3_fragment'] = df['clean_text'].apply(detect_fragment)

print(f'Signal 3 hits: {df["s3_fragment"].sum()}')
print()
for t in df.loc[df['s3_fragment'], 'clean_text']:
    print(f'  {repr(t[:120])}')

Signal 3 hits: 1

  'wmfbURG V.(long"s")(1807;SL-39x5,MDD;Black) 250.00'


### Signal 4: Trailing Valuation

Entry ends with a recognizable value token: a decimal number (e.g. `3500.00`),
a comma-formatted integer (e.g. `1,500`), a plain integer, `--`, or `---`.

In [6]:
TRAILING_VALUE_PATTERN = re.compile(
    r'(?:'
    r'\d[\d,]*(?:\.\d+)?'  # number: 3500.00 or 1,500 or 50
    r'|---?'                # dashes: -- or ---
    r')\s*$'
)

df['s4_trailing_value'] = df['clean_text'].apply(
    lambda t: bool(TRAILING_VALUE_PATTERN.search(t))
)

print(f'Signal 4 hits: {df["s4_trailing_value"].sum()} / {len(df)}')
print()
misses = df[~df['s4_trailing_value']]
if len(misses):
    print('Entries WITHOUT trailing value:')
    for t in misses['clean_text']:
        print(f'  {repr(t[:120])}')
else:
    print('All entries have a trailing value.')

Signal 4 hits: 1539 / 1539

All entries have a trailing value.


### Signal 5: Core Structural Anatomy

At least one of:
- Parenthetical with semicolons: `(stuff;stuff)`
- Four-digit year in range 1700-1869
- Decade reference: `1850's`, `1850s`
- c-prefixed year: `c1850`

In [7]:
def detect_structural_anatomy(text):
    """Returns a dict of which structural sub-signals are present."""
    result = {
        'semicolon_paren': bool(re.search(r'\([^)]*;[^)]*\)', text)),
        'four_digit_year': bool(re.search(r'\b1[78]\d{2}\b', text)),
        'decade_ref': bool(re.search(r"1[78]\d0['\'s]", text)),
        'c_year': bool(re.search(r'\bc1[78]\d{2}\b', text, re.IGNORECASE)),
    }
    result['any'] = any(result.values())
    return result

anatomy = df['clean_text'].apply(detect_structural_anatomy).apply(pd.Series)
df['s5_semicolon_paren'] = anatomy['semicolon_paren']
df['s5_four_digit_year'] = anatomy['four_digit_year']
df['s5_decade_ref'] = anatomy['decade_ref']
df['s5_c_year'] = anatomy['c_year']
df['s5_anatomy'] = anatomy['any']

print(f'Signal 5 hits: {df["s5_anatomy"].sum()} / {len(df)}')
print()
print('Sub-signal breakdown:')
print(f'  Semicolon-parenthetical: {df["s5_semicolon_paren"].sum()}')
print(f'  Four-digit year:         {df["s5_four_digit_year"].sum()}')
print(f'  Decade reference:        {df["s5_decade_ref"].sum()}')
print(f'  c-prefixed year:         {df["s5_c_year"].sum()}')
print()
misses = df[~df['s5_anatomy']]
if len(misses):
    print(f'Entries WITHOUT structural anatomy ({len(misses)}):')
    for t in misses['clean_text']:
        print(f'  {repr(t[:120])}')
else:
    print('All entries have structural anatomy.')

Signal 5 hits: 1495 / 1539

Sub-signal breakdown:
  Semicolon-parenthetical: 706
  Four-digit year:         1464
  Decade reference:        61
  c-prefixed year:         0

Entries WITHOUT structural anatomy (44):
  '(L) See State --'
  'FREDERICKSBURG(backstamp)(E)See Colonial listing --'
  '(L) See State --'
  '(L) See State --'
  '(L) See State --'
  '(L) See State --'
  'Same(Green) 30.00'
  'Same(Green) 35.00'
  'Same(Green) 30.00'
  'Same(Green) 50.00'
  'Same(Green) 40.00'
  'Same(Green) 60.00'
  'Same(Green) 25.00'
  'Same(X/Cts[C-24,green],PAID/5/Cts[DC-28,black]) 75.00'
  'Same(Green) 25.00'
  'Same(Green) 40.00'
  'Same(Green) 20.00'
  'Same(Green) 20.00'
  'ROCKBRIDGE,ALUM SPRINGS,VA.(See Alum Springs) --'
  'Same(Green) 60.00'
  'Same(vivid Blue ink) 60.00'
  'Same(Green) 45.00'
  "Barry's Bridge -- --"
  'Burgess Store -- 30.00'
  "Cooper's -- 15.00"
  'Drapersville -- --'
  '(1)Egypt -- 50.00'
  'Fishersborough -- 25.00'
  "Grayson's Sul Spgs -- --"
  'Hail Stone -- --'


## 3. Classification

In [8]:
def classify_entry(row):
    """Apply signals in priority order. Returns (classification, confidence, reason)."""

    # Signal 1: relationship indicator -> auto listing
    if row['s1_relationship']:
        return 'listing', 'high', 'relationship_indicator'

    # Signal 2: cross-reference
    if row['s2_cross_ref']:
        return 'cross_reference', 'high', 'see_pattern'

    # Signals 4+5 conjunction
    has_value = row['s4_trailing_value']
    has_anatomy = row['s5_anatomy']

    # Signal 3: fragment -- only reject if the row lacks strong listing signals.
    # A lowercase-initial entry with both trailing value and full anatomy is a
    # stylistic catalog entry (e.g. 'wmfbURG'), not a segmentation artifact.
    if row['s3_fragment']:
        if has_value and has_anatomy:
            return 'listing', 'medium', 'fragment_with_anatomy'
        return 'non_entry', 'high', 'fragment'

    if has_value and has_anatomy:
        return 'listing', 'high', 'value_and_anatomy'
    elif has_value and not has_anatomy:
        return 'listing', 'low', 'value_only'
    elif has_anatomy and not has_value:
        return 'listing', 'low', 'anatomy_only'
    else:
        return 'non_entry', 'high', 'no_signals'

classifications = df.apply(classify_entry, axis=1, result_type='expand')
df['classification'] = classifications[0]
df['confidence'] = classifications[1]
df['reason'] = classifications[2]

print('Classification results:')
print(df['classification'].value_counts())
print()
print('By reason:')
print(df.groupby(['classification', 'confidence', 'reason']).size().reset_index(name='count').to_string(index=False))


Classification results:


classification
listing            1537
cross_reference       2
Name: count, dtype: int64

By reason:
 classification confidence                 reason  count
cross_reference       high            see_pattern      2
        listing       high relationship_indicator    260
        listing       high      value_and_anatomy   1254
        listing        low             value_only     22
        listing     medium  fragment_with_anatomy      1


## 4. Review: Low-Confidence and Non-Listings

These are the entries that need human eyes.

In [9]:
review = df[(df['confidence'] == 'low') | (df['classification'] != 'listing')].copy()
print(f'Entries requiring review: {len(review)} / {len(df)}')
print()

for _, row in review.iterrows():
    print(f'[{row["classification"]}] ({row["confidence"]}, {row["reason"]})')
    print(f'  {row["clean_text"][:140]}')
    print()

Entries requiring review: 24 / 1539

[cross_reference] (high, see_pattern)
  FREDERICKSBURG(backstamp)(E)See Colonial listing --

[cross_reference] (high, see_pattern)
  ROCKBRIDGE,ALUM SPRINGS,VA.(See Alum Springs) --

[listing] (low, value_only)
  Barry's Bridge -- --

[listing] (low, value_only)
  Burgess Store -- 30.00

[listing] (low, value_only)
  Cooper's -- 15.00

[listing] (low, value_only)
  Drapersville -- --

[listing] (low, value_only)
  (1)Egypt -- 50.00

[listing] (low, value_only)
  Fishersborough -- 25.00

[listing] (low, value_only)
  Grayson's Sul Spgs -- --

[listing] (low, value_only)
  Hail Stone -- --

[listing] (low, value_only)
  Hoysville -- --

[listing] (low, value_only)
  King William Ch -- 25.00

[listing] (low, value_only)
  Mechum's River -- 30.00

[listing] (low, value_only)
  Nelson Sta -- --

[listing] (low, value_only)
  Oak Grove 20.00

[listing] (low, value_only)
  Peytonsburgh -- 30.00

[listing] (low, value_only)
  Powells Tavern -- 30.00

[listi

## 5. Signal Summary

Full signal report for every entry.

In [10]:
report_cols = [
    'clean_text',
    's1_relationship',
    's2_cross_ref',
    's3_fragment',
    's4_trailing_value',
    's5_anatomy',
    'classification',
    'confidence',
    'reason',
]

# Truncate text for display
report = df[report_cols].copy()
report['clean_text'] = report['clean_text'].str[:80]

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 82)
pd.set_option('display.width', 200)
report

,clean_text,s1_relationship,s2_cross_ref,s3_fragment,s4_trailing_value,s5_anatomy,classification,confidence,reason
0,"Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) 1500.00",False,False,False,True,True,listing,high,value_and_anatomy
1,"(L)(Sept.15,1774) 1000.00",True,False,False,True,True,listing,high,relationship_indicator
2,"Colchester(June 30,1774;Ms;Black) 1500.00",False,False,False,True,True,listing,high,value_and_anatomy
3,"Fredericksburg(May 10,1755;Ms;Black) 2000.00",False,False,False,True,True,listing,high,value_and_anatomy
4,"Fredg.(Aug.2,1772;Ms;Black) 1500.00",False,False,False,True,True,listing,high,value_and_anatomy
5,"Frk sg(Fredericksburg)(Nov.10,1772;Ms;Black) 1500.00",False,False,False,True,True,listing,high,value_and_anatomy
6,"FREDERICKSBURG(""F""5mm high,used as bkstp) (March 1,1775;SL-50x3,MDD below;Red,Bl",False,False,False,True,True,listing,high,value_and_anatomy
7,"(L)(June 27,1775) 1000.00",True,False,False,True,True,listing,high,relationship_indicator
8,"Hampton(Oct.7,1765;Ms;Black) 1500.00",False,False,False,True,True,listing,high,value_and_anatomy
9,"Hpton (Hampton)(--, 1771;Ms;Black) 1000.00",False,False,False,True,True,listing,high,value_and_anatomy


## 6. Observations

This section summarizes what the signal detection found on this particular input file,
to inform refinement of thresholds before running on other state catalogs.

In [11]:
total = len(df)
print(f'Total entries: {total}')
print()
print('Signal coverage:')
print(f'  S1 (relationship indicator): {df["s1_relationship"].sum()} ({df["s1_relationship"].sum()/total*100:.1f}%)')
print(f'  S2 (cross-reference):        {df["s2_cross_ref"].sum()} ({df["s2_cross_ref"].sum()/total*100:.1f}%)')
print(f'  S3 (fragment):               {df["s3_fragment"].sum()} ({df["s3_fragment"].sum()/total*100:.1f}%)')
print(f'  S4 (trailing value):         {df["s4_trailing_value"].sum()} ({df["s4_trailing_value"].sum()/total*100:.1f}%)')
print(f'  S5 (structural anatomy):     {df["s5_anatomy"].sum()} ({df["s5_anatomy"].sum()/total*100:.1f}%)')
print()
print('Classification summary:')
for cls in ['listing', 'cross_reference', 'non_entry']:
    subset = df[df['classification'] == cls]
    n = len(subset)
    low = (subset['confidence'] == 'low').sum()
    if n > 0:
        conf_note = f' ({low} low-confidence)' if low else ''
        print(f'  {cls}: {n} ({n/total*100:.1f}%){conf_note}')

Total entries: 1539

Signal coverage:
  S1 (relationship indicator): 260 (16.9%)
  S2 (cross-reference):        7 (0.5%)
  S3 (fragment):               1 (0.1%)
  S4 (trailing value):         1539 (100.0%)
  S5 (structural anatomy):     1495 (97.1%)

Classification summary:
  listing: 1537 (99.9%) (22 low-confidence)
  cross_reference: 2 (0.1%)


# Step 2: Structural Segmentation

Splits each listing-classified row into three zones that all downstream extraction
steps depend on: **head**, **paren_body**, and **tail**.

Every subsequent field extraction (town name, dates, size, rates, colors, valuation)
needs to know *which part of the string* to look in. The parenthetical must be isolated
before semicolon fields can be parsed; the valuation must be separated from the attribute
block. This is the chokepoint.

### Three entry forms

| Form | Rule | Example |
|------|------|---------|
| `semicolon_paren` | Last balanced paren group contains `;` | `*Kaskaskia(April 8,1800;Ms;Black) 3500.00` |
| `simple_paren` | No semicolon-paren, but S1 (relationship indicator) hit and entry has parens | `Same(Green) 50.00` |
| `no_paren` | Everything else; any parens are name annotations | `*Ursine 1851 --` |

Why the S1 gate on `simple_paren`: in the ASCC format, non-relationship entries with
parens that lack semicolons are overwhelmingly name annotations -- county disambiguation
like `Shippensville (Clarion Co.)`, abbreviated spellings like `Lewisburg(h)`, or catalog
flags like `(1)Sistersville`. Only relationship-indicator entries use a single-attribute
paren as their modification block (e.g. `Same(Green)`).

### Segmentation zones

- **seg_head** -- text before the attribute parenthetical (town name, relationship indicators, name annotations)
- **seg_paren** -- content inside the attribute parens, excluding the parens themselves (null for no_paren)
- **seg_tail** -- text after the closing paren / trailing valuation

## 2.1 Entry Form Classification

In [12]:
# Only segment rows that Step 1 classified as listings
listings = df[df['classification'] == 'listing'].copy()
print(f'Segmenting {len(listings)} listings')

def find_last_semicolon_paren(text):
    """Find the last balanced paren group that contains a semicolon.
    Returns (open_pos, close_pos) or None."""
    # Walk backward through all ')' positions
    i = len(text) - 1
    while i >= 0:
        if text[i] == ')':
            close_pos = i
            depth = 1
            j = i - 1
            while j >= 0 and depth > 0:
                if text[j] == ')':
                    depth += 1
                elif text[j] == '(':
                    depth -= 1
                j -= 1
            if depth == 0:
                open_pos = j + 1
                body = text[open_pos + 1:close_pos]
                if ';' in body:
                    return (open_pos, close_pos)
            # This paren group had no semicolon; keep scanning left
            i = (j + 1) - 1 if depth == 0 else i - 1
        else:
            i -= 1
    return None

def find_last_paren_group(text):
    """Find the last balanced paren group (any content).
    Returns (open_pos, close_pos) or None."""
    close_pos = text.rfind(')')
    if close_pos == -1:
        return None
    depth = 1
    j = close_pos - 1
    while j >= 0 and depth > 0:
        if text[j] == ')':
            depth += 1
        elif text[j] == '(':
            depth -= 1
        j -= 1
    if depth != 0:
        return None  # unmatched
    open_pos = j + 1
    return (open_pos, close_pos)

def classify_entry_form(row):
    """Determine structural form: semicolon_paren, simple_paren, or no_paren."""
    text = row['clean_text']

    # Form 1: last paren group with semicolons
    if find_last_semicolon_paren(text) is not None:
        return 'semicolon_paren'

    # Form 2: relationship indicator + has parens (single-attribute modification)
    if row['s1_relationship'] and '(' in text:
        return 'simple_paren'

    # Form 3: everything else
    return 'no_paren'

listings['entry_form'] = listings.apply(classify_entry_form, axis=1)

print()
print('Entry form distribution:')
for form, count in listings['entry_form'].value_counts().items():
    print(f'  {form}: {count} ({count/len(listings)*100:.1f}%)')

Segmenting 1537 listings

Entry form distribution:
  no_paren: 792 (51.5%)
  semicolon_paren: 706 (45.9%)
  simple_paren: 39 (2.5%)


## 2.2 Segmentation

In [13]:
# Trailing value pattern: captures slash-separated valuations, decimals, integers, dashes
TRAILING_VALUE_RE = re.compile(
    r'(\d[\d,]*(?:\.\d+)?(?:/\d[\d,]*(?:\.\d+)?)*|---?)\s*$'
)

def segment_entry(row):
    """Split entry into head / paren_body / tail based on entry_form."""
    text = row['clean_text']
    form = row['entry_form']

    if form == 'semicolon_paren':
        bounds = find_last_semicolon_paren(text)
        if bounds is None:
            return pd.Series({'seg_head': text, 'seg_paren': None, 'seg_tail': None,
                              'seg_error': 'semicolon_paren but no match'})
        open_pos, close_pos = bounds
        head = text[:open_pos].strip()
        paren_body = text[open_pos + 1:close_pos]
        tail = text[close_pos + 1:].strip()
        return pd.Series({'seg_head': head, 'seg_paren': paren_body,
                          'seg_tail': tail, 'seg_error': None})

    elif form == 'simple_paren':
        bounds = find_last_paren_group(text)
        if bounds is None:
            return pd.Series({'seg_head': text, 'seg_paren': None, 'seg_tail': None,
                              'seg_error': 'simple_paren but no paren found'})
        open_pos, close_pos = bounds
        head = text[:open_pos].strip()
        paren_body = text[open_pos + 1:close_pos]
        tail = text[close_pos + 1:].strip()
        return pd.Series({'seg_head': head, 'seg_paren': paren_body,
                          'seg_tail': tail, 'seg_error': None})

    else:  # no_paren
        m = TRAILING_VALUE_RE.search(text)
        if m is None:
            return pd.Series({'seg_head': text, 'seg_paren': None, 'seg_tail': None,
                              'seg_error': 'no_paren but no trailing value'})
        tail = m.group(1)
        head = text[:m.start()].strip()
        return pd.Series({'seg_head': head, 'seg_paren': None,
                          'seg_tail': tail, 'seg_error': None})

segments = listings.apply(segment_entry, axis=1)
listings = pd.concat([listings, segments], axis=1)

# Report errors
errors = listings[listings['seg_error'].notna()]
print(f'Segmentation errors: {len(errors)} / {len(listings)}')
if len(errors):
    for _, row in errors.iterrows():
        print(f'  [{row["entry_form"]}] {row["seg_error"]}')
        print(f'    {row["clean_text"][:140]}')
    print()

print(f'Successful segmentations: {len(listings) - len(errors)}')

if len(errors):
    detail = errors[['clean_text', 'entry_form', 'seg_error']].head(30).copy()
    detail.insert(0, 'csv_line', detail.index + 2)
    raise AssertionError(
        f'Silent drop: {len(errors)} listing(s) failed segmentation. '
        f'These rows would be carried forward with empty seg_head/seg_paren/seg_tail.\n'
        f'First offenders (csv_line = line number in the input CSV):\n{detail.to_string(index=False)}'
    )

Segmentation errors: 0 / 1537
Successful segmentations: 1537


## 2.3 Validation

In [14]:
ok = listings[listings['seg_error'].isna()]

print('=== SEMICOLON_PAREN examples ===')
sp = ok[ok['entry_form'] == 'semicolon_paren']
print(f'Count: {len(sp)}')
for _, row in sp.head(8).iterrows():
    print(f'  raw:   {row["clean_text"][:100]}')
    print(f'  head:  {row["seg_head"]}')
    print(f'  paren: {row["seg_paren"]}')
    print(f'  tail:  {row["seg_tail"]}')
    print()

print('=== SIMPLE_PAREN examples ===')
simp = ok[ok['entry_form'] == 'simple_paren']
print(f'Count: {len(simp)}')
for _, row in simp.head(8).iterrows():
    print(f'  raw:   {row["clean_text"][:100]}')
    print(f'  head:  {row["seg_head"]}')
    print(f'  paren: {row["seg_paren"]}')
    print(f'  tail:  {row["seg_tail"]}')
    print()

print('=== NO_PAREN examples ===')
np_ = ok[ok['entry_form'] == 'no_paren']
print(f'Count: {len(np_)}')
for _, row in np_.head(8).iterrows():
    print(f'  raw:   {row["clean_text"][:100]}')
    print(f'  head:  {row["seg_head"]}')
    print(f'  tail:  {row["seg_tail"]}')
    print()

=== SEMICOLON_PAREN examples ===
Count: 706
  raw:   Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) 1500.00
  head:  Alexa.(Alexandria)(E)
  paren: May 21,1772;Ms;Black
  tail:  1500.00

  raw:   Colchester(June 30,1774;Ms;Black) 1500.00
  head:  Colchester
  paren: June 30,1774;Ms;Black
  tail:  1500.00

  raw:   Fredericksburg(May 10,1755;Ms;Black) 2000.00
  head:  Fredericksburg
  paren: May 10,1755;Ms;Black
  tail:  2000.00

  raw:   Fredg.(Aug.2,1772;Ms;Black) 1500.00
  head:  Fredg.
  paren: Aug.2,1772;Ms;Black
  tail:  1500.00

  raw:   Frk sg(Fredericksburg)(Nov.10,1772;Ms;Black) 1500.00
  head:  Frk sg(Fredericksburg)
  paren: Nov.10,1772;Ms;Black
  tail:  1500.00

  raw:   FREDERICKSBURG("F"5mm high,used as bkstp) (March 1,1775;SL-50x3,MDD below;Red,Black) 1200.00
  head:  FREDERICKSBURG("F"5mm high,used as bkstp)
  paren: March 1,1775;SL-50x3,MDD below;Red,Black
  tail:  1200.00

  raw:   Hampton(Oct.7,1765;Ms;Black) 1500.00
  head:  Hampton
  paren: Oct.7,1765;Ms;Black
  tail:

### Sanity checks

- Every listing should have a non-empty seg_tail (the valuation is always present -- verified by Signal 4).
- Every non-relationship listing should have a non-empty seg_head (the town name).
- seg_paren should be non-empty for semicolon_paren and simple_paren forms.

In [15]:
ok = listings[listings['seg_error'].isna()]

# Check 1: seg_tail should always be non-empty
empty_tail = ok[ok['seg_tail'].isna() | (ok['seg_tail'] == '')]
print(f'Empty seg_tail: {len(empty_tail)}')
if len(empty_tail):
    for _, row in empty_tail.iterrows():
        print(f'  [{row["entry_form"]}] {row["clean_text"][:120]}')
    print()

# Check 2: seg_head should be non-empty for non-relationship listings
non_rel = ok[~ok['s1_relationship']]
empty_head = non_rel[non_rel['seg_head'].isna() | (non_rel['seg_head'] == '')]
print(f'Empty seg_head (non-relationship): {len(empty_head)}')
if len(empty_head):
    for _, row in empty_head.iterrows():
        print(f'  [{row["entry_form"]}] {row["clean_text"][:120]}')
    print()

# Check 3: seg_paren should be non-empty for paren forms
paren_forms = ok[ok['entry_form'].isin(['semicolon_paren', 'simple_paren'])]
empty_paren = paren_forms[paren_forms['seg_paren'].isna() | (paren_forms['seg_paren'] == '')]
print(f'Empty seg_paren (paren forms): {len(empty_paren)}')
if len(empty_paren):
    for _, row in empty_paren.iterrows():
        print(f'  [{row["entry_form"]}] {row["clean_text"][:120]}')
    print()

# Check 4: show all simple_paren entries for manual verification
# (these are the highest-risk form -- make sure none are name annotations)
print(f'\n=== ALL simple_paren entries ({len(simp)}) ===')
for _, row in ok[ok['entry_form'] == 'simple_paren'].iterrows():
    print(f'  head={row["seg_head"]!r}  paren={row["seg_paren"]!r}  tail={row["seg_tail"]!r}')

Empty seg_tail: 0
Empty seg_head (non-relationship): 0
Empty seg_paren (paren forms): 0

=== ALL simple_paren entries (39) ===
  head='(L)'  paren='Sept.15,1774'  tail='1000.00'
  head='(L)'  paren='June 27,1775'  tail='1000.00'
  head='(L)'  paren='Dec.6,1765'  tail='1200.00'
  head='(L)'  paren='Oct.2,1775'  tail='1200.00'
  head='(L)'  paren='--,1757'  tail='2000.00'
  head='(L)'  paren='--,1757'  tail='1500.00'
  head='(L)'  paren='Nov.10,1774'  tail='1750.00'
  head='(L)'  paren='Aug.1776'  tail='500.00'
  head='(L)'  paren='July 9,1777'  tail='500.00'
  head=''  paren='L'  tail='See State --'
  head='(L)'  paren='Dec.1779'  tail='750.00'
  head='(L)'  paren='April 21,1781'  tail='750.00'
  head='(L)'  paren='Aug.16,1775,SL-50x3,MDD,Red,Black'  tail='750.00'
  head='(L)'  paren='--, 1778'  tail='400.00'
  head='(L)'  paren='--, 1778'  tail='--'
  head='(L)'  paren='March 13,1788'  tail='400.00'
  head=''  paren='L'  tail='See State --'
  head='(L)'  paren='Aug.23,1788'  tail='400.

## 2.4 Step 2 Summary

In [16]:
total = len(listings)
ok_count = len(ok)
err_count = len(errors)

print(f'Step 2: Structural Segmentation')
print(f'  Input listings: {total}')
print(f'  Successful:     {ok_count} ({ok_count/total*100:.1f}%)')
print(f'  Errors:         {err_count} ({err_count/total*100:.1f}%)')
print()
print(f'Entry form breakdown:')
for form in ['semicolon_paren', 'simple_paren', 'no_paren']:
    n = (listings['entry_form'] == form).sum()
    print(f'  {form}: {n} ({n/total*100:.1f}%)')

Step 2: Structural Segmentation
  Input listings: 1537
  Successful:     1537 (100.0%)
  Errors:         0 (0.0%)

Entry form breakdown:
  semicolon_paren: 706 (45.9%)
  simple_paren: 39 (2.5%)
  no_paren: 792 (51.5%)


# Step 3: Paren Field Splitting and Tail Extraction

Two mechanical splits:

1. **Paren fields** -- split `seg_paren` on semicolons into positional tokens. No semantic
   interpretation yet; just count and index. Only applies to `semicolon_paren` and
   `simple_paren` forms.

2. **Tail decomposition** -- split `seg_tail` into `tail_annotation` (nullable) and
   `tail_valuation`. For paren forms, `seg_tail` may contain annotation text between
   the closing `)` and the trailing valuation (~3% of semicolon-paren entries have this:
   "Soldier's mail", "See Way Mail Listing", etc.). For `no_paren` entries, Step 2 already
   isolated the valuation, so `tail_annotation` is null.

### Valuation patterns

The trailing valuation token is one of:
- Plain integer: `125`
- Comma-formatted: `1,200`
- Decimal: `3500.00`
- Slash-separated tiers: `200/15`
- Range: `125-200`
- Dashes (unpriced): `--` or `---`

Slash-separated valuations represent date-period pricing tiers per the domain model
(position 1 = earliest period). These are split into individual `PostmarkValuation`
records in a later step.

## 3.1 Paren Field Splitting

In [17]:
def split_paren_fields(row):
    """Split seg_paren on semicolons into positional list."""
    paren = row['seg_paren']
    if paren is None or (isinstance(paren, float) and pd.isna(paren)):
        return []
    fields = [f.strip() for f in paren.split(';')]
    return fields

listings['paren_fields'] = listings.apply(split_paren_fields, axis=1)
listings['paren_field_count'] = listings['paren_fields'].apply(len)

print('Paren field count distribution:')
fc = listings['paren_field_count'].value_counts().sort_index()
for n, count in fc.items():
    print(f'  {n} fields: {count} ({count/len(listings)*100:.1f}%)')
print()
print(f'Entries with 0 fields (no_paren): {(listings["paren_field_count"] == 0).sum()}')

Paren field count distribution:
  0 fields: 792 (51.5%)
  1 fields: 39 (2.5%)
  2 fields: 28 (1.8%)
  3 fields: 341 (22.2%)
  4 fields: 335 (21.8%)
  5 fields: 2 (0.1%)

Entries with 0 fields (no_paren): 792


In [18]:
# Show examples at each field count
for n in sorted(listings['paren_field_count'].unique()):
    subset = listings[listings['paren_field_count'] == n]
    print(f'=== {n} FIELDS ({len(subset)} entries) ===')
    for _, row in subset.head(5).iterrows():
        print(f'  raw:    {row["clean_text"][:110]}')
        if n > 0:
            print(f'  fields: {row["paren_fields"]}')
        print()

=== 0 FIELDS (792 entries) ===
  raw:    ALEX(1791-1846, see District of Columbia) --

  raw:    *Abingdon or Abbingdon 1804-28 --

  raw:    Accomack C.H 1835 15.00

  raw:    Accoman 1847 15.00

  raw:    Accotink 1851 15.00

=== 1 FIELDS (39 entries) ===
  raw:    (L)(Sept.15,1774) 1000.00
  fields: ['Sept.15,1774']

  raw:    (L)(June 27,1775) 1000.00
  fields: ['June 27,1775']

  raw:    (L)(Dec.6,1765) 1200.00
  fields: ['Dec.6,1765']

  raw:    (L)(Oct.2,1775) 1200.00
  fields: ['Oct.2,1775']

  raw:    (L)(--,1757) 2000.00
  fields: ['--,1757']

=== 2 FIELDS (28 entries) ===
  raw:    Same(May 6,1775;MDD same line) 1500.00
  fields: ['May 6,1775', 'MDD same line']

  raw:    Same(July 25,1775;Black) 1200.00
  fields: ['July 25,1775', 'Black']

  raw:    WBurg(Williamsburg)(E)(Nov.14,1734;Ms,Black) 2500.00
  fields: ['Nov.14,1734', 'Ms,Black']

  raw:    Same(--,1745;Way[ms]) See Way Mail section 2000.00
  fields: ['--,1745', 'Way[ms]']

  raw:    (E)(Nov.27,1773;Red) 1750.00
  

## 3.2 Tail Decomposition

Split `seg_tail` into `tail_annotation` (text before the valuation, nullable) and
`tail_valuation` (the trailing value token). The valuation regex is anchored at end
of string and handles slash-separated tiers (`200/15`), ranges (`125-200`), decimals
(`3500.00`), and dashes (`--`/`---`).

In [19]:
# Extended trailing value pattern: captures slash tiers, ranges, decimals, dashes
TAIL_VALUE_RE = re.compile(
    r'('
    r'\d[\d,]*(?:\.\d+)?'        # first number: 125, 1,200, 3500.00
    r'(?:[-/]\d[\d,]*(?:\.\d+)?)*'  # optional slash tiers or range: /15, -200
    r'|---?'                      # dashes: -- or ---
    r')\s*$'
)

def decompose_tail(row):
    """Split seg_tail into annotation (nullable) and valuation."""
    tail = row['seg_tail']
    form = row['entry_form']

    if tail is None or (isinstance(tail, float) and pd.isna(tail)) or tail.strip() == '':
        return pd.Series({'tail_annotation': None, 'tail_valuation': None,
                          'tail_error': 'empty tail'})

    tail = tail.strip()

    # For no_paren, Step 2 already isolated the valuation
    if form == 'no_paren':
        return pd.Series({'tail_annotation': None, 'tail_valuation': tail,
                          'tail_error': None})

    # For paren forms, split on trailing value
    m = TAIL_VALUE_RE.search(tail)
    if m is None:
        return pd.Series({'tail_annotation': tail if tail else None,
                          'tail_valuation': None,
                          'tail_error': 'no valuation found in tail'})

    valuation = m.group(1)
    annotation = tail[:m.start()].strip()
    if annotation in ('', '.', '*'):
        annotation = None
    return pd.Series({
        'tail_annotation': annotation,
        'tail_valuation': valuation,
        'tail_error': None
    })

tail_parts = listings.apply(decompose_tail, axis=1)
listings = pd.concat([listings, tail_parts], axis=1)

# Report
errors = listings[listings['tail_error'].notna()]
print(f'Tail decomposition errors: {len(errors)} / {len(listings)}')
if len(errors):
    for _, row in errors.head(10).iterrows():
        print(f'  [{row["entry_form"]}] {row["tail_error"]}')
        print(f'    seg_tail={row["seg_tail"]!r}')
        print(f'    raw: {row["clean_text"][:120]}')
        print()

if len(errors):
    detail = errors[['clean_text', 'entry_form', 'seg_tail', 'tail_error']].head(30).copy()
    detail.insert(0, 'csv_line', detail.index + 2)
    raise AssertionError(
        f'Silent drop: {len(errors)} listing(s) failed tail decomposition. '
        f'These rows would carry null tail_valuation and vanish from postmark_valuation.\n'
        f'First offenders (csv_line = line number in the input CSV):\n{detail.to_string(index=False)}'
    )

has_annotation = listings['tail_annotation'].notna() & (listings['tail_annotation'] != '')
print(f'Entries with tail annotation: {has_annotation.sum()} ({has_annotation.sum()/len(listings)*100:.1f}%)')

Tail decomposition errors: 0 / 1537
Entries with tail annotation: 40 (2.6%)


In [20]:
# Show all tail annotations for manual inspection
annotated = listings[listings['tail_annotation'].notna() & (listings['tail_annotation'] != '')]
if len(annotated):
    print(f'=== ALL TAIL ANNOTATIONS ({len(annotated)}) ===')
    for _, row in annotated.iterrows():
        print(f'  annotation={row["tail_annotation"]!r}  val={row["tail_valuation"]!r}')
        print(f'    raw: {row["clean_text"][:120]}')
        print()
else:
    print('No tail annotations found in this file.')

=== ALL TAIL ANNOTATIONS (40) ===
  annotation='datelined Lisbon'  val='--'
    raw: Norfolk sh(Sept.2,1754;Ms;Brown) datelined Lisbon --

  annotation='See Way Mail section'  val='2000.00'
    raw: Same(--,1745;Way[ms]) See Way Mail section 2000.00

  annotation='See State'  val='--'
    raw: (L) See State --

  annotation='See State'  val='--'
    raw: (L) See State --

  annotation='See Way section'  val='2000.00'
    raw: (L)(April 21,1780;Way) See Way section 2000.00

  annotation='See State'  val='--'
    raw: (L) See State --

  annotation='See State'  val='--'
    raw: (L) See State --

  annotation='See State'  val='--'
    raw: (L) See State --

  annotation='datelined Martinique'  val='--'
    raw: Same(Nov.2,1776;Ms;Brown-red) datelined Martinique --

  annotation='State use'  val='200.00'
    raw: ALEX,(1788-90;SL-12x3,MDD;Black) State use 200.00

  annotation='See Inland Waterways'  val='--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  annotation="Soldier'

## 3.3 Valuation Tier Splitting

Split slash-separated valuations into positional tiers. `200/15` becomes
`['200', '15']` where position 1 = earliest date period.

In [21]:
def split_valuation_tiers(val_str):
    """Split a valuation string into positional tiers.
    Returns list of tier strings. Dashes become [None]. Ranges stay intact."""
    if val_str is None or (isinstance(val_str, float) and pd.isna(val_str)):
        return []
    val_str = val_str.strip()
    if val_str in ('--', '---'):
        return [None]  # unpriced
    # Split on / for tier separation
    tiers = val_str.split('/')
    return tiers

listings['valuation_tiers'] = listings['tail_valuation'].apply(split_valuation_tiers)
listings['valuation_tier_count'] = listings['valuation_tiers'].apply(len)

print('Valuation tier count distribution:')
tc = listings['valuation_tier_count'].value_counts().sort_index()
for n, count in tc.items():
    print(f'  {n} tiers: {count} ({count/len(listings)*100:.1f}%)')
print()

# Show multi-tier examples
multi = listings[listings['valuation_tier_count'] > 1]
if len(multi):
    print(f'=== MULTI-TIER VALUATIONS ({len(multi)}) ===')
    for _, row in multi.head(10).iterrows():
        print(f'  {row["tail_valuation"]} -> {row["valuation_tiers"]}')
        print(f'    raw: {row["clean_text"][:100]}')
        print()

# Show unpriced entries
unpriced = listings[listings['valuation_tiers'].apply(lambda t: len(t) == 1 and t[0] is None)]
if len(unpriced):
    print(f'=== UNPRICED ENTRIES ({len(unpriced)}) ===')
    for _, row in unpriced.head(10).iterrows():
        print(f'  {row["clean_text"][:100]}')

Valuation tier count distribution:
  1 tiers: 1517 (98.7%)
  2 tiers: 20 (1.3%)

=== MULTI-TIER VALUATIONS (20) ===
  50/15.00 -> ['50', '15.00']
    raw: Aquia(s) 1811,1849-55 50/15.00

  65/25.00 -> ['65', '25.00']
    raw: Front Royal 1809,1819-23,1836 65/25.00

  90/40.00 -> ['90', '40.00']
    raw: (1)Greenbriar C.H 1802-16,1829 90/40.00

  60/30.00 -> ['60', '30.00']
    raw: (1)Guyandotte 1811-21,1836-52 60/30.00

  95/75.00 -> ['95', '75.00']
    raw: Hungry Town 1801,1814-16 95/75.00

  100/50.00 -> ['100', '50.00']
    raw: (1)Kanawha C.H 1802-12 100/50.00

  60/50.00 -> ['60', '50.00']
    raw: (1)Lorentz Store 1833,1850 60/50.00

  75/40.00 -> ['75', '40.00']
    raw: (1)Martinsburg(h) 1799-1800,1803-29 75/40.00

  75/50.00 -> ['75', '50.00']
    raw: (1)Monroe C.H. 1808-20 75/50.00

  65/35.00 -> ['65', '35.00']
    raw: Patrich CH 1812,1847 65/35.00

=== UNPRICED ENTRIES (100) ===
  Norfolk sh(Sept.2,1754;Ms;Brown) datelined Lisbon --
  (L) See State --
  Fredbg(E)(May 3,

## 3.4 Step 3 Summary

In [22]:
total = len(listings)
seg_err = listings['seg_error'].notna().sum() if 'seg_error' in listings.columns else 0
tail_err = listings['tail_error'].notna().sum()
has_ann = (listings['tail_annotation'].notna() & (listings['tail_annotation'] != '')).sum()

print(f'Step 3: Paren Field Splitting and Tail Extraction')
print(f'  Input listings: {total}')
print()
print(f'  Paren field counts:')
for n, count in listings['paren_field_count'].value_counts().sort_index().items():
    print(f'    {n} fields: {count}')
print()
print(f'  Tail decomposition:')
print(f'    Errors:      {tail_err}')
print(f'    Annotations: {has_ann}')
print()
print(f'  Valuation tiers:')
for n, count in listings['valuation_tier_count'].value_counts().sort_index().items():
    label = 'unpriced' if n == 1 and listings[listings['valuation_tier_count'] == n]['valuation_tiers'].apply(lambda t: t[0] is None).all() else f'{n}-tier'
    print(f'    {n} tiers: {count}')

Step 3: Paren Field Splitting and Tail Extraction
  Input listings: 1537

  Paren field counts:
    0 fields: 792
    1 fields: 39
    2 fields: 28
    3 fields: 341
    4 fields: 335
    5 fields: 2

  Tail decomposition:
    Errors:      0
    Annotations: 40

  Valuation tiers:


    1 tiers: 1517
    2 tiers: 20


# Step 4: Head Parsing

Extracts structured components from `seg_head`. The head contains everything
before the attribute parenthetical: relationship indicators, the first-of-town
marker, the town/postmark inscription text, and name-annotation parens.

### Outputs

- **head_first_of_town** (bool) — leading `*` was present (ASCC convention: first listing for this town)
- **head_rel_type** (nullable str) — relationship indicator token: `Same`, `(L)`, `(E)`, `(L)*`, `(E)*`, or null
- **head_name_body** (nullable str) — text remaining after stripping `*`/`+`/rel-indicator AND removing annotation parens. This is the raw town-marking inscription (e.g. `LINCOLN/ILL`, `Kaskaskia`). Null when the head is a pure relationship indicator with no residual text.
- **head_annotations** (list of str) — parenthesized annotation contents from the head, in order. Captures period markers `(E)`/`(L)`, canonical-name expansions `(Cahokia)`, place-type qualifiers `(Fort)`, county disambiguations `(Clarion Co.)`, and letter-variant appendages `(h)`.

Section-context columns (`Default Shape`, `Page`, `Images Above`, `Institutional Ownership`) are already carried on the `listings` DataFrame from the input CSV and are validated at the end of this step.

In [23]:
# --- Step 4.1: Head parsing ---

PAREN_GROUP_RE = re.compile(r'\(([^)]*)\)')

# Relationship indicator at head start (after * and + have been stripped).
# Matches: Same, (L), (E), (L)*, (E)*
# Does NOT consume a leading * -- that's already handled as first_of_town.
REL_INDICATOR_RE = re.compile(
    r'^(?:'
    r'Same'
    r'|[(\[{][LE][)\]}]\*?'
    r')'
)

def parse_head(row):
    """Extract structured components from seg_head."""
    head = str(row['seg_head']) if pd.notna(row['seg_head']) else ''

    # 1. First-of-town marker (leading *)
    first_of_town = head.startswith('*')
    if first_of_town:
        head = head[1:]

    # 2. Plus prefix (rare; allowed by S1 regex but uncommon)
    plus_prefix = head.startswith('+')
    if plus_prefix:
        head = head[1:]

    # 3. Relationship indicator
    rel_type = None
    if row['s1_relationship']:
        m = REL_INDICATOR_RE.match(head)
        if m:
            rel_type = m.group(0)
            head = head[m.end():]

    # 4. Annotations: all (...) groups remaining in head
    annotations = PAREN_GROUP_RE.findall(head)

    # 5. Name body: head text with annotation parens removed, stripped
    name_body = PAREN_GROUP_RE.sub('', head).strip()
    name_body = name_body if name_body else None

    return pd.Series({
        'head_first_of_town': first_of_town,
        'head_rel_type': rel_type,
        'head_name_body': name_body,
        'head_annotations': annotations,
    })

head_parts = listings.apply(parse_head, axis=1)
listings = pd.concat([listings, head_parts], axis=1)

print(f'Step 4: Head parsing applied to {len(listings)} listings')
print(f'  First-of-town markers: {listings["head_first_of_town"].sum()}')
print(f'  Relationship indicators: {listings["head_rel_type"].notna().sum()}')
has_name = listings['head_name_body'].notna()
print(f'  Entries with name body: {has_name.sum()} ({has_name.sum()/len(listings)*100:.1f}%)')
has_ann = listings['head_annotations'].apply(lambda a: len(a) > 0)
print(f'  Entries with annotations: {has_ann.sum()} ({has_ann.sum()/len(listings)*100:.1f}%)')

Step 4: Head parsing applied to 1537 listings
  First-of-town markers: 52
  Relationship indicators: 255
  Entries with name body: 1365 (88.8%)
  Entries with annotations: 476 (31.0%)


In [24]:
# --- Step 4.2: Relationship indicator distribution ---

rel_counts = listings['head_rel_type'].value_counts(dropna=False)
print('Relationship indicator distribution:')
for val, count in rel_counts.items():
    label = repr(val) if val is not None else '(none -- independent entry)'
    print(f'  {label}: {count}')
print()

# Show a few examples per rel type
for rt in listings['head_rel_type'].dropna().unique():
    subset = listings[listings['head_rel_type'] == rt]
    print(f'=== rel_type={rt!r} ({len(subset)} entries) ===')
    for _, row in subset.head(4).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  ->  name_body={row["head_name_body"]!r}  ann={row["head_annotations"]}')
    print()

Relationship indicator distribution:
  nan: 1282
  'Same': 233
  '(L)': 21
  '(E)': 1

=== rel_type='(L)' (21 entries) ===
  seg_head='(L)'  ->  name_body=nan  ann=[]
  seg_head='(L)'  ->  name_body=nan  ann=[]
  seg_head='(L)'  ->  name_body=nan  ann=[]
  seg_head='(L)'  ->  name_body=nan  ann=[]

=== rel_type='Same' (233 entries) ===
  seg_head='Same'  ->  name_body=nan  ann=[]
  seg_head='Same'  ->  name_body=nan  ann=[]
  seg_head='Same'  ->  name_body=nan  ann=[]
  seg_head='Same'  ->  name_body=nan  ann=[]

=== rel_type='(E)' (1 entries) ===
  seg_head='(E)'  ->  name_body=nan  ann=[]



In [25]:
# --- Step 4.3: Name body and annotation inspection ---

# Independent entries (no rel indicator) should always have a name body
independent = listings[listings['head_rel_type'].isna()]
missing_name = independent[independent['head_name_body'].isna()]
print(f'Independent entries missing name body: {len(missing_name)} / {len(independent)}')
if len(missing_name):
    for _, row in missing_name.head(5).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  clean_text={row["clean_text"][:100]}')
print()

# Annotation value distribution
all_annotations = [a for ann_list in listings['head_annotations'] for a in ann_list]
ann_counts = pd.Series(all_annotations).value_counts()
print(f'Annotation values ({len(all_annotations)} total across {has_ann.sum()} entries):')
for val, count in ann_counts.head(20).items():
    print(f'  {val!r}: {count}')
print()

# Entries with >1 annotation
multi_ann = listings[listings['head_annotations'].apply(len) > 1]
if len(multi_ann):
    print(f'=== MULTI-ANNOTATION HEADS ({len(multi_ann)}) ===')
    for _, row in multi_ann.head(10).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  ->  name={row["head_name_body"]!r}  ann={row["head_annotations"]}')
    print()

# Rel-indicator entries WITH a name body (e.g. Same/ILL)
rel_with_name = listings[listings['head_rel_type'].notna() & listings['head_name_body'].notna()]
if len(rel_with_name):
    print(f'=== REL-INDICATOR ENTRIES WITH NAME BODY ({len(rel_with_name)}) ===')
    for _, row in rel_with_name.head(15).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  rel={row["head_rel_type"]!r}  name={row["head_name_body"]!r}  ann={row["head_annotations"]}')

Independent entries missing name body: 6 / 1282
  seg_head=''  clean_text=(L) See State --
  seg_head=''  clean_text=(L) See State --
  seg_head=''  clean_text=(L) See State --
  seg_head=''  clean_text=(L) See State --
  seg_head=''  clean_text=(L) See State --

Annotation values (531 total across 476 entries):
  '1': 283
  '"A"high': 37
  '"a"high': 29
  'E': 23
  'between two lines': 8
  '"G"&"A"high': 6
  '"A"small': 6
  'backstamp': 4
  '"D"high': 3
  'Alum Springs': 3
  '"S"&"A"high': 3
  'large letters': 3
  '"G"&"A"high,fleuron': 3
  'between double lines': 3
  'long"s"': 3
  'Co': 3
  'Alexandria': 2
  'Fredericksburg': 2
  'Norfolk': 2
  'Williamsburg': 2

=== MULTI-ANNOTATION HEADS (51) ===
  seg_head='Alexa.(Alexandria)(E)'  ->  name='Alexa.'  ann=['Alexandria', 'E']
  seg_head='Norf(Norfolk)(E)'  ->  name='Norf'  ann=['Norfolk', 'E']
  seg_head='NORFOLK(backstamp)(E)'  ->  name='NORFOLK'  ann=['backstamp', 'E']
  seg_head='WBurg(Williamsburg)(E)'  ->  name='WBurg'  ann=['W

In [26]:
# --- Step 4.4: Section-context pass-through validation ---

# Report on each optional column carried through on the listings DataFrame.
# Absence is informational only -- required-column enforcement happened in Cell 2.
print('Optional CSV columns carried on listings DataFrame:')
for col in OPTIONAL_COLS:
    if col in listings.columns:
        non_null = listings[col].notna().sum()
        nunique = listings[col].nunique()
        print(f'  {col}: {non_null} non-null, {nunique} distinct values')
        if nunique <= 15:
            for val, count in listings[col].value_counts().head(10).items():
                print(f'    {val!r}: {count}')
    else:
        print(f'  {col}: (absent)')
    print()


Optional CSV columns carried on listings DataFrame:
  Manuscript: 1441 non-null, 2 distinct values
    True: 791
    False: 650

  Default Shape: 650 non-null, 1 distinct values
    'C - Circle': 650

  Institutional Ownership: (absent)



## 4.5 Step 4 Summary

Head parsing is purely mechanical decomposition of `seg_head`. It does not resolve
inherited town names (that requires parent-child linkage in Step 6) or classify
annotation semantics (period marker vs. canonical name vs. county -- also Step 6).

At this point each listing row carries:
- **Structural segments** (Step 2): `seg_head`, `seg_paren`, `seg_tail`
- **Paren fields** (Step 3.1): `paren_fields`, `paren_field_count`
- **Tail parts** (Step 3.2-3.3): `tail_annotation`, `tail_valuation`, `valuation_tiers`
- **Head parts** (Step 4): `head_first_of_town`, `head_rel_type`, `head_name_body`, `head_annotations`
- **Section context**: `Default Shape`, `Page`, `Images Above`, `Institutional Ownership`

In [27]:
# --- Step 4.5: Summary ---

total = len(listings)
print(f'Step 4: Head Parsing')
print(f'  Input listings: {total}')
print()
print(f'  First-of-town: {listings["head_first_of_town"].sum()} ({listings["head_first_of_town"].sum()/total*100:.1f}%)')
print()
print(f'  Relationship indicators:')
for val, count in listings['head_rel_type'].value_counts(dropna=True).items():
    print(f'    {val!r}: {count}')
none_count = listings['head_rel_type'].isna().sum()
print(f'    (independent): {none_count}')
print()
print(f'  Name body present: {listings["head_name_body"].notna().sum()}')
print(f'  Entries with annotations: {(listings["head_annotations"].apply(len) > 0).sum()}')

Step 4: Head Parsing
  Input listings: 1537

  First-of-town: 52 (3.4%)

  Relationship indicators:
    'Same': 233
    '(L)': 21
    '(E)': 1
    (independent): 1282

  Name body present: 1365
  Entries with annotations: 476


# Step 5: Paren Field-Type Classification

Classifies each semicolon-delimited token in `paren_fields` as one of six
content types, using intrinsic text signals only (no positional assumptions).

### Field types

| Type | Signal | Examples |
|------|--------|----------|
| `date` | Month name, 4-digit year 1500-1899, decade, c-prefix | `April 8,1800`, `1850's`, `c1840` |
| `ms` | Exact token `Ms` | `Ms` |
| `size` | Shape-code prefix, WxH dimension, bare 1-3 digit number, dateformat suffix | `SL-16.5x5,MDD`, `DC-25,YD`, `32` |
| `rate` | PAID/FREE/STEAM/DUE keyword, `[...]` bracket, P.M. notation, frank | `FREE`, `25[ms]`, `P.M.Free`, `PAID 6,40` |
| `color` | All comma-separated tokens are known ASCC color names | `Black`, `Red,Blue`, `Olive-Yellow` |
| `other` | None of the above | Residual; inspect for missed patterns |

### Priority order

1. `ms` (exact match -- fastest, no ambiguity)
2. `date` (month names and years are unique to date fields)
3. `rate` (brackets and rate keywords disambiguate from size before bare-number fallback)
4. `size` (shape codes, dimension patterns, dateformat suffixes)
5. `color` (known color vocabulary)
6. Bare 1-3 digit number without other signals -> `size` (ASCC convention: bare numbers in paren fields are mm dimensions; rate values use brackets or keywords)
7. `other`

In [28]:
# --- Step 5.1: Field-type detectors ---

MONTHS_PAT = (
    r'(?:\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|June?|July?'
    r'|Aug(?:ust)?|Sep(?:t(?:ember)?)?\\.?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\b)'
)

DATE_FIELD_RE = re.compile(
    r'(?:' + MONTHS_PAT + r'|c?1[5-8]\d{2})',
    re.IGNORECASE
)

# Tokens that appear as comma-suffixes in the marking-description field:
# date-format codes (MDD, MD, YD, YMD, YMDD) plus NOR (No Outer Rim)
SIZE_SUFFIX_PAT = r'(?:YMDD|MDD|YMD|YD|MD|NOR)'

# Shape codes from ASCC header -- ordered longest-first to avoid prefix collisions
SHAPE_CODE_PAT = (
    r'(?:DLDC|DLDO|DLC|DLO|Octagon|Box|Arc|Pmk|SL|DC|DO|NOR|O|C)'
)

SIZE_FIELD_RE = re.compile(
    r'(?:'
    + SHAPE_CODE_PAT + r'[\-\s]?\d'
    + r'|\d+\.?\d*\s*x\s*\d'
    + r'|^-{1,2}(?:\s*,'
    + SIZE_SUFFIX_PAT + r')?$'
    + r'|\d+\.?\d*\s*,'
    + SIZE_SUFFIX_PAT
    + r'|^'
    + SIZE_SUFFIX_PAT + r'$'
    + r')',
    re.IGNORECASE
)

RATE_FIELD_RE = re.compile(
    r'(?:'
    r'\bPAID\b|\bFREE\b|\bSTEAM\b|\bDUE\b'
    r'|\bP\.?M\.?'
    r'|\bfrank\b'
    r'|\[[^\]]*\]'         # brackets: [ms], [C], [hdstp rate], [cogged circle]
    r'|\bwith\s+\d'        # "with 24" = with adhesive
    r')',
    re.IGNORECASE
)

KNOWN_COLORS = {
    'black', 'red', 'blue', 'green', 'brown', 'orange', 'purple',
    'magenta', 'yellow', 'olive', 'violet', 'carmine', 'vermilion',
    'pink', 'gray', 'grey', 'buff', 'salmon', 'rose', 'maroon',
    'crimson', 'indigo', 'lilac', 'scarlet', 'amber',
}

def is_color_token(tok):
    """Check a single token (possibly hyphen-compound) against color vocabulary."""
    parts = tok.strip().lower().split('-')
    return all(p in KNOWN_COLORS for p in parts if p)

def is_color_field(field):
    """True if all comma-separated tokens in the field are known colors."""
    tokens = [t.strip() for t in field.split(',') if t.strip()]
    return bool(tokens) and all(is_color_token(t) for t in tokens)

BARE_NUMBER_RE = re.compile(r'^\d{1,3}(?:\.\d+)?$')

def classify_paren_field(field_text):
    """Classify a single paren field by intrinsic content signals.
    Returns one of: date, ms, size, rate, color, other, empty."""
    f = field_text.strip()
    if not f:
        return 'empty'

    # 1. Manuscript (exact)
    if f == 'Ms':
        return 'ms'

    # 2. Date expression
    if DATE_FIELD_RE.search(f):
        return 'date'

    # 3. Rate/auxmark (checked before size -- brackets disambiguate)
    if RATE_FIELD_RE.search(f):
        return 'rate'

    # 4. Size/shape/dateformat composite
    if SIZE_FIELD_RE.search(f):
        return 'size'

    # 5. Color
    if is_color_field(f):
        return 'color'

    # 6. Bare small number -> size by ASCC convention
    if BARE_NUMBER_RE.match(f):
        return 'size'

    return 'other'

# Quick self-test
assert classify_paren_field('April 8,1800') == 'date'
assert classify_paren_field('1850-53') == 'date'
assert classify_paren_field("1850's") == 'date'
assert classify_paren_field('Ms') == 'ms'
assert classify_paren_field('SL-16.5x5,MDD') == 'size'
assert classify_paren_field('DC-25,YD') == 'size'
assert classify_paren_field('32') == 'size'
assert classify_paren_field('--,YD') == 'size'
assert classify_paren_field('FREE') == 'rate'
assert classify_paren_field('25[ms]') == 'rate'
assert classify_paren_field('P.M.Free') == 'rate'
assert classify_paren_field('Geo.Fisher P.M.frank') == 'rate'
assert classify_paren_field('Black') == 'color'
assert classify_paren_field('Red,Blue') == 'color'
assert classify_paren_field('Olive-Yellow') == 'color'
print('All self-tests passed')

All self-tests passed


In [29]:
# --- Step 5.2: Apply classifier to all paren fields ---

def classify_all_fields(paren_fields):
    """Classify each field in the list. Returns parallel list of type labels."""
    types = [classify_paren_field(f) for f in paren_fields]
    
    # Positional disambiguation: ASCC entries have at most one size field.
    # If a second 'size' appears and it's a bare number (no shape code, no
    # dateformat, no dimension separator), reclassify it as 'rate'.
    size_seen = False
    for i, (field, ftype) in enumerate(zip(paren_fields, types)):
        if ftype == 'size':
            if size_seen and BARE_NUMBER_RE.match(field.strip()):
                types[i] = 'rate'
            else:
                size_seen = True

    return types

listings['paren_field_types'] = listings['paren_fields'].apply(classify_all_fields)

# Flatten for aggregate stats
all_fields = []
for _, row in listings.iterrows():
    for pos, (field, ftype) in enumerate(zip(row['paren_fields'], row['paren_field_types'])):
        all_fields.append({
            'position': pos,
            'field_text': field,
            'field_type': ftype,
            'field_count': row['paren_field_count'],
        })

field_df = pd.DataFrame(all_fields)
total_fields = len(field_df)

print(f'Total paren fields classified: {total_fields}')
print()
print('Overall type distribution:')
for ftype, count in field_df['field_type'].value_counts().items():
    print(f'  {ftype}: {count} ({count/total_fields*100:.1f}%)')

Total paren fields classified: 2468

Overall type distribution:
  color: 704 (28.5%)
  date: 694 (28.1%)
  size: 618 (25.0%)
  rate: 365 (14.8%)
  other: 47 (1.9%)
  ms: 40 (1.6%)


In [30]:
# --- Step 5.3: Type distribution by field position ---

print('Field type by position:')
print()
for pos in sorted(field_df['position'].unique()):
    subset = field_df[field_df['position'] == pos]
    print(f'Position {pos} ({len(subset)} fields):')
    for ftype, count in subset['field_type'].value_counts().items():
        print(f'  {ftype}: {count} ({count/len(subset)*100:.1f}%)')
    print()

Field type by position:

Position 0 (745 fields):
  date: 694 (93.2%)
  size: 26 (3.5%)
  color: 13 (1.7%)
  other: 8 (1.1%)
  rate: 4 (0.5%)

Position 1 (706 fields):
  size: 582 (82.4%)
  ms: 40 (5.7%)
  rate: 36 (5.1%)
  other: 26 (3.7%)
  color: 22 (3.1%)

Position 2 (678 fields):
  color: 336 (49.6%)
  rate: 323 (47.6%)
  other: 11 (1.6%)
  size: 8 (1.2%)

Position 3 (337 fields):
  color: 331 (98.2%)
  size: 2 (0.6%)
  other: 2 (0.6%)
  rate: 2 (0.6%)

Position 4 (2 fields):
  color: 2 (100.0%)



In [31]:
# --- Step 5.4: Entry type-signature patterns ---

# Build a type-signature string per entry: e.g. "date|size|rate|color"
listings['field_type_sig'] = listings['paren_field_types'].apply(
    lambda types: '|'.join(types) if types else '(none)'
)

sig_counts = listings['field_type_sig'].value_counts()
print(f'Distinct type signatures: {len(sig_counts)}')
print()
print('Type signature distribution:')
for sig, count in sig_counts.head(20).items():
    print(f'  {sig}: {count} ({count/len(listings)*100:.1f}%)')

# Flag any signature containing 'other'
has_other = listings[listings['field_type_sig'].str.contains('other')]
print(f'\nEntries with "other" fields: {has_other.sum() if isinstance(has_other, pd.Series) else len(has_other)}')

Distinct type signatures: 33

Type signature distribution:
  (none): 792 (51.5%)
  date|size|rate|color: 304 (19.8%)
  date|size|color: 253 (16.5%)
  date|ms|color: 40 (2.6%)
  date: 19 (1.2%)
  date|color: 18 (1.2%)
  date|rate|color: 16 (1.0%)
  date|other|color: 14 (0.9%)
  color: 13 (0.8%)
  date|size|other|color: 10 (0.7%)
  size|rate|rate|color: 7 (0.5%)
  other: 6 (0.4%)
  size|rate|color: 6 (0.4%)
  date|other: 4 (0.3%)
  size|other|color: 4 (0.3%)
  date|other|rate|color: 4 (0.3%)
  rate|color: 3 (0.2%)
  size|size|color: 3 (0.2%)
  date|rate: 2 (0.1%)
  date|size|size|color: 2 (0.1%)

Entries with "other" fields: 47


In [32]:
# --- Step 5.5: Sample inspection by type ---

for ftype in ['date', 'ms', 'size', 'rate', 'color', 'other']:
    subset = field_df[field_df['field_type'] == ftype]
    if len(subset) == 0:
        continue
    print(f'=== {ftype.upper()} ({len(subset)} fields) ===')

    # Show unique values (up to 30)
    uniques = subset['field_text'].value_counts()
    shown = 0
    for val, count in uniques.items():
        print(f'  {val!r}: {count}')
        shown += 1
        if shown >= 30:
            remaining = len(uniques) - shown
            if remaining > 0:
                print(f'  ... and {remaining} more distinct values')
            break
    print()

=== DATE (694 fields) ===
  "1850's": 29
  '1855': 20
  '1854': 15
  '1853': 13
  '1852': 12
  '1851': 12
  '1846': 11
  '1851-52': 10
  '1850': 9
  '1851-55': 7
  '1861': 7
  '1847': 7
  '1852-55': 6
  '1854-55': 6
  '1833': 6
  '1851-54': 5
  '1839': 5
  '1853-54': 5
  '1832': 5
  '1853-55': 5
  '1835': 5
  '1859': 5
  '1813': 4
  '1851-53': 4
  '1850-52': 4
  '1834': 4
  '1856': 4
  '1825': 4
  '1820': 4
  '1797': 4
  ... and 363 more distinct values

=== MS (40 fields) ===
  'Ms': 40

=== SIZE (618 fields) ===
  '30': 152
  '32': 65
  '--': 42
  '31': 42
  '29': 34
  '33': 26
  '28': 22
  '35': 14
  '27': 8
  '26': 8
  '34': 7
  '25': 6
  '36': 6
  'DC-26': 5
  '37': 5
  'SL-29x3,MDD': 4
  '24': 4
  '28,NOR': 4
  'DC-29': 3
  'MDD': 3
  '29,NOR': 3
  'SL-20x3,YMDD': 3
  'SL-38x4,MDD': 3
  'SL-24x4,MDD': 2
  'SL-19x3,MDD': 2
  'DC-30': 2
  '30.5': 2
  '23': 2
  'DC-27': 2
  '5': 2
  ... and 130 more distinct values

=== RATE (365 fields) ===
  'PAID': 52
  'PAID/3[C]': 51
  'FREE': 

In [33]:
# --- Step 5.6: Other/unclassified fields ---

other_fields = field_df[field_df['field_type'] == 'other']
if len(other_fields) == 0:
    print('No unclassified fields -- all paren tokens matched a known type.')
else:
    print(f'=== UNCLASSIFIED FIELDS ({len(other_fields)}) ===')
    print()
    for _, frow in other_fields.iterrows():
        # Find the parent listing
        match = listings[
            listings['paren_fields'].apply(lambda pf: frow['field_text'] in pf)
        ]
        if len(match):
            first = match.iloc[0]
            print(f'  field={frow["field_text"]!r}  pos={frow["position"]}')
            print(f'    fields={first["paren_fields"]}')
            print(f'    types={first["paren_field_types"]}')
            print(f'    raw: {first["clean_text"][:120]}')
            print()

=== UNCLASSIFIED FIELDS (47) ===

  field='MDD same line'  pos=1
    fields=['May 6,1775', 'MDD same line']
    types=['date', 'other']
    raw: Same(May 6,1775;MDD same line) 1500.00

  field='MDD below'  pos=1
    fields=['July 25,1775', 'MDD below', 'Red']
    types=['date', 'other', 'color']
    raw: Same(July 25,1775;MDD below;Red) 1500.00

  field='Ms,Black'  pos=1
    fields=['Nov.14,1734', 'Ms,Black']
    types=['date', 'other']
    raw: WBurg(Williamsburg)(E)(Nov.14,1734;Ms,Black) 2500.00

  field='L'  pos=0
    fields=['L']
    types=['other']
    raw: (L) See State --

  field='Ms,Black'  pos=1
    fields=['Nov.14,1734', 'Ms,Black']
    types=['date', 'other']
    raw: WBurg(Williamsburg)(E)(Nov.14,1734;Ms,Black) 2500.00

  field='L'  pos=0
    fields=['L']
    types=['other']
    raw: (L) See State --

  field='Way'  pos=1
    fields=['April 21,1780', 'Way']
    types=['date', 'other']
    raw: (L)(April 21,1780;Way) See Way section 2000.00

  field='L'  pos=0
    fields=['

In [34]:
# --- Step 5.7: Positional consistency check ---

# Verify: position 0 should overwhelmingly be 'date'
if len(field_df[field_df['position'] == 0]) > 0:
    pos0 = field_df[field_df['position'] == 0]
    non_date_pos0 = pos0[pos0['field_type'] != 'date']
    print(f'Position 0 non-date fields: {len(non_date_pos0)} / {len(pos0)}')
    if len(non_date_pos0):
        for _, frow in non_date_pos0.iterrows():
            match = listings[
                listings['paren_fields'].apply(lambda pf: len(pf) > 0 and pf[0] == frow['field_text'])
            ]
            if len(match):
                first = match.iloc[0]
                print(f'  {frow["field_type"]}: {frow["field_text"]!r}')
                print(f'    raw: {first["clean_text"][:120]}')
                print()

# Last position should overwhelmingly be 'color' (for multi-field entries)
multi = field_df[field_df['field_count'] >= 3].copy()
if len(multi):
    last_pos = multi.groupby('field_count')['position'].transform('max')
    last_fields = multi[multi['position'] == last_pos]
    non_color_last = last_fields[last_fields['field_type'] != 'color']
    print(f'Last-position non-color fields (3+ field entries): {len(non_color_last)} / {len(last_fields)}')
    if len(non_color_last):
        for _, frow in non_color_last.head(10).iterrows():
            print(f'  pos={frow["position"]} type={frow["field_type"]}: {frow["field_text"]!r}')

Position 0 non-date fields: 51 / 745
  other: 'L'
    raw: (L) See State --

  other: 'L'
    raw: (L) See State --

  other: 'L'
    raw: (L) See State --

  other: 'L'
    raw: (L) See State --

  other: 'L'
    raw: (L) See State --

  color: 'Green'
    raw: Same(Green) 30.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  size: 'DL box 54x30'
    raw: Same printed postmark(DL box 54x30; "VA."instead of"VIRG'A.";Black) 600.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  color: 'Green'
    raw: Same(Green) 30.00

  color: 'Green'
    raw: Same(Green) 30.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  color: 'Green'
    raw: Same(Green) 30.00

  rate: '--,10[oval 31x18]'
    raw: Same(--,10[oval 31x18];Red,Blue) 30.00



  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  other: '185-'
    raw: DREWRYSVILLE/Va.(185-;28;PAID/3[C];Black) 50.00

  color: 'Green'
    raw: Same(Green) 30.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  color: 'Green'
    raw: Same(Green) 30.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  color: 'Green'
    raw: Same(Green) 30.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  rate: '3[oval]'
    raw: Same(3[oval];Red) 25.00

  size: '--'
    raw: Same(--;STEAMBOAT;Black) See Inland Waterways --

  rate: 'X/Cts[C-24,green],PAID/5/Cts[DC-28,black]'
    raw: Same(X/Cts[C-24,green],PAID/5/Cts[DC-28,black]) 75.00

  color: 'Green'
    raw: Same(Green) 30.00

  

Last-position non-color fields (3+ field entries): 9 / 678


  pos=2 type=rate: 'DUE,Black'
  pos=3 type=size: '--'
  pos=3 type=other: 'Claret'
  pos=3 type=other: 'Red,Brownish'
  pos=2 type=size: '--'
  pos=2 type=size: '--'
  pos=3 type=size: '--'
  pos=2 type=size: '--'
  pos=2 type=size: '--'


## 5.8 Step 5 Summary

Field-type classification is content-signal-based, with no reliance on field position.
Position is used only as a validation cross-check (field 0 should be date, last field
should be color). Entries whose fields don't follow the expected positional pattern are
not reclassified -- they're flagged for review in Step 6.

At this point each listing row carries:
- **Structural segments** (Step 2): `seg_head`, `seg_paren`, `seg_tail`
- **Paren fields** (Step 3.1): `paren_fields`, `paren_field_count`
- **Tail parts** (Step 3.2-3.3): `tail_annotation`, `tail_valuation`, `valuation_tiers`
- **Head parts** (Step 4): `head_first_of_town`, `head_rel_type`, `head_name_body`, `head_annotations`
- **Field types** (Step 5): `paren_field_types`, `field_type_sig`
- **Section context**: `Default Shape`, `Page`, `Images Above`, `Institutional Ownership`

In [35]:
# --- Step 5.8: Summary ---

total = len(listings)
has_fields = listings[listings['paren_field_count'] > 0]
total_fields = field_df.shape[0]

print(f'Step 5: Paren Field-Type Classification')
print(f'  Listings with paren fields: {len(has_fields)} / {total}')
print(f'  Total fields classified: {total_fields}')
print()
print(f'  Type counts:')
for ftype, count in field_df['field_type'].value_counts().items():
    print(f'    {ftype}: {count} ({count/total_fields*100:.1f}%)')
print()
print(f'  Top type signatures:')
for sig, count in sig_counts.head(5).items():
    print(f'    {sig}: {count}')
print()
other_count = (field_df['field_type'] == 'other').sum()
print(f'  Unclassified ("other"): {other_count} ({other_count/total_fields*100:.1f}%)')

Step 5: Paren Field-Type Classification
  Listings with paren fields: 745 / 1537
  Total fields classified: 2468

  Type counts:
    color: 704 (28.5%)
    date: 694 (28.1%)
    size: 618 (25.0%)
    rate: 365 (14.8%)
    other: 47 (1.9%)
    ms: 40 (1.6%)

  Top type signatures:
    (none): 792
    date|size|rate|color: 304
    date|size|color: 253
    date|ms|color: 40
    date: 19

  Unclassified ("other"): 47 (1.9%)


# Step 6: Field-Level Sub-Parsing
 
Each classified paren field from Step 5 is decomposed into atomic components.
This is purely mechanical text decomposition — no cross-record logic, no
domain-entity assembly.
 
### Sub-parsers
 
| Type | Outputs |
|------|---------|
| `date` | month, day, year_start, year_end, granularity (DAY/MONTH/YEAR/DECADE/RANGE), is_circa |
| `size` | shape_code, dim1, dim2, dateformat, is_irregular, qualifier |
| `rate` | list of rate tokens, each with: keyword, amount, bracket_desc, is_manuscript |
| `color` | list of individual color names |
| `ms` | (no sub-parsing needed — flag consumed at assembly time) |
| `other` | attempted reclassification; survivors flagged for review |

In [36]:
# --- Step 6.1: Date sub-parser ---

MONTH_MAP = {
    'jan': 1, 'january': 1,
    'feb': 2, 'february': 2,
    'mar': 3, 'march': 3,
    'apr': 4, 'april': 4,
    'may': 5,
    'jun': 6, 'june': 6,
    'jul': 7, 'july': 7,
    'aug': 8, 'august': 8,
    'sep': 9, 'sept': 9, 'september': 9,
    'oct': 10, 'october': 10,
    'nov': 11, 'november': 11,
    'dec': 12, 'december': 12,
}

# Full date: Month day,year  e.g. "April 8,1800", "Oct.22,1803", "Feb. 2, 1818"
FULL_DATE_RE = re.compile(
    r'(' + MONTHS_PAT + r')\.?\s*'
    r'(\d{1,2})\s*,\s*'
    r'(\d{4})',
    re.IGNORECASE
)

# Month + year (no day): "April 1800"
MONTH_YEAR_RE = re.compile(
    r'(' + MONTHS_PAT + r')\.?\s+'
    r'(\d{4})',
    re.IGNORECASE
)

# Year range: "1850-53", "1842-52", "1850-1853"
YEAR_RANGE_RE = re.compile(
    r'c?(\d{4})\s*[-]\s*(\d{2,4})'
)

# Decade: "1850's", "1860's"
DECADE_RE = re.compile(r"c?(\d{4})'s", re.IGNORECASE)

# Bare year: "1852", possibly with circa prefix "c1840"
BARE_YEAR_RE = re.compile(r'c?(\d{4})$')

# Circa prefix
CIRCA_RE = re.compile(r'^c\d', re.IGNORECASE)


def parse_date_field(text):
    """Decompose a date-classified paren field into structured components."""
    t = text.strip()
    is_circa = bool(CIRCA_RE.match(t))

    # 1. Full date: Month day, year
    m = FULL_DATE_RE.search(t)
    if m:
        month_str = m.group(1).lower().rstrip('.')
        month = MONTH_MAP.get(month_str)
        day = int(m.group(2))
        year = int(m.group(3))
        return {
            'date_month': month,
            'date_day': day,
            'date_year_start': year,
            'date_year_end': year,
            'date_granularity': 'DAY',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 2. Decade: 1850's
    m = DECADE_RE.search(t)
    if m:
        base = int(m.group(1))
        return {
            'date_month': None,
            'date_day': None,
            'date_year_start': base,
            'date_year_end': base + 9,
            'date_granularity': 'DECADE',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 3. Year range: 1850-53
    m = YEAR_RANGE_RE.search(t)
    if m:
        y1 = int(m.group(1))
        y2_str = m.group(2)
        if len(y2_str) == 2:
            y2 = int(str(y1)[:2] + y2_str)
        else:
            y2 = int(y2_str)
        return {
            'date_month': None,
            'date_day': None,
            'date_year_start': y1,
            'date_year_end': y2,
            'date_granularity': 'RANGE',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 4. Month + year (no day)
    m = MONTH_YEAR_RE.search(t)
    if m:
        month_str = m.group(1).lower().rstrip('.')
        month = MONTH_MAP.get(month_str)
        year = int(m.group(2))
        return {
            'date_month': month,
            'date_day': None,
            'date_year_start': year,
            'date_year_end': year,
            'date_granularity': 'MONTH',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 5. Bare year
    m = BARE_YEAR_RE.search(t.lstrip('c'))
    if m:
        year = int(m.group(1))
        return {
            'date_month': None,
            'date_day': None,
            'date_year_start': year,
            'date_year_end': year,
            'date_granularity': 'YEAR',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    return {
        'date_month': None, 'date_day': None,
        'date_year_start': None, 'date_year_end': None,
        'date_granularity': None, 'date_is_circa': is_circa,
        'date_raw': t,
        'date_error': f'unparsed date: {t!r}',
    }


# Self-tests
assert parse_date_field('April 8,1800') == {
    'date_month': 4, 'date_day': 8, 'date_year_start': 1800,
    'date_year_end': 1800, 'date_granularity': 'DAY',
    'date_is_circa': False, 'date_raw': 'April 8,1800', 'date_error': None,
}
assert parse_date_field("1850's")['date_granularity'] == 'DECADE'
assert parse_date_field("1850's")['date_year_end'] == 1859
assert parse_date_field('1850-53')['date_year_end'] == 1853
assert parse_date_field('1852')['date_granularity'] == 'YEAR'
assert parse_date_field('c1840')['date_is_circa'] is True
assert parse_date_field('c1840')['date_year_start'] == 1840
assert parse_date_field('Oct.22,1803')['date_month'] == 10
print('Date sub-parser self-tests passed')

Date sub-parser self-tests passed


In [37]:
# --- Step 6.2: Size sub-parser ---

# Shape codes ordered longest-first
SHAPE_CODES = ['DLDC', 'DLDO', 'DLC', 'DLO', 'Octagon', 'Box', 'Arc',
               'Pmk', 'SL', 'DC', 'DO', 'NOR', 'O', 'C']
SHAPE_CODE_SET = {s.upper() for s in SHAPE_CODES}

SIZE_DATEFORMAT_CODES = {'YMDD', 'MDD', 'YMD', 'YD', 'MD'}

# Pattern: optional "irregular" prefix, shape code, optional dash, dimensions,
#          optional comma + dateformat/qualifier
SIZE_PARSE_RE = re.compile(
    r'^(irregular\s+)?'              # optional irregular prefix
    r'(' + SHAPE_CODE_PAT + r')?'    # optional shape code
    r'[\s\-]*'                       # separator
    r'('                             # dimension group
    r'  \d+\.?\d*\s*x\s*\d+\.?\d*'  # WxH
    r'  |\d+\.?\d*'                  # single diameter
    r'  |--?'                        # dash = unknown
    r')?'
    r'(?:\s*,\s*(.+))?'             # optional suffix (dateformat, qualifier)
    r'$',
    re.IGNORECASE | re.VERBOSE
)

# Match: <shape_code> ( '&' <word> )+ [ separator + rest ]
# Used to collapse ampersand-joined shape lists (e.g. "arc & SL-46x26" or
# "arc & SL,46") down to the first valid shape before the main parse.
_AMP_SHAPE_RE = re.compile(
    r'^(' + SHAPE_CODE_PAT + r')'           # first token: known shape
    r'((?:\s*&\s*[A-Za-z]+)+)'              # one or more '& word' alternatives
    r'(\s*[\s\-,].*)?$',                    # optional dimensions/suffix
    re.IGNORECASE
)


def _collapse_ampersand_shape(t):
    """If t looks like '<shape_a> & <shape_b> [& ...] <rest>', return
    '<first valid shape> <rest>'. Otherwise return t unchanged.
    A token is "valid" if its upper-cased form is in SHAPE_CODE_SET.

    The shape list may be separated from the dimensions by a dash or a
    comma ("arc & SL-46x26", "arc & SL,46"). A comma followed by a digit is
    normalized to a dash so the dimension parses as a dimension; a comma
    followed by a letter (suffix codes like ",YD") is left untouched.
    """
    if '&' not in t:
        return t
    m = _AMP_SHAPE_RE.match(t)
    if not m:
        return t
    first_token = m.group(1)
    alternatives = m.group(2)
    rest = m.group(3) or ''
    rest_stripped = rest.lstrip()
    if rest_stripped.startswith(',') and re.match(r'\s*\d', rest_stripped[1:]):
        rest = '-' + rest_stripped[1:].lstrip()
    # Ordered candidate list: first_token, then each '& word' alternative
    candidates = [first_token]
    candidates.extend(re.findall(r'&\s*([A-Za-z]+)', alternatives))
    for cand in candidates:
        if cand.upper() in SHAPE_CODE_SET:
            return cand + rest
    return t


def parse_size_field(text):
    """Decompose a size-classified paren field into components."""
    t = text.strip()

    # Catch bare dashes
    if t in ('-', '--'):
        return {
            'size_shape_code': None, 'size_dim1': None, 'size_dim2': None,
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t, 'size_error': None,
        }

    # Collapse ampersand-joined shape lists ("arc & SL-46x26" -> "arc-46x26")
    # before matching; size_raw below still records the original text.
    m = SIZE_PARSE_RE.match(_collapse_ampersand_shape(t))
    if not m:
        return {
            'size_shape_code': None, 'size_dim1': None, 'size_dim2': None,
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t,
            'size_error': f'unparsed size: {t!r}',
        }

    irregular_prefix = m.group(1)
    shape_raw = m.group(2)
    dim_raw = m.group(3)
    suffix_raw = m.group(4)

    is_irregular = bool(irregular_prefix)
    shape_code = shape_raw.upper() if shape_raw else None

    # Dimensions
    dim1, dim2 = None, None
    if dim_raw and dim_raw not in ('-', '--'):
        if 'x' in dim_raw.lower():
            parts = re.split(r'\s*x\s*', dim_raw, flags=re.IGNORECASE)
            dim1 = float(parts[0]) if parts[0] else None
            dim2 = float(parts[1]) if len(parts) > 1 and parts[1] else None
        else:
            dim1 = float(dim_raw)

    # Suffix: dateformat code, NOR, or free-text qualifier
    dateformat = None
    qualifier = None
    if suffix_raw:
        # May contain multiple tokens: "YD", "MDD", "NOR", "YMDD below"
        suffix_upper = suffix_raw.strip().upper()
        # Check if it starts with a known dateformat code
        for code in sorted(SIZE_DATEFORMAT_CODES, key=len, reverse=True):
            if suffix_upper.startswith(code):
                dateformat = code
                remainder = suffix_raw.strip()[len(code):].strip()
                if remainder:
                    qualifier = remainder
                break
        else:
            if suffix_upper == 'NOR':
                qualifier = 'NOR'
            else:
                qualifier = suffix_raw.strip()

    return {
        'size_shape_code': shape_code,
        'size_dim1': dim1,
        'size_dim2': dim2,
        'size_dateformat': dateformat,
        'size_is_irregular': is_irregular,
        'size_qualifier': qualifier,
        'size_raw': t,
        'size_error': None,
    }


# Self-tests
r = parse_size_field('SL-16.5x5,MDD')
assert r['size_shape_code'] == 'SL'
assert r['size_dim1'] == 16.5
assert r['size_dim2'] == 5.0
assert r['size_dateformat'] == 'MDD'

r = parse_size_field('DC-25,YD')
assert r['size_shape_code'] == 'DC'
assert r['size_dim1'] == 25.0
assert r['size_dateformat'] == 'YD'

r = parse_size_field('32')
assert r['size_shape_code'] is None
assert r['size_dim1'] == 32.0
assert r['size_dateformat'] is None

r = parse_size_field('--,YD')
assert r['size_dim1'] is None
assert r['size_dateformat'] == 'YD'

r = parse_size_field('DO-30x24')
assert r['size_shape_code'] == 'DO'
assert r['size_dim1'] == 30.0
assert r['size_dim2'] == 24.0

r = parse_size_field('SL-45x4,YMDD below')
assert r['size_shape_code'] == 'SL'
assert r['size_dateformat'] == 'YMDD'
assert r['size_qualifier'] == 'below'

# Ampersand-joined shape lists: pick the first valid shape.
# Dash-separated dimensions:
r = parse_size_field('arc & SL-46x26')
assert r['size_shape_code'] == 'ARC'
assert r['size_dim1'] == 46.0
assert r['size_dim2'] == 26.0
assert r['size_raw'] == 'arc & SL-46x26'

r = parse_size_field('arc&SL-46x26')
assert r['size_shape_code'] == 'ARC'
assert r['size_dim1'] == 46.0

# Comma-separated dimensions:
r = parse_size_field('arc & SL,46')
assert r['size_shape_code'] == 'ARC'
assert r['size_dim1'] == 46.0

r = parse_size_field('arc & SL,46x26')
assert r['size_shape_code'] == 'ARC'
assert r['size_dim1'] == 46.0
assert r['size_dim2'] == 26.0

# Comma followed by a dateformat code stays a suffix, not a dimension:
r = parse_size_field('arc & SL,YD')
assert r['size_shape_code'] == 'ARC'
assert r['size_dim1'] is None
assert r['size_dateformat'] == 'YD'

# Unknown first, known second: pick the known one
r = parse_size_field('Arc & nonsense-25')
assert r['size_shape_code'] == 'ARC'

# Existing behaviour preserved: no ampersand -> unchanged
r = parse_size_field('arc-46x26')
assert r['size_shape_code'] == 'ARC'

print('Size sub-parser self-tests passed')

Size sub-parser self-tests passed


## Step 6.3: Rate sub-parser

A rate paren field contains one or more rate tokens separated by commas.

Commas INSIDE brackets are not separators, e.g. "PAID/3[C]" is one token.

Each token is a distinct rate marking or rate value observed on the postmark.

Token anatomy examples:
  * PAID          -> keyword=PAID, amount=None, bracket=None

  * FREE          -> keyword=FREE, amount=None, bracket=None
 
  * PAID 3        -> keyword=PAID, amount=3, bracket=None
 
  * PAID/3[C]     -> keyword=PAID, amount=3, bracket=C
 
  * 5[C]          -> keyword=None, amount=5, bracket=C
 
  * 25[ms]        -> keyword=None, amount=25, bracket=ms (manuscript rate)
 
  * P.M.Free      -> keyword=PM_FREE, amount=None, bracket=None
 
  * Geo.Fisher P.M.frank -> keyword=PM_FRANK, amount=None, bracket=None
 
  * STEAM         -> keyword=STEAM, amount=None, bracket=None
 
  * DUE 3         -> keyword=DUE, amount=3, bracket=None
 
  * with 24       -> keyword=WITH_ADHESIVE, amount=24, bracket=None
 
  * X             -> keyword=None, amount=None, bracket=None, roman='X'
 
  * negative 5[C] -> keyword=None, amount=5, bracket=C, impression=Negative
  

In [38]:

def split_rate_tokens(field_text):
    """Split rate field on commas, respecting brackets.
    Returns list of raw token strings."""
    tokens = []
    current = []
    depth = 0
    for ch in field_text:
        if ch == '[':
            depth += 1
            current.append(ch)
        elif ch == ']':
            depth -= 1
            current.append(ch)
        elif ch == ',' and depth == 0:
            tokens.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    if current:
        tokens.append(''.join(current).strip())
    return [t for t in tokens if t]


RATE_AMOUNT_RE = re.compile(
    r'(\d+(?:[/-]\d+(?:/\d+)?)?)'  # amount: "3", "12-1/2", "3/CENTS"
)

RATE_BRACKET_RE = re.compile(r'\[([^\]]+)\]')

RATE_KEYWORD_RE = re.compile(
    r'\b(PAID|FREE|STEAM|DUE)\b', re.IGNORECASE
)

PM_RE = re.compile(r'P\.?M\.?\s*(Free|frank)', re.IGNORECASE)

NEGATIVE_RE = re.compile(r'^negative\s+', re.IGNORECASE)

ROMAN_RE = re.compile(r'^[IVXLCDM]+$')


def parse_rate_token(tok):
    """Parse a single rate token into structured components."""
    t = tok.strip()

    result = {
        'rate_keyword': None,
        'rate_amount_raw': None,
        'rate_bracket': None,
        'rate_is_manuscript': False,
        'rate_impression': None,
        'rate_raw': t,
    }

    # Check for negative impression prefix
    neg_m = NEGATIVE_RE.match(t)
    if neg_m:
        result['rate_impression'] = 'Negative'
        t = t[neg_m.end():]

    # P.M. notation
    pm_m = PM_RE.search(t)
    if pm_m:
        pm_type = pm_m.group(1).lower()
        if pm_type == 'free':
            result['rate_keyword'] = 'PM_FREE'
        else:
            result['rate_keyword'] = 'PM_FRANK'
        # May have trailing rate: "P.M.Free-Paid 10"
        remainder = t[pm_m.end():].strip().lstrip('-')
        if remainder:
            kw_m = RATE_KEYWORD_RE.search(remainder)
            if kw_m:
                result['rate_keyword'] = 'PM_FREE'  # compound; keep PM_FREE
                amt_after = remainder[kw_m.end():].strip()
                if amt_after:
                    amt_m = RATE_AMOUNT_RE.search(amt_after)
                    if amt_m:
                        result['rate_amount_raw'] = amt_m.group(1)
        return result

    # Bracket: [ms], [C], [F], [box], etc.
    br_m = RATE_BRACKET_RE.search(t)
    if br_m:
        bracket_val = br_m.group(1).strip()
        if bracket_val.lower() == 'ms':
            result['rate_is_manuscript'] = True
        else:
            result['rate_bracket'] = bracket_val

    # Keyword: PAID, FREE, STEAM, DUE
    kw_m = RATE_KEYWORD_RE.search(t)
    if kw_m:
        result['rate_keyword'] = kw_m.group(1).upper()

    # "with NN" = adhesive
    with_m = re.search(r'\bwith\s+(\d+)', t, re.IGNORECASE)
    if with_m:
        result['rate_keyword'] = 'WITH_ADHESIVE'
        result['rate_amount_raw'] = with_m.group(1)
        return result

    # Amount: first numeric sequence not inside a keyword or bracket-only context
    # Strip bracket content and keyword to find rate amount
    t_stripped = RATE_BRACKET_RE.sub('', t)
    t_stripped = RATE_KEYWORD_RE.sub('', t_stripped)
    t_stripped = PM_RE.sub('', t_stripped)
    t_stripped = t_stripped.replace('/', ' ').strip()
    amt_m = RATE_AMOUNT_RE.search(t_stripped)
    if amt_m:
        result['rate_amount_raw'] = amt_m.group(1)

    # Roman numeral check (V, X, etc.) when no other signal
    if (result['rate_keyword'] is None and result['rate_amount_raw'] is None
            and not result['rate_is_manuscript']):
        clean = RATE_BRACKET_RE.sub('', t).strip()
        if ROMAN_RE.match(clean):
            result['rate_amount_raw'] = clean

    return result


def parse_rate_field(text):
    """Decompose a rate-classified paren field into a list of parsed tokens."""
    tokens = split_rate_tokens(text)
    return [parse_rate_token(t) for t in tokens]


# Self-tests
assert split_rate_tokens('PAID/3[C],FREE') == ['PAID/3[C]', 'FREE']
assert split_rate_tokens('5,10,PAID') == ['5', '10', 'PAID']
assert split_rate_tokens('25,12-1/2[ms]Black') == ['25', '12-1/2[ms]Black']

r = parse_rate_token('PAID/3[C]')
assert r['rate_keyword'] == 'PAID'
assert r['rate_amount_raw'] == '3'
assert r['rate_bracket'] == 'C'

r = parse_rate_token('25[ms]')
assert r['rate_amount_raw'] == '25'
assert r['rate_is_manuscript'] is True

r = parse_rate_token('FREE')
assert r['rate_keyword'] == 'FREE'
assert r['rate_amount_raw'] is None

r = parse_rate_token('P.M.Free')
assert r['rate_keyword'] == 'PM_FREE'

r = parse_rate_token('with 24')
assert r['rate_keyword'] == 'WITH_ADHESIVE'
assert r['rate_amount_raw'] == '24'

r = parse_rate_token('negative 5[C]')
assert r['rate_impression'] == 'Negative'
assert r['rate_amount_raw'] == '5'
assert r['rate_bracket'] == 'C'

r = parse_rate_token('STEAM')
assert r['rate_keyword'] == 'STEAM'

print('Rate sub-parser self-tests passed')

Rate sub-parser self-tests passed


In [39]:
# --- Step 6.4: Color sub-parser ---

def parse_color_field(text):
    """Split a color field into individual normalized color names (UPPER)."""
    tokens = [t.strip().upper() for t in text.split(',') if t.strip()]
    return tokens


# Self-tests
assert parse_color_field('Black') == ['BLACK']
assert parse_color_field('Red,Blue,Black') == ['RED', 'BLUE', 'BLACK']
assert parse_color_field('Olive-Yellow') == ['OLIVE-YELLOW']
print('Color sub-parser self-tests passed')

Color sub-parser self-tests passed


In [40]:
# --- Step 6.5: Other-field triage ---
#
# Attempt to reclassify "other" fields that slipped through Step 5.
# Known reclassification patterns:
#   "185-", "186-", "183-51" -> truncated dates (date-like)
#   "DC--", "DLC--", "arc--" -> size with unknown dimension
#   "5,10", "5,V,10", "V,X"  -> rate (bare amounts/roman without keywords)
#   "12-1/2"                  -> rate (fractional cent amount)
#   "Double 50"               -> rate (descriptive)
#   "large 5,10"              -> rate (with qualifier)
#   "fancy shaded 10"         -> rate (with qualifier)
#   "irregular 34"            -> size (with irregular prefix)
#   "30,32"                   -> size (multiple dimensions)
#   "Red,Purple,Blue,Brownish" -> color (unknown color term "Brownish")

TRUNCATED_DATE_RE = re.compile(r'^\d{3}-\d{0,2}$')
SIZE_WITH_DASH_RE = re.compile(
    r'^(?:' + SHAPE_CODE_PAT + r'|arc)[\s\-]*-{1,2}$', re.IGNORECASE
)
BARE_RATE_RE = re.compile(
    r'^(?:(?:large|fancy|shaded|Double|small)\s+)?'
    r'(?:\d+(?:-\d+(?:/\d+)?)?|[IVXLCDM]+)'
    r'(?:\s*,\s*(?:\d+(?:-\d+(?:/\d+)?)?|[IVXLCDM]+))*$'
)
IRREGULAR_SIZE_RE = re.compile(r'^irregular\s+\d', re.IGNORECASE)
MULTI_DIM_RE = re.compile(r'^\d{2,3}\s*,\s*\d{2,3}$')


def triage_other_field(text):
    """Attempt reclassification of an 'other' field.
    Returns (new_type, parsed_result) or ('other', None) if unresolvable."""
    t = text.strip()

    # Truncated date: "185-", "186-", "183-51"
    if TRUNCATED_DATE_RE.match(t):
        # Treat as approximate date range
        prefix = t.split('-')[0]
        suffix = t.split('-')[1] if '-' in t else ''
        if len(prefix) == 3:
            decade_base = int(prefix + '0')
            if suffix and suffix.isdigit():
                year_end = int(prefix + suffix) if len(suffix) == 1 else int('1' + suffix) if len(suffix) == 2 else decade_base + 9
            else:
                year_end = decade_base + 9
            return 'date', {
                'date_month': None, 'date_day': None,
                'date_year_start': decade_base,
                'date_year_end': year_end,
                'date_granularity': 'RANGE',
                'date_is_circa': False,
                'date_raw': t,
                'date_error': 'reclassified from other (truncated date)',
            }

    # Size with unknown dim: "DC--", "DLC--", "arc--"
    if SIZE_WITH_DASH_RE.match(t):
        # Extract shape code
        shape = re.match(r'^([A-Za-z]+)', t).group(1).upper()
        return 'size', {
            'size_shape_code': shape,
            'size_dim1': None, 'size_dim2': None,
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t, 'size_error': None,
        }

    # Irregular size: "irregular 34"
    if IRREGULAR_SIZE_RE.match(t):
        return 'size', parse_size_field(t)

    # Multi-dimension: "30,32"
    if MULTI_DIM_RE.match(t):
        dims = t.split(',')
        return 'size', {
            'size_shape_code': None,
            'size_dim1': float(dims[0].strip()),
            'size_dim2': float(dims[1].strip()),
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t, 'size_error': None,
        }

    # Bare rate amounts or roman+amount combos: "5,10", "12-1/2", "V,X", "Double 50"
    if BARE_RATE_RE.match(t):
        return 'rate', parse_rate_field(t)

    # Color with unknown terms (partial match)
    tokens = [tok.strip() for tok in t.split(',') if tok.strip()]
    known_count = sum(1 for tok in tokens if is_color_token(tok))
    if known_count > 0 and known_count >= len(tokens) - 1:
        # At least one unknown term but majority are colors -> reclassify
        return 'color', [t.upper() for t in tokens]

    return 'other', None


# Self-tests
assert triage_other_field('185-')[0] == 'date'
assert triage_other_field('186-')[0] == 'date'
assert triage_other_field('DC--')[0] == 'size'
assert triage_other_field('DLC--')[0] == 'size'
assert triage_other_field('5,10')[0] == 'rate'
assert triage_other_field('12-1/2')[0] == 'rate'
assert triage_other_field('irregular 34')[0] == 'size'
assert triage_other_field('30,32')[0] == 'size'
assert triage_other_field('Red,Purple,Blue,Brownish')[0] == 'color'
assert triage_other_field('Double 50')[0] == 'rate'
print('Other-field triage self-tests passed')

Other-field triage self-tests passed


In [41]:
# --- Step 6.6: Apply sub-parsers to all classified paren fields ---

_MS_TRUTHY = {'1', 'true', 'yes', 'y', 't'}


def _csv_manuscript_truthy(row):
    """Return True if the optional Manuscript column is present and truthy for this row."""
    val = row.get('Manuscript')
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    return str(val).strip().lower() in _MS_TRUTHY


def subparse_fields(row):
    """Apply the appropriate sub-parser to each paren field based on its type.
    Returns parallel lists: parsed_dates, parsed_sizes, parsed_rates, parsed_colors,
    plus is_manuscript flag and other_fields list.

    is_manuscript is derived from paren `(ms)` fields, then *unioned* with the
    optional per-row `Manuscript` CSV column (truthy values promote; the column
    cannot demote a paren-detected manuscript).
    """
    fields = row['paren_fields']
    types = row['paren_field_types']

    parsed_dates = []
    parsed_sizes = []
    parsed_rates = []
    parsed_colors = []
    is_manuscript = False
    other_fields = []
    reclassified = []

    for i, (field, ftype) in enumerate(zip(fields, types)):
        if ftype == 'ms':
            is_manuscript = True
        elif ftype == 'date':
            parsed_dates.append(parse_date_field(field))
        elif ftype == 'size':
            parsed_sizes.append(parse_size_field(field))
        elif ftype == 'rate':
            parsed_rates.append(parse_rate_field(field))
        elif ftype == 'color':
            parsed_colors.extend(parse_color_field(field))
        elif ftype == 'other':
            new_type, parsed = triage_other_field(field)
            if new_type != 'other':
                reclassified.append({
                    'position': i, 'original_type': 'other',
                    'new_type': new_type, 'field': field,
                })
                if new_type == 'date':
                    parsed_dates.append(parsed)
                elif new_type == 'size':
                    parsed_sizes.append(parsed)
                elif new_type == 'rate':
                    if isinstance(parsed, list):
                        parsed_rates.append(parsed)
                    else:
                        parsed_rates.append([parsed])
                elif new_type == 'color':
                    parsed_colors.extend(parsed)
            else:
                other_fields.append(field)

    # Union the optional CSV `Manuscript` column (if present + truthy).
    if _csv_manuscript_truthy(row):
        is_manuscript = True

    return pd.Series({
        'parsed_dates': parsed_dates,
        'parsed_sizes': parsed_sizes,
        'parsed_rates': parsed_rates,
        'parsed_colors': parsed_colors,
        'is_manuscript': is_manuscript,
        'other_fields': other_fields,
        'reclassified_fields': reclassified,
    })


parsed = listings.apply(subparse_fields, axis=1)
listings = pd.concat([listings, parsed], axis=1)

print('Step 6: Field-level sub-parsing applied')
print(f'  Listings processed: {len(listings)}')
print(f'  Manuscript entries: {listings["is_manuscript"].sum()}')
if 'Manuscript' in listings.columns:
    csv_ms_hits = listings.apply(_csv_manuscript_truthy, axis=1).sum()
    print(f'    (of which CSV Manuscript column contributed: {csv_ms_hits})')
print(f'  Entries with dates: {listings["parsed_dates"].apply(len).gt(0).sum()}')
print(f'  Entries with sizes: {listings["parsed_sizes"].apply(len).gt(0).sum()}')
print(f'  Entries with rates: {listings["parsed_rates"].apply(len).gt(0).sum()}')
print(f'  Entries with colors: {listings["parsed_colors"].apply(len).gt(0).sum()}')
total_reclassified = listings['reclassified_fields'].apply(len).sum()
print(f'  Other fields reclassified: {total_reclassified}')
remaining_other = listings['other_fields'].apply(len).sum()
print(f'  Remaining unresolved other fields: {remaining_other}')


Step 6: Field-level sub-parsing applied


  Listings processed: 1537
  Manuscript entries: 831
    (of which CSV Manuscript column contributed: 791)
  Entries with dates: 695
  Entries with sizes: 611
  Entries with rates: 366
  Entries with colors: 707
  Other fields reclassified: 20
  Remaining unresolved other fields: 27


In [42]:
# --- Step 6.7: Audit sub-parsing errors ---

# Date parse errors
date_errors = []
for idx, row in listings.iterrows():
    for d in row['parsed_dates']:
        if d.get('date_error') and 'reclassified' not in str(d.get('date_error', '')):
            date_errors.append((idx, d['date_raw'], d['date_error']))

print(f'Date parse errors: {len(date_errors)}')
for idx, raw, err in date_errors[:10]:
    print(f'  [{idx}] {raw!r}: {err}')
print()

# Size parse errors
size_errors = []
for idx, row in listings.iterrows():
    for s in row['parsed_sizes']:
        if s.get('size_error'):
            size_errors.append((idx, s['size_raw'], s['size_error']))

print(f'Size parse errors: {len(size_errors)}')
for idx, raw, err in size_errors[:10]:
    print(f'  [{idx}] {raw!r}: {err}')
print()

# Reclassification audit
reclass_all = []
for idx, row in listings.iterrows():
    for rc in row['reclassified_fields']:
        reclass_all.append((idx, rc['field'], rc['new_type']))

print(f'Reclassified other fields: {len(reclass_all)}')
for idx, field, new_type in reclass_all[:15]:
    print(f'  [{idx}] {field!r} -> {new_type}')
print()

# Remaining unresolved
unresolved = []
for idx, row in listings.iterrows():
    for f in row['other_fields']:
        unresolved.append((idx, f, row['clean_text'][:100]))

print(f'Unresolved other fields: {len(unresolved)}')
for idx, field, raw in unresolved:
    print(f'  [{idx}] {field!r}')
    print(f'    raw: {raw}')
    print()

Date parse errors: 1
  [298] '1857,--': unparsed date: '1857,--'



Size parse errors: 20
  [127] 'DL box 54x30': unparsed size: 'DL box 54x30'
  [183] 'MDD': unparsed size: 'MDD'
  [234] 'truncated box 29x23': unparsed size: 'truncated box 29x23'
  [250] 'DC-28-14': unparsed size: 'DC-28-14'
  [289] 'very ornate O-44x19': unparsed size: 'very ornate O-44x19'
  [311] 'stencil C-25,NOR': unparsed size: 'stencil C-25,NOR'
  [339] 'hollow letter arc 25x5-7': unparsed size: 'hollow letter arc 25x5-7'
  [386] 'rough C-26-28': unparsed size: 'rough C-26-28'
  [402] 'ornate O-46x27,doves at top': unparsed size: 'ornate O-46x27,doves at top'
  [446] 'rough C-27': unparsed size: 'rough C-27'

Reclassified other fields: 20
  [25] 'Ms,Black' -> color
  [41] 'L' -> rate
  [51] 'Ms,Black' -> color
  [61] 'L' -> rate
  [74] 'L' -> rate
  [82] 'L' -> rate
  [85] 'L' -> rate
  [181] '31,32' -> size
  [216] '5,3' -> rate
  [229] '185-' -> date
  [281] 'SL--' -> size
  [296] 'DC--' -> size
  [366] '30,33' -> size
  [401] 'O--' -> size
  [452] 'Red,Brownish' -> color

Un

In [43]:
# --- Step 6.8: Parsed value distributions ---

# Date granularity
all_dates = [d for dlist in listings['parsed_dates'] for d in dlist]
if all_dates:
    gran_counts = pd.Series([d['date_granularity'] for d in all_dates]).value_counts()
    print('Date granularity distribution:')
    for g, c in gran_counts.items():
        print(f'  {g}: {c}')
    print()

# Size shape codes
all_sizes = [s for slist in listings['parsed_sizes'] for s in slist]
if all_sizes:
    shape_counts = pd.Series([s['size_shape_code'] or '(bare dimension)' for s in all_sizes]).value_counts()
    print('Size shape code distribution:')
    for s, c in shape_counts.items():
        print(f'  {s}: {c}')
    print()

    # Dateformat on size
    df_counts = pd.Series([s['size_dateformat'] or '(none)' for s in all_sizes]).value_counts()
    print('Size dateformat distribution:')
    for d, c in df_counts.items():
        print(f'  {d}: {c}')
    print()

# Rate keywords
all_rate_tokens = [t for rlist in listings['parsed_rates'] for toks in rlist for t in toks]
if all_rate_tokens:
    kw_counts = pd.Series([t['rate_keyword'] or '(bare amount)' for t in all_rate_tokens]).value_counts()
    print('Rate keyword distribution:')
    for k, c in kw_counts.items():
        print(f'  {k}: {c}')
    print()

    ms_rate = sum(1 for t in all_rate_tokens if t['rate_is_manuscript'])
    print(f'Manuscript rate tokens: {ms_rate}')
    neg_rate = sum(1 for t in all_rate_tokens if t['rate_impression'] == 'Negative')
    print(f'Negative-impression rate tokens: {neg_rate}')
    print()

# Color names
all_colors = [c for clist in listings['parsed_colors'] for c in clist]
if all_colors:
    color_counts = pd.Series(all_colors).value_counts()
    print('Color distribution:')
    for c, n in color_counts.items():
        print(f'  {c}: {n}')
    print()

Date granularity distribution:
  RANGE: 324
  YEAR: 269
  DAY: 70
  DECADE: 31

Size shape code distribution:
  (bare dimension): 487
  SL: 90
  DC: 22
  O: 12
  DO: 4
  C: 2
  BOX: 2
  ARC: 2
  DLO: 2
  DLC: 1
  DLDC: 1

Size dateformat distribution:
  (none): 559
  MDD: 47
  YMDD: 14
  YD: 4
  MD: 1

Rate keyword distribution:
  PAID: 284
  (bare amount): 280
  FREE: 65
  DUE: 10
  PM_FRANK: 1
  STEAM: 1

Manuscript rate tokens: 1
Negative-impression rate tokens: 0

Color distribution:
  BLACK: 500
  RED: 209
  BLUE: 128
  GREEN: 44
  BROWN: 13
  ORANGE: 6
  MAGENTA: 3
  MS: 2
  BROWN-BLACK: 2
  BROWN-RED: 1
  BLACK-BROWN: 1
  BROWNISH: 1
  RED-ORANGE: 1
  BLUE-GREEN: 1
  VIOLET: 1
  PURPLE: 1



## 6.9 Step 6 Summary
 
Step 6 decomposes each typed paren field into atomic components, adding seven
new columns to the `listings` DataFrame:
 
- **parsed_dates** -- list of date dicts (month, day, year_start, year_end, granularity, is_circa)
- **parsed_sizes** -- list of size dicts (shape_code, dim1, dim2, dateformat, is_irregular, qualifier)
- **parsed_rates** -- list of rate token lists, each token with (keyword, amount_raw, bracket, is_manuscript, impression)
- **parsed_colors** -- flat list of individual color names
- **is_manuscript** -- boolean (from Ms field)
- **other_fields** -- list of any unresolved fields
- **reclassified_fields** -- audit trail of other-to-type reclassifications
 
No cross-record logic. No domain entity assembly. That begins in Step 7
(relationship resolution).

In [44]:
# --- Step 6.9: Summary ---

total = len(listings)
print('Step 6: Field-Level Sub-Parsing')
print(f'  Listings processed: {total}')
print()
print(f'  Dates parsed:        {sum(len(d) for d in listings["parsed_dates"])}')
print(f'  Sizes parsed:        {sum(len(s) for s in listings["parsed_sizes"])}')
print(f'  Rate fields parsed:  {sum(len(r) for r in listings["parsed_rates"])}')
print(f'    Rate tokens total: {sum(len(t) for rlist in listings["parsed_rates"] for t in rlist)}')
print(f'  Colors extracted:    {sum(len(c) for c in listings["parsed_colors"])}')
print(f'  Manuscript entries:  {listings["is_manuscript"].sum()}')
print()
print(f'  Reclassified others: {sum(len(r) for r in listings["reclassified_fields"])}')
print(f'  Unresolved others:   {sum(len(o) for o in listings["other_fields"])}')

Step 6: Field-Level Sub-Parsing
  Listings processed: 1537

  Dates parsed:        695
  Sizes parsed:        625
  Rate fields parsed:  374
    Rate tokens total: 641
  Colors extracted:    914
  Manuscript entries:  831

  Reclassified others: 20
  Unresolved others:   27


# Step 7: Relationship Resolution

Walks listings in catalog order, resolves `Same`/`(L)`/`(E)` inheritance so
every record has a fully resolved inscription text and town root.

First step with cross-record dependencies.

### Rules

1. **Independent entry** (`head_rel_type` is null): inscription = `head_name_body`;
   town root = everything before the first `/` (or the entire string if no `/`).
   Becomes the active parent for subsequent rel entries.

2. **`Same` with no name body**: inherit parent inscription and town.

3. **`Same` with name body**: different device, same town. Name body should start
   with `/` (state abbreviation variant). Resolved inscription = parent town root +
   child name body. Flagged if name body does not start with `/`.

4. **`(L)` or `(E)`**: later / earlier observation of same device. Inherit parent
   inscription and town.

Parent resolution: walk backward unconditionally to nearest preceding independent
entry. No section-boundary guard. Cross-section inheritance is flagged as a warning.

### Outputs

- **parent_idx** (nullable int) -- DataFrame index of resolved parent. Null for independent entries.
- **resolved_inscription** (str) -- Inscription text on the physical device.
- **resolved_town** (str) -- Town root for PostOffice grouping (pre-`/` text, unstripped).
- **s7_warnings** (list of str) -- Confidence-lowering flags for this record.

### Warning codes

- `orphan_rel` -- rel entry with no preceding independent entry found
- `independent_no_name` -- independent entry has null name body
- `same_name_body_no_slash` -- Same entry has name body not starting with `/`
- `cross_section_parent` -- parent is in a different Default Shape section

In [45]:
# --- Step 7.1: Relationship resolution ---

def extract_town_root(inscription):
    """Town root = everything before the first '/', or whole string if no '/'."""
    if inscription is None or (isinstance(inscription, float) and pd.isna(inscription)):
        return ''
    if '/' in inscription:
        return inscription.split('/')[0]
    return inscription

def resolve_relationships(listings_df):
    """Walk listings in catalog order, resolve inheritance.
    
    Modifies listings_df in place, adding:
      parent_idx, resolved_inscription, resolved_town, s7_warnings
    """
    n = len(listings_df)
    parent_idx = [None] * n
    resolved_inscription = [None] * n
    resolved_town = [None] * n
    s7_warnings = [[] for _ in range(n)]

    # Track the most recent independent entry by iteration position
    current_parent_pos = None

    for pos in range(n):
        row = listings_df.iloc[pos]
        warnings = []

        if pd.isna(row['head_rel_type']) or row['head_rel_type'] is None:
            # --- Independent entry ---
            inscription = row['head_name_body']
            if inscription is None or (isinstance(inscription, float) and pd.isna(inscription)):
                warnings.append('independent_no_name')
                inscription = ''

            town = extract_town_root(inscription)

            parent_idx[pos] = None
            resolved_inscription[pos] = inscription
            resolved_town[pos] = town
            current_parent_pos = pos

        else:
            # --- Relationship entry ---
            if current_parent_pos is None:
                warnings.append('orphan_rel')
                # Best-effort: use own name body if any
                _nb = row['head_name_body']
                fallback = '' if (_nb is None or (isinstance(_nb, float) and pd.isna(_nb))) else (_nb or '')
                parent_idx[pos] = None
                resolved_inscription[pos] = fallback
                resolved_town[pos] = extract_town_root(fallback) if fallback else ''
            else:
                parent_idx[pos] = listings_df.index[current_parent_pos]
                p_inscription = resolved_inscription[current_parent_pos]
                p_town = resolved_town[current_parent_pos]

                rel = row['head_rel_type']
                name_body = row['head_name_body']

                if rel == 'Same' and pd.notna(name_body):
                    # Different device, same town: reconstruct inscription
                    if not name_body.startswith('/'):
                        warnings.append('same_name_body_no_slash')
                    resolved_inscription[pos] = p_town + name_body
                    resolved_town[pos] = p_town
                else:
                    # Same device (Same w/o name, (L), (E)): inherit
                    resolved_inscription[pos] = p_inscription
                    resolved_town[pos] = p_town

                # Cross-section check
                parent_row = listings_df.iloc[current_parent_pos]
                if row.get('Default Shape') != parent_row.get('Default Shape'):
                    warnings.append('cross_section_parent')

        s7_warnings[pos] = warnings

    listings_df['parent_idx'] = parent_idx
    listings_df['resolved_inscription'] = resolved_inscription
    listings_df['resolved_town'] = resolved_town
    listings_df['s7_warnings'] = s7_warnings
    return listings_df

listings = resolve_relationships(listings)

print(f'Step 7: Relationship resolution applied to {len(listings)} listings')
print(f'  Independent entries: {listings["parent_idx"].isna().sum()}')
print(f'  Resolved from parent: {listings["parent_idx"].notna().sum()}')
print(f'  Distinct resolved towns: {listings["resolved_town"].nunique()}')

# --- Step 7.1b: Attribute inheritance ---
#
# For child records (those with parent_idx), inherit parsed attributes
# from the parent when the child's own paren body is silent on them.
#
# Rules:
#   is_manuscript : inherit if child has no 'ms' or 'size' in paren_field_types
#   parsed_colors : inherit if child's parsed_colors is empty
#   parsed_sizes  : inherit if child's parsed_sizes is empty
#
# Walk in catalog order so that transitive chains (unlikely but safe)
# see already-inherited parent values.

inherited_ms_count = 0
inherited_color_count = 0
inherited_size_count = 0

for pos in range(len(listings)):
    row = listings.iloc[pos]
    if pd.isna(row['parent_idx']):
        continue

    parent = listings.loc[row['parent_idx']]
    child_types = set(row['paren_field_types'])

    # is_manuscript
    if 'ms' not in child_types and 'size' not in child_types:
        if parent['is_manuscript'] != row['is_manuscript']:
            listings.iat[pos, listings.columns.get_loc('is_manuscript')] = parent['is_manuscript']
            inherited_ms_count += 1

    # parsed_colors
    if not row['parsed_colors']:
        if parent['parsed_colors']:
            listings.iat[pos, listings.columns.get_loc('parsed_colors')] = parent['parsed_colors'].copy()
            inherited_color_count += 1

    # parsed_sizes
    if not row['parsed_sizes']:
        if parent['parsed_sizes']:
            listings.iat[pos, listings.columns.get_loc('parsed_sizes')] = parent['parsed_sizes'].copy()
            inherited_size_count += 1

print()
print('Step 7.1b: Attribute inheritance')
print(f'  is_manuscript inherited:  {inherited_ms_count}')
print(f'  parsed_colors inherited:  {inherited_color_count}')
print(f'  parsed_sizes inherited:   {inherited_size_count}')


Step 7: Relationship resolution applied to 1537 listings
  Independent entries: 1282
  Resolved from parent: 255
  Distinct resolved towns: 1232



Step 7.1b: Attribute inheritance
  is_manuscript inherited:  10
  parsed_colors inherited:  30
  parsed_sizes inherited:   60


In [46]:
# --- Step 7.2: Resolution examples by rel_type ---

for rt in [None, 'Same', '(L)', '(E)']:
    if rt is None:
        subset = listings[listings['head_rel_type'].isna()]
        label = '(independent)'
    else:
        subset = listings[listings['head_rel_type'] == rt]
        label = rt
    print(f'=== rel_type={label} ({len(subset)} entries) ===')
    for _, row in subset.head(6).iterrows():
        parent_info = ''
        if pd.notna(row['parent_idx']):
            p = listings.loc[row['parent_idx']]
            parent_info = f'  parent_head={p["seg_head"]!r}'
        print(f'  seg_head={row["seg_head"]!r}')
        print(f'    -> inscription={row["resolved_inscription"]!r}  town={row["resolved_town"]!r}{parent_info}')
        if row['s7_warnings']:
            print(f'    WARNINGS: {row["s7_warnings"]}')
    print()

# Same-with-name-body examples
same_with_name = listings[
    (listings['head_rel_type'] == 'Same') & listings['head_name_body'].notna()
]
print(f'=== Same with name body ({len(same_with_name)} entries) ===')
for _, row in same_with_name.head(10).iterrows():
    if pd.notna(row['parent_idx']):
        p = listings.loc[row['parent_idx']]
        print(f'  head={row["seg_head"]!r}  name_body={row["head_name_body"]!r}')
        print(f'    parent inscription={p["resolved_inscription"]!r}  parent town={p["resolved_town"]!r}')
        print(f'    -> resolved={row["resolved_inscription"]!r}  town={row["resolved_town"]!r}')
        if row['s7_warnings']:
            print(f'    WARNINGS: {row["s7_warnings"]}')
        print()
    else:
        print(f'  head={row["seg_head"]!r}  name_body={row["head_name_body"]!r}  *** ORPHAN ***')


=== rel_type=(independent) (1282 entries) ===
  seg_head='Alexa.(Alexandria)(E)'
    -> inscription='Alexa.'  town='Alexa.'
  seg_head='Colchester'
    -> inscription='Colchester'  town='Colchester'
  seg_head='Fredericksburg'
    -> inscription='Fredericksburg'  town='Fredericksburg'
  seg_head='Fredg.'
    -> inscription='Fredg.'  town='Fredg.'
  seg_head='Frk sg(Fredericksburg)'
    -> inscription='Frk sg'  town='Frk sg'
  seg_head='FREDERICKSBURG("F"5mm high,used as bkstp)'
    -> inscription='FREDERICKSBURG'  town='FREDERICKSBURG'

=== rel_type=Same (233 entries) ===
  seg_head='Same'
    -> inscription='Nfk'  town='Nfk'  parent_head='Nfk(Norfolk)'
    WARNINGS: ['cross_section_parent']
  seg_head='Same'
    -> inscription='NORFOLK'  town='NORFOLK'  parent_head='NORFOLK(backstamp)(E)'
    WARNINGS: ['cross_section_parent']
  seg_head='Same'
    -> inscription='NORFOLK'  town='NORFOLK'  parent_head='NORFOLK(backstamp)(E)'
    WARNINGS: ['cross_section_parent']
  seg_head='Same'
   

In [47]:
# --- Step 7.3: Validation ---

# V1: Every listing must have resolved_inscription and resolved_town
missing_inscription = listings['resolved_inscription'].isna() | (listings['resolved_inscription'] == '')
missing_town = listings['resolved_town'].isna() | (listings['resolved_town'] == '')
print(f'V1: Missing resolved_inscription: {missing_inscription.sum()}')
print(f'V1: Missing resolved_town: {missing_town.sum()}')
# Missing town is allowed: some markings (e.g. no-town CDS, manuscript-only) have
# no extractable town and still yield a valid postmark -- warn, don't block.
# Missing inscription, however, means we have no identifying head text at all;
# that row cannot become a usable postmark record.
if missing_town.any():
    town_problem = listings[missing_town]
    town_cols = [c for c in ['Postmark Key', 'head_rel_type', 'clean_text',
                             'resolved_inscription'] if c in town_problem.columns]
    town_view = town_problem[town_cols].head(30).copy()
    town_view.insert(0, 'csv_line', town_view.index + 2)
    print(f'WARNING: {len(town_problem)} listing(s) have no resolved_town '
          f'(expected for no-town markings; will fall back to UNKNOWN PostOffice).')
    print('csv_line = line number in the input CSV')
    print(town_view.to_string(index=False))

# Empty-string inscription is valid for no-town CDS entries:
# Step 8.8 buckets them under an UNKNOWN PostOffice. Only null (NaN)
# indicates a genuine processing failure where the row was never assigned.
null_inscription = listings['resolved_inscription'].isna()
empty_inscription = (~null_inscription) & (listings['resolved_inscription'] == '')
if empty_inscription.any():
    no_town = listings[empty_inscription]
    cols_nt = [c for c in ['head_rel_type', 'clean_text'] if c in no_town.columns]
    detail_nt = no_town[cols_nt].head(20).copy()
    detail_nt.insert(0, 'csv_line', detail_nt.index + 2)
    print(f'WARNING: {len(no_town)} listing(s) have empty resolved_inscription '
          f'(no-town CDS or unresolvable head_name_body). '
          f'Step 8.8 UNKNOWN PostOffice fallback will apply.\n'
          + detail_nt.to_string(index=False))
if null_inscription.any():
    problem = listings[null_inscription]
    cols = [c for c in ['Postmark Key', 'head_rel_type', 'clean_text',
                        'resolved_inscription', 'resolved_town'] if c in problem.columns]
    detail = problem[cols].head(30).copy()
    detail.insert(0, 'csv_line', detail.index + 2)
    raise AssertionError(
        f'Silent drop: {len(problem)} listing(s) have null resolved_inscription '
        f'(should never happen -- resolve_relationships always assigns a value).\n'
        f'First offenders (csv_line = line number in the input CSV):\n{detail.to_string(index=False)}'
    )
print()

# V2: No rel entry should be orphaned
rel_entries = listings[listings['head_rel_type'].notna()]
orphans = rel_entries[rel_entries['parent_idx'].isna()]
print(f'V2: Orphan rel entries: {len(orphans)}')
if len(orphans):
    cols = [c for c in ['head_rel_type', 'clean_text'] if c in orphans.columns]
    detail = orphans[cols].copy()
    detail.insert(0, 'csv_line', detail.index + 2)
    # resolve_relationships already applied a best-effort fallback inscription
    # (own name body if present, otherwise empty string). Warn here so the
    # human can inspect cross-page or cross-section parent misses, but let
    # the row proceed -- it will get an UNKNOWN PostOffice via Step 8.8.
    print(f'WARNING: {len(orphans)} relationship entry(s) have no resolvable parent '
          f'(orphan_rel). They inherit from best-effort fallback inscription.\n'
          + detail.to_string(index=False))
print()

# V3: head_first_of_town should co-occur with independent entry
#     (*(L) and *(E) are the exception: first_of_town + rel indicator)
fot_entries = listings[listings['head_first_of_town']]
fot_rel = fot_entries[fot_entries['head_rel_type'].notna()]
print(f'V3: first_of_town entries: {len(fot_entries)}')
print(f'    of which are rel indicators: {len(fot_rel)}')
if len(fot_rel):
    for _, row in fot_rel.iterrows():
        print(f'  [{row["head_rel_type"]}] {row["clean_text"][:100]}')
print()

# V4: Warning distribution
all_warnings = [w for wlist in listings['s7_warnings'] for w in wlist]
print(f'V4: Total warnings: {len(all_warnings)}')
if all_warnings:
    warn_counts = pd.Series(all_warnings).value_counts()
    for w, c in warn_counts.items():
        print(f'  {w}: {c}')
print()

# V5: Same-with-name-body slash check
same_with_name = listings[
    (listings['head_rel_type'] == 'Same') & listings['head_name_body'].notna()
]
no_slash = same_with_name[~same_with_name['head_name_body'].str.startswith('/')]
print(f'V5: Same-with-name entries: {len(same_with_name)}')
print(f'    starting with /: {len(same_with_name) - len(no_slash)}')
print(f'    NOT starting with / (flagged): {len(no_slash)}')
if len(no_slash):
    for _, row in no_slash.iterrows():
        print(f'  head={row["seg_head"]!r}  name_body={row["head_name_body"]!r}')
        print(f'    resolved={row["resolved_inscription"]!r}')

V1: Missing resolved_inscription: 6
V1: Missing resolved_town: 6
csv_line = line number in the input CSV
 csv_line head_rel_type                                             clean_text resolved_inscription
       43           NaN                                       (L) See State --                     
       63           NaN                                       (L) See State --                     
       76           NaN                                       (L) See State --                     
       84           NaN                                       (L) See State --                     
       87           NaN                                       (L) See State --                     
      598           NaN (no town cds)(1853-54;PAID/1;Blue) Circular rate 40.00                     
 csv_line head_rel_type                                             clean_text
       43           NaN                                       (L) See State --
       63           NaN              

In [48]:
# --- Step 7.4: Resolved town distribution ---

town_counts = listings['resolved_town'].value_counts()
print(f'Distinct towns: {len(town_counts)}')
print(f'Towns with most listings:')
for town, count in town_counts.head(20).items():
    print(f'  {town!r}: {count}')
print()

# Single-listing towns
single = (town_counts == 1).sum()
print(f'Single-listing towns: {single} ({single/len(town_counts)*100:.1f}%)')
print()

# Inscription variants per town (shows device diversity)
inscription_per_town = listings.groupby('resolved_town')['resolved_inscription'].nunique()
multi_variant = inscription_per_town[inscription_per_town > 1]
print(f'Towns with multiple inscription variants: {len(multi_variant)}')
for town, n_variants in multi_variant.head(15).items():
    variants = listings[listings['resolved_town'] == town]['resolved_inscription'].unique()
    print(f'  {town!r}: {n_variants} variants')
    for v in variants:
        print(f'    {v!r}')

Distinct towns: 1232
Towns with most listings:
  'NORFOLK': 22
  'RICHMONDNa.': 18
  'PETERSBURG': 10
  'RICHMOND,': 9
  'ALEXANDRIA': 9
  'WHEELING': 9
  'NORFOLK Va.': 8
  'ABINGDON': 7
  '': 6
  'HARRISONBURG': 6
  'LYNCHBURG': 6
  'WINCHESTER': 6
  'SUFFOLK': 5
  'Wmsburg': 5
  'HARPERS FERRY': 5
  'KANAWHA CH. VA': 5
  'LEWISBURGH': 5
  'RICHD.VA': 5
  'WINCHESTER.VA': 5
  'Alexa.': 4

Single-listing towns: 1102 (89.4%)

Towns with multiple inscription variants: 57
  'ABINGDON': 5 variants
    'ABINGDON/*VA.*'
    'ABINGDON/VA.'
    'ABINGDON/VA'
    'ABINGDON/Va.'
    'ABINGDONVa./PAID'
  'ACCOMACK': 2 variants
    'ACCOMACK/Va.'
    'ACCOMACKC.H./Va.'
  'ALEXANDRIA': 5 variants
    'ALEXANDRIA/Va.'
    'ALEXANDRIA/VA.'
    'ALEXANDRIAVA./5'
    'ALEXANDRIAVA/3 PAID'
    'ALEXANDRIA/VA'
  'BERRYVILLE': 2 variants
    'BERRYVILLE/VA.'
    'BERRYVILLE/Va.'
  'BETHANY': 2 variants
    'BETHANY/Va.'
    'BETHANY/VA'
  'BIG LICK': 2 variants
    'BIG LICK/Va.'
    'BIG LICK/VA.'
  'CA

## 7.5 Step 7 Summary

Step 7 resolves cross-record inheritance, adding four new columns to `listings`:

- **parent_idx** -- DataFrame index of the parent entry for rel-indicator rows; null for independent entries
- **resolved_inscription** -- full inscription text on the physical device
- **resolved_town** -- town root for PostOffice grouping (text before first `/`)
- **s7_warnings** -- list of confidence-lowering flags

Step 7.1b propagates parsed attributes from parent to child when the child's own
paren body is silent on the attribute:

- **is_manuscript** -- inherited if child has no `ms` or `size` field type
- **parsed_colors** -- inherited if child's color list is empty
- **parsed_sizes** -- inherited if child's size list is empty

This ensures that relationship-indicator entries (`(L)`, `(E)`, `Same`) carry
the full physical device characteristics of their parent without requiring each
child to repeat them. Inheritance happens before color fan-out and shape
resolution so downstream steps see fully resolved attribute sets.

In [49]:
# --- Step 7.5: Summary ---

total = len(listings)
rel_count = listings['head_rel_type'].notna().sum()
warn_count = sum(len(w) for w in listings['s7_warnings'])

print('Step 7: Relationship Resolution')
print(f'  Listings processed: {total}')
print()
print(f'  Independent entries:  {total - rel_count}')
print(f'  Resolved from parent: {rel_count}')
print(f'    Same (no name):     {((listings["head_rel_type"] == "Same") & listings["head_name_body"].isna()).sum()}')
print(f'    Same (with name):   {((listings["head_rel_type"] == "Same") & listings["head_name_body"].notna()).sum()}')
print(f'    (L):                {(listings["head_rel_type"] == "(L)").sum()}')
print(f'    (E):                {(listings["head_rel_type"] == "(E)").sum()}')
print()
print(f'  Distinct towns:       {listings["resolved_town"].nunique()}')
print(f'  Total warnings:       {warn_count}')
print()

# Inheritance impact
rel_entries = listings[listings['parent_idx'].notna()]
print('  Attribute inheritance impact (rel entries only):')
ms_from_parent = rel_entries[rel_entries['is_manuscript']].shape[0]
print(f'    Manuscript after inheritance: {ms_from_parent}')
has_colors = rel_entries[rel_entries['parsed_colors'].apply(len) > 0].shape[0]
print(f'    With colors after inheritance: {has_colors}')
has_sizes = rel_entries[rel_entries['parsed_sizes'].apply(len) > 0].shape[0]
print(f'    With sizes after inheritance: {has_sizes}')

Step 7: Relationship Resolution
  Listings processed: 1537

  Independent entries:  1282
  Resolved from parent: 255
    Same (no name):     144
    Same (with name):   89
    (L):                21
    (E):                1

  Distinct towns:       1232
  Total warnings:       63

  Attribute inheritance impact (rel entries only):
    Manuscript after inheritance: 14
    With colors after inheritance: 255
    With sizes after inheritance: 237


# Step 8: Postmark Assembly

Transforms the per-listing parsed data from Steps 1-7 into normalized domain
entity DataFrames aligned with the WorldCovers data model. This step handles:

- **Value table construction** -- Color (discovered from data), Shape (from model seeds)
- **Shape code decomposition** -- Compound ASCC codes (DC, DLC, DLDC, etc.) resolved to a base Shape
- **Color fan-out** -- Multi-color listings exploded into one Postmark per color
- **Postmark record assembly** -- All scalar fields populated, FK references resolved
- **Dependent entity assembly** -- DateObserved, PostmarkValuation
- **PostOffice normalization** -- Town root dedup, case normalization

Output: DataFrames ready for export:
- `postmarks_df` -- one row per Postmark entity
- `date_observed_df` -- one row per date observation
- `postmark_valuation_df` -- one row per valuation tier
- `post_offices_df` -- one row per normalized post office
- `colors_df`, `shapes_df` -- value/lookup tables

## 8.1 Value Table Construction

Shape from model seed values (fixed vocabulary).
Color discovered from all unique color names across listings.

In [50]:
# --- Step 8.1: Value table construction ---

# Shape seeds (from model.md)
SHAPE_SEEDS = [
    'SL - Straight Line', 'Box', 'O - Oval', 'C - Circle',
    'ARC - Arc or Semi-circle', 'Octagon',
    'Pictorial', 'Ornamental Mortised', 'Other',
]

shapes_df = pd.DataFrame({
    'shape_id': range(1, len(SHAPE_SEEDS) + 1),
    'name': SHAPE_SEEDS,
})
shape_lookup = dict(zip(shapes_df['name'].str.upper(), shapes_df['shape_id']))

# Color: discover from data (UPPER normalized)
all_color_names = sorted({
    c for clist in listings['parsed_colors'] for c in clist
})

colors_df = pd.DataFrame({
    'color_id': range(1, len(all_color_names) + 1),
    'name': all_color_names,
})
color_lookup = dict(zip(colors_df['name'], colors_df['color_id']))

# Lettering seeds (from model.md)
LETTERING_SEEDS = [
    'Italic', 'Serif', 'Sans-serif', 'Small', 'Large',
    'Outline', 'Bold', 'Block', 'Gothic',
]

letterings_df = pd.DataFrame({
    'lettering_id': range(1, len(LETTERING_SEEDS) + 1),
    'name': LETTERING_SEEDS,
})
lettering_lookup = dict(zip(
    letterings_df['name'].str.lower(),
    letterings_df['lettering_id'],
))
# Common plural/variant aliases used in ASCC bracket descriptors
_lettering_aliases = {
    'italics': 'italic',
    'serifs': 'serif',
    'sans serifs': 'sans-serif',
    'sans serif': 'sans-serif',
}
for alias, canonical in _lettering_aliases.items():
    if canonical in lettering_lookup:
        lettering_lookup[alias] = lettering_lookup[canonical]

print(f'Value tables constructed:')
print(f'  Shapes:     {len(shapes_df)} seeds')
print(f'  Colors:     {len(colors_df)} discovered')
print(f'  Letterings: {len(letterings_df)} seeds')
print()
print(f'Colors: {all_color_names}')

Value tables constructed:
  Shapes:     9 seeds
  Colors:     16 discovered
  Letterings: 9 seeds

Colors: ['BLACK', 'BLACK-BROWN', 'BLUE', 'BLUE-GREEN', 'BROWN', 'BROWN-BLACK', 'BROWN-RED', 'BROWNISH', 'GREEN', 'MAGENTA', 'MS', 'ORANGE', 'PURPLE', 'RED', 'RED-ORANGE', 'VIOLET']


## 8.2 Shape Code Decomposition

Compound ASCC codes are resolved to a base Shape (FK to shapes_df). Each
listing's effective shape is resolved by priority: paren-body shape code >
section Default Shape > catalog-wide SL fallback.

Decomposition table:

| Code | Base Shape |
|------|-----------|
| SL | SL |
| C | C |
| O | O |
| DC | C |
| DO | O |
| DLC | C |
| DLO | O |
| DLDC | C |
| DLDO | O |
| NOR | C |
| BOX | BOX |
| ARC | ARC |
| OCTAGON | Octagon |
| PMK | Other |


In [51]:
# --- Step 8.2: Shape code decomposition ---

# Decomposition table: ASCC code -> base_shape_name
SHAPE_DECOMP = {
    'SL':      'SL - Straight Line',
    'C':       'C - Circle',
    'O':       'O - Oval',
    'DC':      'C - Circle',
    'DO':      'O - Oval',
    'DLC':     'C - Circle',
    'DLO':     'O - Oval',
    'DLDC':    'C - Circle',
    'DLDO':    'O - Oval',
    'NOR':     'C - Circle',
    'BOX':     'Box',
    'ARC':     'ARC - Arc or Semi-circle',
    'OCTAGON': 'Octagon',
    'PMK':     'Other',
}

# Codes where DL prefix creates inner/outer ring ambiguity
DL_AMBIGUOUS_CODES = {'DLC', 'DLO', 'DLDC', 'DLDO'}

CATALOG_FALLBACK_SHAPE = 'SL'


def resolve_effective_shape(row):
    """Determine the effective shape code for a listing.
    Priority: paren-body shape code > Default Shape column > catalog-wide SL.
    Returns (effective_code_upper, source_label)."""

    # 1. Paren-body shape (from parsed_sizes -- use first non-None)
    for s in row['parsed_sizes']:
        if s.get('size_shape_code'):
            return s['size_shape_code'].upper(), 'paren_body'

    # 2. Section-level Default Shape
    default = row.get('Default Shape')
    if pd.notna(default) and str(default).strip():
        # Default Shape column may use full names or codes
        ds = str(default).strip().upper()
        # Map common full names to codes
        name_to_code = {
            'CIRCLE': 'C', 'OVAL': 'O', 'STRAIGHT LINE': 'SL',
            'BOX': 'BOX', 'ARC': 'ARC', 'OCTAGON': 'OCTAGON',
            'DOUBLE CIRCLE': 'DC', 'DOUBLE OVAL': 'DO',
            'DOUBLE LINE CIRCLE': 'DLC', 'DOUBLE LINE OVAL': 'DLO',
            'NO OUTER RIM': 'NOR', 'FANCY': 'C',  # fancy = circle variant
        }
        # Try direct code match first, then name map
        if ds in SHAPE_DECOMP:
            return ds, 'default_shape'
        if ds in name_to_code:
            return name_to_code[ds], 'default_shape'
        # Partial match: check if ds starts with a known code
        for code in sorted(SHAPE_DECOMP.keys(), key=len, reverse=True):
            if ds.startswith(code):
                return code, 'default_shape'
        # Unrecognized default shape -- fall through to catalog default

    # 3. Catalog-wide fallback
    return CATALOG_FALLBACK_SHAPE, 'catalog_fallback'


def decompose_shape(code_upper):
    """Decompose a shape code into (base_shape_name, error_or_None)."""
    if code_upper in SHAPE_DECOMP:
        return SHAPE_DECOMP[code_upper], None
    # Unknown code -- map to Other
    return 'Other', f'unknown shape code: {code_upper}'


# Apply to all listings
shape_resolution = listings.apply(
    lambda row: pd.Series(resolve_effective_shape(row),
                          index=['effective_shape_code', 'shape_source']),
    axis=1
)
listings = pd.concat([listings, shape_resolution], axis=1)

# Decompose
decomposed = listings['effective_shape_code'].apply(
    lambda c: pd.Series(decompose_shape(c),
                        index=['base_shape_name', 'shape_error'])
)
listings = pd.concat([listings, decomposed], axis=1)

# Resolve shape FK
listings['shape_id'] = listings['base_shape_name'].apply(
    lambda n: shape_lookup.get(n.upper())
)

# Report
print('Shape resolution source distribution:')
for src, count in listings['shape_source'].value_counts().items():
    print(f'  {src}: {count}')
print()

print('Base shape distribution:')
for name, count in listings['base_shape_name'].value_counts().items():
    print(f'  {name}: {count}')
print()

dl_count = listings['effective_shape_code'].isin(DL_AMBIGUOUS_CODES).sum()
print(f'DL-prefix codes (ring order ambiguous): {dl_count}')
print()

errors = listings[listings['shape_error'].notna()]
if len(errors):
    print(f'Shape decomposition errors: {len(errors)}')
    for _, row in errors.head(10).iterrows():
        print(f'  {row["shape_error"]}  raw: {row["clean_text"][:80]}')
else:
    print('No shape decomposition errors.')

Shape resolution source distribution:
  catalog_fallback: 855
  default_shape: 527
  paren_body: 155

Base shape distribution:
  SL - Straight Line: 957
  C - Circle: 558
  O - Oval: 18
  Box: 2
  ARC - Arc or Semi-circle: 2

DL-prefix codes (ring order ambiguous): 4

No shape decomposition errors.


## 8.3 Color Fan-Out

Listings with multiple `parsed_colors` are exploded into N Postmark rows, one
per color. All other attributes are duplicated identically. Listings with zero
colors get one row with null color. Listings with one color get one row.

Multi-color fan-out is flagged for the final confidence score since it is
inherently ambiguous whether multiple colors represent the same device in
different ink, or separate observations.

In [52]:
# --- Step 8.3: Color fan-out ---

expanded_rows = []
next_postmark_id = 1

for idx, row in listings.iterrows():
    colors = row['parsed_colors']
    is_multi_color = len(colors) > 1

    if not colors:
        r = {
            'postmark_id': next_postmark_id,
            'source_listing_idx': idx,
            'color_name': None,
            'color_id': None,
            'is_multi_color_fanout': False,
            'fanout_idx': 0,
        }
        expanded_rows.append(r)
        next_postmark_id += 1
    else:
        for i, color_name in enumerate(colors):
            r = {
                'postmark_id': next_postmark_id,
                'source_listing_idx': idx,
                'color_name': color_name,
                'color_id': color_lookup.get(color_name),
                'is_multi_color_fanout': is_multi_color,
                'fanout_idx': i,
            }
            expanded_rows.append(r)
            next_postmark_id += 1

fanout_df = pd.DataFrame(expanded_rows)

multi_color_listings = listings[listings['parsed_colors'].apply(len) > 1]
total_postmarks = len(fanout_df)
print(f'Color fan-out results:')
print(f'  Input listings: {len(listings)}')
print(f'  Output postmark rows: {total_postmarks}')
print(f'  Net expansion: {total_postmarks - len(listings)} rows')
print(f'  Multi-color source listings: {len(multi_color_listings)}')
print(f'  Rows from multi-color fan-out: {fanout_df["is_multi_color_fanout"].sum()}')
print()

color_counts = listings['parsed_colors'].apply(len).value_counts().sort_index()
print('Colors-per-listing distribution:')
for n, count in color_counts.items():
    print(f'  {n} colors: {count} listings')


Color fan-out results:
  Input listings: 1537
  Output postmark rows: 1757
  Net expansion: 220 rows
  Multi-color source listings: 174
  Rows from multi-color fan-out: 394

Colors-per-listing distribution:
  0 colors: 800 listings
  1 colors: 563 listings
  2 colors: 133 listings
  3 colors: 36 listings
  4 colors: 5 listings


## 8.4 Postmark Record Assembly

Build the main `postmarks_df` with all Postmark model fields. Each row in
`fanout_df` becomes one Postmark record. Scalar fields are pulled from the
source listing; color comes from the fan-out.

In [53]:
# --- Step 8.4: Postmark record assembly ---

def build_postmark(fan_row):
    """Assemble a single Postmark record from a fan-out row + source listing."""
    src = listings.loc[fan_row['source_listing_idx']]

    is_ms = bool(src['is_manuscript'])

    # Dimensions: first parsed_sizes entry (if any)
    width, height = None, None
    is_irreg = None if is_ms else False
    date_format = None
    if src['parsed_sizes']:
        s = src['parsed_sizes'][0]
        width = s.get('size_dim1')
        height = s.get('size_dim2')
        if width is not None and height is None:
            height = width
        if s.get('size_is_irregular'):
            is_irreg = True
        if s.get('size_dateformat'):
            date_format = s['size_dateformat']

    # Shape: null for manuscript, required for handstamped
    shape_id = None if is_ms else src['shape_id']

    # Impression: null for manuscript, default Normal for handstamped
    impression = None if is_ms else 'Normal'

    # Lettering: null for manuscript (per invariant), resolved from annotations otherwise
    lettering_id = None if is_ms else src.get('lettering_id')

    # Parent listing (intermediate for review)
    parent_listing_idx = src.get('parent_idx')
    if pd.notna(parent_listing_idx):
        parent_listing_idx = int(parent_listing_idx)
    else:
        parent_listing_idx = None

    # Images above: catalog page image count (intermediate field for human review)
    images_above = src.get('Images Above')
    if pd.notna(images_above):
        images_above = int(images_above)
    else:
        images_above = None

    # Page and chunk from the OCR extractor. Together they identify the
    # extracted image files used in Step 11:
    # backend/media/<state>/va-<page>-<chunk>-<counter>.png
    page = src.get('Page')
    if pd.notna(page):
        page = int(page)
    else:
        page = None

    chunk = src.get('Chunk')
    if pd.notna(chunk):
        chunk = int(chunk)
    else:
        chunk = None

    # Postmark code: minted from RW_CODE + REGION_ABBREV + the listing's
    # position in the original input CSV (1-based) + fanout index. The
    # .{fanout_idx} suffix keeps codes unique across multi-color fan-outs
    # of the same listing. This is an intermediate join key that wires
    # postmarks_df to cover_markings_df['marking_code'] during construction;
    # it is resolved to the integer marking_id at emit time (Step 10) and
    # never appears in the final markings.csv. source_listing_idx preserves
    # the original CSV row index because listings is a filter-copy of df.
    listing_pos = int(fan_row['source_listing_idx']) + 1
    code = f"{RW_CODE}-{REGION_ABBREV}-{listing_pos}.{int(fan_row['fanout_idx'])}"

    return {
        'postmark_id': fan_row['postmark_id'],
        'code': code,
        'catalog_text': src['clean_text'],
        'inscription_text': src['resolved_inscription'],
        'is_manuscript': is_ms,
        'shape_id': shape_id,
        'lettering_id': lettering_id,
        'color_id': fan_row['color_id'],
        'width': width,
        'height': height,
        'is_irregular': is_irreg,
        'date_format': date_format,
        'date_type': None,
        'impression': impression,
        'post_office_id': None,  # filled in 8.8
        'source_listing_idx': fan_row['source_listing_idx'],
        'color_name': fan_row['color_name'],
        'is_multi_color_fanout': fan_row['is_multi_color_fanout'],
        'images_above': images_above,
        'page': page,
        'chunk': chunk,
        'parent_listing_idx': parent_listing_idx,
    }


postmarks_df = pd.DataFrame(
    [build_postmark(row) for _, row in fanout_df.iterrows()]
)

# --- Resolve parent_listing_idx -> parent_postmark_id ---
_pm_by_listing_color = {}
for _, row in postmarks_df.iterrows():
    key = (row['source_listing_idx'], row['color_name'])
    _pm_by_listing_color[key] = row['postmark_id']

_pm_first_by_listing = postmarks_df.groupby('source_listing_idx')['postmark_id'].first()

def resolve_parent_pm(row):
    pidx = row['parent_listing_idx']
    if pd.isna(pidx) or pidx is None:
        return None
    pidx = int(pidx)
    key = (pidx, row['color_name'])
    if key in _pm_by_listing_color:
        return _pm_by_listing_color[key]
    return _pm_first_by_listing.get(pidx)

postmarks_df['parent_postmark_id'] = postmarks_df.apply(resolve_parent_pm, axis=1)
postmarks_df.drop(columns=['parent_listing_idx'], inplace=True)

print(f'Postmarks assembled: {len(postmarks_df)}')
print(f'  Manuscript: {postmarks_df["is_manuscript"].sum()}')
print(f'  Handstamped: {(~postmarks_df["is_manuscript"]).sum()}')
print(f'  With code: {postmarks_df["code"].notna().sum()}')
print(f'  With color: {postmarks_df["color_id"].notna().sum()}')
print(f'  With dimensions: {postmarks_df["width"].notna().sum()}')
print(f'  With images_above: {postmarks_df["images_above"].notna().sum()}')
print(f'  With parent (inherited): {postmarks_df["parent_postmark_id"].notna().sum()}')
print(f'  With date_fmt: {postmarks_df["date_format"].notna().sum()}')
print(f'  Multi-color fan-out rows: {postmarks_df["is_multi_color_fanout"].sum()}')
print()

# Postmark.code uniqueness check
coded = postmarks_df[postmarks_df['code'].notna()]
if len(coded):
    dup = coded['code'][coded['code'].duplicated()]
    if len(dup):
        print(f'WARNING: duplicate Postmark codes detected ({len(dup)}):')
        for c in dup.unique()[:10]:
            print(f'  {c!r}')
    else:
        print(f'Postmark.code unique across all {len(coded)} coded rows.')
    print()

# Parent linkage
if postmarks_df['parent_postmark_id'].notna().any():
    children = postmarks_df[postmarks_df['parent_postmark_id'].notna()]
    color_matched = 0
    fallback_used = 0
    for _, row in children.iterrows():
        parent_pm = postmarks_df[postmarks_df['postmark_id'] == row['parent_postmark_id']].iloc[0]
        if row['color_name'] == parent_pm['color_name']:
            color_matched += 1
        else:
            fallback_used += 1
    print(f'Parent-postmark linkage:')
    print(f'  Color-matched: {color_matched}')
    print(f'  Fallback (first of parent): {fallback_used}')
    print()

shape_dist = postmarks_df['shape_id'].value_counts(dropna=False).sort_index()
print('Shape distribution on postmarks:')
for sid, count in shape_dist.items():
    if pd.isna(sid):
        print(f'  (null -- manuscript): {count}')
    else:
        name = shapes_df.loc[shapes_df['shape_id'] == sid, 'name'].iloc[0]
        print(f'  {name}: {count}')
print()

lettering_hits = postmarks_df['lettering_id'].notna().sum()
print(f'Lettering assigned on postmarks: {lettering_hits}')
if lettering_hits:
    for lid, count in postmarks_df['lettering_id'].value_counts(dropna=True).items():
        name = letterings_df.loc[letterings_df['lettering_id'] == lid, 'name'].iloc[0]
        print(f'  {name}: {count}')


Postmarks assembled: 1757
  Manuscript: 841
  Handstamped: 916
  With code: 1757
  With color: 957
  With dimensions: 821
  With images_above: 1757
  With parent (inherited): 347
  With date_fmt: 81
  Multi-color fan-out rows: 394

Postmark.code unique across all 1757 coded rows.

Parent-postmark linkage:
  Color-matched: 206
  Fallback (first of parent): 141

Shape distribution on postmarks:
  SL - Straight Line: 124
  Box: 2
  O - Oval: 22
  C - Circle: 766
  ARC - Arc or Semi-circle: 2
  (null -- manuscript): 841

Lettering assigned on postmarks: 0


## 8.5 DateObserved Assembly

Each parsed date from Step 6 becomes a DateObserved row linked to every Postmark
that was produced from that listing (including fan-out siblings).

Granularity mapping:
- DAY -> granularity=DAY, date = full date
- MONTH -> granularity=MONTH, date = year-month-01
- YEAR -> granularity=YEAR, date = year-01-01
- RANGE and DECADE -> two bookend YEAR rows (start, end) since the model
  uses individual date points, not spans

In [54]:
# --- Step 8.5: DateObserved assembly ---

from datetime import date as dt_date

date_rows = []
next_date_id = 1

for _, pm in postmarks_df.iterrows():
    src = listings.loc[pm['source_listing_idx']]
    for d in src['parsed_dates']:
        gran = d['date_granularity']

        if gran == 'DAY':
            try:
                obs_date = dt_date(d['date_year_start'], d['date_month'], d['date_day'])
            except (ValueError, TypeError):
                obs_date = None
            date_rows.append({
                'date_observed_id': next_date_id,
                'postmark_id': pm['postmark_id'],
                'date': str(obs_date) if obs_date else None,
                'granularity': 'DAY',
                'date_raw': d.get('date_raw'),
                'date_error': d.get('date_error'),
            })
            next_date_id += 1

        elif gran == 'MONTH':
            try:
                obs_date = dt_date(d['date_year_start'], d['date_month'], 1)
            except (ValueError, TypeError):
                obs_date = None
            date_rows.append({
                'date_observed_id': next_date_id,
                'postmark_id': pm['postmark_id'],
                'date': str(obs_date) if obs_date else None,
                'granularity': 'MONTH',
                'date_raw': d.get('date_raw'),
                'date_error': d.get('date_error'),
            })
            next_date_id += 1

        elif gran == 'YEAR':
            try:
                obs_date = dt_date(d['date_year_start'], 1, 1)
            except (ValueError, TypeError):
                obs_date = None
            date_rows.append({
                'date_observed_id': next_date_id,
                'postmark_id': pm['postmark_id'],
                'date': str(obs_date) if obs_date else None,
                'granularity': 'YEAR',
                'date_raw': d.get('date_raw'),
                'date_error': d.get('date_error'),
            })
            next_date_id += 1

        elif gran in ('RANGE', 'DECADE'):
            # Two bookend YEAR rows
            for yr in (d['date_year_start'], d['date_year_end']):
                try:
                    obs_date = dt_date(int(yr), 1, 1)
                except (ValueError, TypeError):
                    obs_date = None
                date_rows.append({
                    'date_observed_id': next_date_id,
                    'postmark_id': pm['postmark_id'],
                    'date': str(obs_date) if obs_date else None,
                    'granularity': 'YEAR',
                    'date_raw': d.get('date_raw'),
                    'date_error': d.get('date_error'),
                })
                next_date_id += 1

date_observed_df = pd.DataFrame(date_rows) if date_rows else pd.DataFrame(
    columns=['date_observed_id', 'postmark_id', 'date', 'granularity',
             'date_raw', 'date_error']
)

print(f'DateObserved rows: {len(date_observed_df)}')
if len(date_observed_df):
    print(f'  Linked to {date_observed_df["postmark_id"].nunique()} postmarks')
    gran_dist = date_observed_df['granularity'].value_counts()
    for g, c in gran_dist.items():
        print(f'  {g}: {c}')
    errors = date_observed_df[date_observed_df['date'].isna()]
    if len(errors):
        print(f'  Date construction errors: {len(errors)}')

DateObserved rows: 1460
  Linked to 911 postmarks
  YEAR: 1386
  DAY: 74


## 8.6 PostmarkValuation Assembly

Each valuation tier from Step 3 becomes a PostmarkValuation row. Position is
1-based ordinal (position 1 = earliest date period). Multi-color fan-out
siblings share the same valuation (each Postmark gets a copy of all tiers).

In [55]:
# --- Step 8.6: PostmarkValuation assembly ---

val_rows = []
next_val_id = 1

for _, pm in postmarks_df.iterrows():
    src = listings.loc[pm['source_listing_idx']]
    tiers = src['valuation_tiers']

    for pos_0, tier_str in enumerate(tiers):
        amount = None
        if tier_str is not None:
            # Strip commas, parse as float
            try:
                amount = float(tier_str.replace(',', ''))
            except (ValueError, AttributeError):
                amount = None  # unparseable -- leave null

        val_rows.append({
            'valuation_id': next_val_id,
            'postmark_id': pm['postmark_id'],
            'amount': amount,
            'appraisal_position': pos_0 + 1,  # 1-based
            'appraisal_date': None,  # catalog publication date; TBD
        })
        next_val_id += 1

postmark_valuation_df = pd.DataFrame(val_rows) if val_rows else pd.DataFrame(
    columns=['valuation_id', 'postmark_id', 'amount',
             'appraisal_position', 'appraisal_date']
)

print(f'PostmarkValuation rows: {len(postmark_valuation_df)}')
if len(postmark_valuation_df):
    print(f'  Linked to {postmark_valuation_df["postmark_id"].nunique()} postmarks')
    pos_dist = postmark_valuation_df['appraisal_position'].value_counts().sort_index()
    for pos, c in pos_dist.items():
        print(f'  Position {pos}: {c}')
    unpriced = postmark_valuation_df['amount'].isna().sum()
    print(f'  Unpriced (null amount): {unpriced}')

PostmarkValuation rows: 1777
  Linked to 1757 postmarks
  Position 1: 1757
  Position 2: 20
  Unpriced (null amount): 104


## 8.8 PostOffice Normalization

Normalize `resolved_town` into canonical PostOffice records. Normalization:
- Strip trailing periods and commas
- Uppercase for grouping (preserve one representative display form)
- Assign PostOffice IDs
- Back-fill `post_office_id` on postmarks_df

In [56]:
# --- Step 8.8: PostOffice normalization ---

# Normalize resolved_town: strip trailing punctuation/whitespace, uppercase,
# treat blanks as missing. Nullable string dtype handles NaN/None uniformly.
listings['normalized_town'] = (
    listings['resolved_town'].astype('string')
    .str.strip()
    .str.replace(r'[.,\s]+$', '', regex=True)
    .str.upper()
    .replace('', pd.NA)
)

# state_code is the run-level REGION_ABBREV (derived from the input CSV
# filename and validated against regions.csv in cell 2). The munger
# processes one region per run, so every listing gets the same value;
# there is no per-row override.
listings['state_code'] = pd.Series(
    REGION_ABBREV, index=listings.index, dtype='string'
)

# One PostOffice per (state_code, normalized_town). Deterministic sort; NA first.
# PostOffice rows no longer carry an inline region_id -- the link to Region
# moves to post_office_regions_df, built at the end of this cell.
post_offices_df = (
    listings[['state_code', 'normalized_town']]
    .dropna(subset=['normalized_town'])
    .drop_duplicates()
    .sort_values(['state_code', 'normalized_town'], na_position='first')
    .reset_index(drop=True)
)
post_offices_df.insert(0, 'post_office_id', post_offices_df.index + 1)
post_offices_df['name'] = post_offices_df['normalized_town']
post_offices_df = post_offices_df[['post_office_id', 'name', 'state_code']]

# Back-fill post_office_id onto listings via a composite-key dict lookup.
# None stands in for NaN/pd.NA so dict equality is stable.
def _nkey(v):
    return None if pd.isna(v) else v

po_id_by_key = {
    (_nkey(sc), nt): pid
    for pid, nt, sc in post_offices_df[['post_office_id', 'name', 'state_code']].itertuples(index=False)
}
listings['post_office_id'] = [
    po_id_by_key.get((_nkey(sc), nt)) if pd.notna(nt) else None
    for sc, nt in zip(listings['state_code'], listings['normalized_town'])
]

# Back-fill post_office_id onto postmarks_df via source_listing_idx.
postmarks_df['post_office_id'] = postmarks_df['source_listing_idx'].map(
    listings['post_office_id']
)

# --- Fallback: assign one UNKNOWN post office per state for unresolved rows ---
# Without this, postmarks whose resolved_town came out empty have no FK to
# PostOffice, so they never appear in state-scoped listings. Rather than drop
# them, bucket them under an UNKNOWN PO keyed on their state_code.
unresolved = postmarks_df['post_office_id'].isna()
if unresolved.any():
    unresolved_states = (
        postmarks_df.loc[unresolved, 'source_listing_idx']
        .map(listings['state_code'])
    )
    distinct_states = unresolved_states.unique()
    next_po_id = int(post_offices_df['post_office_id'].max() or 0) + 1
    unknown_rows = []
    unknown_by_state = {}
    for sc in distinct_states:
        unknown_rows.append({
            'post_office_id': next_po_id,
            'name': 'UNKNOWN',
            'state_code': sc,
        })
        unknown_by_state[_nkey(sc)] = next_po_id
        next_po_id += 1
    post_offices_df = pd.concat(
        [post_offices_df, pd.DataFrame(unknown_rows)],
        ignore_index=True,
    )
    postmarks_df.loc[unresolved, 'post_office_id'] = unresolved_states.map(
        lambda sc: unknown_by_state[_nkey(sc)]
    ).values
    print(f'  Assigned {unresolved.sum()} postmarks to UNKNOWN post office(s) across {len(unknown_by_state)} state(s)')

# --- post_office_regions junction ---
# One row per PostOffice, linking it to the run's single REGION_ID. Built
# after the UNKNOWN-fallback fan-out so every PostOffice (including UNKNOWN
# buckets) gets a junction row. The notebook is one-bundle-per-region, so
# every junction row points at the same REGION_ID; cross-region dedupe of
# PostOffices is out of scope here.
post_office_regions_df = pd.DataFrame({
    'post_office_region_id': range(1, len(post_offices_df) + 1),
    'post_office_id': post_offices_df['post_office_id'].astype(int).values,
    'region_id': REGION_ID,
})

# --- Diagnostics ---
print(f'PostOffice records: {len(post_offices_df)}')
print(f'  From {listings["resolved_town"].nunique()} raw distinct towns')
print(f'  To {len(post_offices_df)} normalized post offices')
print(f'PostOfficeRegion links: {len(post_office_regions_df)} (all -> region_id={REGION_ID})')
if post_offices_df['state_code'].notna().any():
    per_state = post_offices_df.groupby('state_code').size()
    print(f'  Per state:')
    for sc, n in per_state.items():
        print(f'    {sc}: {n} post offices')
print()

raw_to_norm = (
    listings.groupby(['state_code', 'normalized_town'], dropna=False)['resolved_town']
    .apply(lambda x: list(x.unique()))
)
collapsed = raw_to_norm[raw_to_norm.apply(len) > 1]
if len(collapsed):
    print(f'Towns collapsed by normalization: {len(collapsed)}')
    for (sc, norm), variants in list(collapsed.items())[:20]:
        tag = f'[{sc}] ' if pd.notna(sc) else ''
        print(f'  {tag}{norm!r} <- {variants}')
else:
    print('No normalization collapses (all raw towns map 1:1).')
print()

missing_po = postmarks_df['post_office_id'].isna().sum()
if missing_po:
    print(f'WARNING: {missing_po} postmarks missing post_office_id')
else:
    print('All postmarks have a post_office_id.')

  Assigned 6 postmarks to UNKNOWN post office(s) across 1 state(s)
PostOffice records: 1206
  From 1232 raw distinct towns
  To 1206 normalized post offices
PostOfficeRegion links: 1206 (all -> region_id=51)
  Per state:
    VA: 1206 post offices

Towns collapsed by normalization: 22
  [VA] 'ALEX' <- ['ALEX', 'ALEX,']
  [VA] 'ALEXA' <- ['Alexa.', 'ALEXA.']
  [VA] 'AYLETTS' <- ['Ayletts', 'AYLETTS']
  [VA] 'CHS.TOWN VA' <- ['CHS.TOWN VA.', 'CHS.TOWN Va.', 'CHS.TOWN VA']
  [VA] 'DUMFRIES' <- ['Dumfries', 'DUMFRIES']
  [VA] 'FREDBG' <- ['Fredbg', 'FREDBG']
  [VA] 'FREDERICKSBURG' <- ['Fredericksburg', 'FREDERICKSBURG']
  [VA] 'FREDERICKSBURG VA' <- ['FREDERICKSBURG VA.', 'FREDERICKSBURG VA']
  [VA] 'FREDS BURG' <- ['FREDS BURG', 'FREDS BURG,']
  [VA] 'FREE' <- ['FREE', 'FREE.']
  [VA] 'HAMPTON' <- ['Hampton', 'HAMPTON']
  [VA] 'HARRISONBURG' <- ['HARRISONBURG', 'Harrisonburg.']
  [VA] 'NORFOLK' <- ['NORFOLK', 'Norfolk']
  [VA] 'PETERSBURG' <- ['Petersburg', 'PETERSBURG', 'PETERSBURG,']
  

## 8.9 Assembly Confidence

Combine warnings from all steps into a per-postmark confidence assessment.
Warnings reduce confidence from HIGH to MEDIUM or LOW.

In [57]:
# --- Step 8.9: Assembly confidence ---

def compute_assembly_warnings(pm_row):
    """Collect all warnings applicable to a single postmark."""
    src = listings.loc[pm_row['source_listing_idx']]
    warnings = list(src.get('s7_warnings', []))

    # Step 1 confidence
    if src.get('confidence') == 'low':
        warnings.append(f'low_classification_confidence:{src.get("reason")}')

    # Multi-color ambiguity
    if pm_row['is_multi_color_fanout']:
        warnings.append('multi_color_fanout')

    # Shape fallback
    if src.get('shape_source') == 'catalog_fallback':
        warnings.append('shape_from_catalog_fallback')

    # Shape decomposition error
    if pd.notna(src.get('shape_error')):
        warnings.append(f'shape_error:{src["shape_error"]}')

    # DL-prefix ring order ambiguity
    if src.get('effective_shape_code') in DL_AMBIGUOUS_CODES:
        warnings.append('dl_ring_order_ambiguous')

    # Missing dates
    if not src['parsed_dates']:
        warnings.append('no_dates_parsed')

    # Multiple size fields (ambiguous dimensions)
    if len(src['parsed_sizes']) > 1:
        warnings.append(f'multiple_size_fields:{len(src["parsed_sizes"])}')

    # Unresolved other fields
    if src['other_fields']:
        warnings.append(f'unresolved_other_fields:{len(src["other_fields"])}')

    return warnings


postmarks_df['s8_warnings'] = postmarks_df.apply(
    compute_assembly_warnings, axis=1
)

# Confidence level: HIGH if no warnings, MEDIUM if 1-2, LOW if 3+
def confidence_level(warnings):
    n = len(warnings)
    if n == 0:
        return 'HIGH'
    elif n <= 2:
        return 'MEDIUM'
    else:
        return 'LOW'

postmarks_df['assembly_confidence'] = postmarks_df['s8_warnings'].apply(confidence_level)

print('Assembly confidence distribution:')
for level, count in postmarks_df['assembly_confidence'].value_counts().items():
    print(f'  {level}: {count}')
print()

# Warning frequency
all_warnings = [w for wlist in postmarks_df['s8_warnings'] for w in wlist]
if all_warnings:
    warn_counts = pd.Series(all_warnings).value_counts()
    print(f'Warning frequency ({len(all_warnings)} total):')
    for w, c in warn_counts.items():
        print(f'  {w}: {c}')
else:
    print('No warnings -- all postmarks at HIGH confidence.')

Assembly confidence distribution:
  MEDIUM: 1297
  HIGH: 422
  LOW: 38

Warning frequency (2252 total):
  shape_from_catalog_fallback: 860
  no_dates_parsed: 845
  multi_color_fanout: 394
  cross_section_parent: 35
  unresolved_other_fields:1: 35
  same_name_body_no_slash: 34
  low_classification_confidence:value_only: 22
  multiple_size_fields:2: 16
  independent_no_name: 6
  dl_ring_order_ambiguous: 5


## 8.10 Step 8 Summary

Step 8 transforms the per-listing parsed data into normalized domain-entity
DataFrames. Output tables:

- **postmarks_df** -- one Postmark per color variant per listing
- **date_observed_df** -- date observations linked to postmarks
- **postmark_valuation_df** -- valuation tiers linked to postmarks
- **post_offices_df** -- normalized post office records
- **colors_df, shapes_df** -- value/lookup tables

PostOffice normalization collapses raw town roots by stripping trailing
punctuation and grouping by uppercased form.

Assembly confidence combines warnings from all pipeline stages into a
per-postmark HIGH/MEDIUM/LOW rating.

In [58]:
# --- Step 8.10: Summary ---

print('=' * 60)
print('Step 8: Postmark Assembly -- Final Summary')
print('=' * 60)
print()
print(f'Input listings:           {len(listings)}')
print(f'Output postmarks:         {len(postmarks_df)}')
print(f'  (expansion from color fan-out: +{len(postmarks_df) - len(listings)})')
print()
print('Domain entity tables:')
print(f'  postmarks_df:           {len(postmarks_df)} rows')
print(f'  date_observed_df:       {len(date_observed_df)} rows')
print(f'  postmark_valuation_df:  {len(postmark_valuation_df)} rows')
print(f'  post_offices_df:        {len(post_offices_df)} rows')
print()
print('Value/lookup tables:')
print(f'  colors_df:              {len(colors_df)} rows')
print(f'  shapes_df:              {len(shapes_df)} rows')
print()
print('Confidence distribution:')
for level in ['HIGH', 'MEDIUM', 'LOW']:
    count = (postmarks_df['assembly_confidence'] == level).sum()
    pct = count / len(postmarks_df) * 100
    print(f'  {level}: {count} ({pct:.1f}%)')
print()

# FK integrity checks
print('FK integrity checks:')
orphan_colors = postmarks_df[
    postmarks_df['color_id'].notna() &
    ~postmarks_df['color_id'].isin(colors_df['color_id'])
]
print(f'  Postmarks with invalid color_id: {len(orphan_colors)}')

orphan_shapes = postmarks_df[
    postmarks_df['shape_id'].notna() &
    ~postmarks_df['shape_id'].isin(shapes_df['shape_id'])
]
print(f'  Postmarks with invalid shape_id: {len(orphan_shapes)}')

orphan_po = postmarks_df[
    postmarks_df['post_office_id'].notna() &
    ~postmarks_df['post_office_id'].isin(post_offices_df['post_office_id'])
]
print(f'  Postmarks with invalid post_office_id: {len(orphan_po)}')

orphan_dates = date_observed_df[
    ~date_observed_df['postmark_id'].isin(postmarks_df['postmark_id'])
] if len(date_observed_df) else pd.DataFrame()
print(f'  DateObserved with invalid postmark_id: {len(orphan_dates)}')

orphan_vals = postmark_valuation_df[
    ~postmark_valuation_df['postmark_id'].isin(postmarks_df['postmark_id'])
] if len(postmark_valuation_df) else pd.DataFrame()
print(f'  PostmarkValuation with invalid postmark_id: {len(orphan_vals)}')

Step 8: Postmark Assembly -- Final Summary

Input listings:           1537
Output postmarks:         1757
  (expansion from color fan-out: +220)

Domain entity tables:
  postmarks_df:           1757 rows
  date_observed_df:       1460 rows
  postmark_valuation_df:  1777 rows
  post_offices_df:        1206 rows

Value/lookup tables:
  colors_df:              16 rows
  shapes_df:              9 rows

Confidence distribution:
  HIGH: 422 (24.0%)
  MEDIUM: 1297 (73.8%)
  LOW: 38 (2.2%)

FK integrity checks:
  Postmarks with invalid color_id: 0
  Postmarks with invalid shape_id: 0
  Postmarks with invalid post_office_id: 0
  DateObserved with invalid postmark_id: 0
  PostmarkValuation with invalid postmark_id: 0


## 8.11 Sample Inspection

Spot-check a few assembled postmarks with their full dependency tree.

In [59]:
# --- Step 8.11: Sample inspection ---

def inspect_postmark(pm_id):
    """Print full assembled state for a single postmark."""
    pm = postmarks_df[postmarks_df['postmark_id'] == pm_id].iloc[0]
    print(f'=== Postmark {pm_id} ===')
    print(f'  catalog_text:    {pm["catalog_text"][:100]}')
    print(f'  inscription:     {pm["inscription_text"]}')
    print(f'  is_manuscript:   {pm["is_manuscript"]}')

    shape_name = '(null)'
    if pd.notna(pm['shape_id']):
        shape_name = shapes_df.loc[shapes_df['shape_id'] == pm['shape_id'], 'name'].iloc[0]
    print(f'  shape:           {shape_name}')

    color_name = pm.get('color_name', '(null)')
    print(f'  color:           {color_name}')
    print(f'  dimensions:      {pm["width"]}x{pm["height"]}')
    print(f'  date_fmt:     {pm["date_format"]}')
    print(f'  impression:      {pm["impression"]}')
    print(f'  is_irreg:    {pm["is_irregular"]}')

    po_name = '(null)'
    if pd.notna(pm['post_office_id']):
        po_name = post_offices_df.loc[
            post_offices_df['post_office_id'] == pm['post_office_id'], 'name'
        ].iloc[0]
    print(f'  post_office:     {po_name}')

    parent_pm = pm.get('parent_postmark_id')
    print(f'  parent_pm_id:    {parent_pm}')

    print(f'  confidence:      {pm["assembly_confidence"]}')
    if pm['s8_warnings']:
        print(f'  warnings:        {pm["s8_warnings"]}')

    # DateObserved
    dates = date_observed_df[date_observed_df['postmark_id'] == pm_id]
    print(f'  dates ({len(dates)}):')
    for _, d in dates.iterrows():
        print(f'    {d["date"]} ({d["granularity"]})')

    # Valuations
    vals = postmark_valuation_df[postmark_valuation_df['postmark_id'] == pm_id]
    print(f'  valuations ({len(vals)}):')
    for _, v in vals.iterrows():
        print(f'    pos {v["appraisal_position"]}: {v["amount"]}')
    print()


# Inspect first independent entry
inspect_postmark(1)

# Inspect a multi-color fan-out case (if any)
multi_color = postmarks_df[postmarks_df['is_multi_color_fanout']]
if len(multi_color):
    first_mc_src = multi_color['source_listing_idx'].iloc[0]
    siblings = postmarks_df[postmarks_df['source_listing_idx'] == first_mc_src]
    print(f'--- Multi-color siblings from listing idx {first_mc_src} ---')
    for _, sib in siblings.iterrows():
        inspect_postmark(sib['postmark_id'])

# Inspect a manuscript entry (if any)
ms_entries = postmarks_df[postmarks_df['is_manuscript']]
if len(ms_entries):
    inspect_postmark(ms_entries['postmark_id'].iloc[0])

# Inspect an inherited entry (child with parent_postmark_id)
inherited = postmarks_df[postmarks_df['parent_postmark_id'].notna()]
if len(inherited):
    sample = inherited.iloc[0]
    print(f'--- Inherited postmark (child -> parent) ---')
    inspect_postmark(int(sample['postmark_id']))
    print(f'  Parent postmark:')
    inspect_postmark(int(sample['parent_postmark_id']))

=== Postmark 1 ===
  catalog_text:    Alexa.(Alexandria)(E)(May 21,1772;Ms;Black) 1500.00
  inscription:     Alexa.
  is_manuscript:   True
  shape:           (null)
  color:           BLACK
  dimensions:      nanxnan
  date_fmt:     nan
  impression:      nan
  is_irreg:    None
  post_office:     ALEXA
  parent_pm_id:    nan
  confidence:      MEDIUM
  warnings:        ['shape_from_catalog_fallback']
  dates (1):
    1772-05-21 (DAY)
  valuations (1):
    pos 1: 1500.0

--- Multi-color siblings from listing idx 6 ---
=== Postmark 7 ===
  catalog_text:    FREDERICKSBURG("F"5mm high,used as bkstp) (March 1,1775;SL-50x3,MDD below;Red,Black) 1200.00
  inscription:     FREDERICKSBURG
  is_manuscript:   False
  shape:           SL - Straight Line
  color:           RED
  dimensions:      50.0x3.0
  date_fmt:     MDD
  impression:      Normal
  is_irreg:    False
  post_office:     FREDERICKSBURG
  parent_pm_id:    nan
  confidence:      MEDIUM
  warnings:        ['multi_color_fanout']
  da

# Step 9: Ratemark & Auxmark Assembly

Transforms parsed rate tokens from Step 6 into normalized Ratemark, Auxmark,
and PostmarkRatemark domain entities. Parallels Step 8 (Postmark Assembly).

### Token classification

Each rate token from `parsed_rates` is classified by what domain entities it
produces:

| Pattern | Produces | Examples |
|---------|----------|----------|
| Keyword only | Auxmark (parent=Postmark) | `FREE`, `PAID`, `STEAM` |
| Amount only | Ratemark | `5[C]`, `25[ms]`, `V` |
| Keyword + amount | Auxmark (parent=Ratemark) + Ratemark | `PAID/3[C]`, `DUE 3` |
| PM_FREE / PM_FRANK | Auxmark (parent=Postmark) | `P.M.Free`, `P.M.frank` |
| WITH_ADHESIVE | Skipped (Cover attribute, not a marking) | `with 24` |

### Bracket -> Shape resolution

Rate token brackets describe the physical form of the marking:
- `[C]` -> Circle, `[box]` -> Box, `[arc]` -> Arc, `[octagon]` -> Octagon
- `[ms]` -> manuscript flag (already consumed by Step 6 rate sub-parser)
- Compound brackets (e.g. `[cogged circle]`) resolved to a base shape

### Color inheritance

Ratemarks and Auxmarks inherit color from the parent Postmark. The ASCC color
field describes the entire entry's ink color, not individual markings.

### Output DataFrames

- `ratemarks_df` -- one row per Ratemark entity
- `auxmarks_df` -- one row per Auxmark entity
- `postmark_ratemark_df` -- junction linking Postmarks to Ratemarks

## 9.1 Rate Amount Parsing

Convert raw amount strings to numeric cents values. Handles ASCC fractional
notation (e.g. `12-1/2` = 12.5) and roman numerals.

In [60]:
# --- Step 9.1: Rate amount parsing ---

ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(s):
    """Convert a roman numeral string to integer. Returns None on failure."""
    if not s or not all(c in ROMAN_VALUES for c in s.upper()):
        return None
    total = 0
    prev = 0
    for ch in reversed(s.upper()):
        val = ROMAN_VALUES[ch]
        if val < prev:
            total -= val
        else:
            total += val
        prev = val
    return total


FRACTION_RE = re.compile(
    r'^(\d+)?'            # optional whole part
    r'[\-\s]*'            # optional separator
    r'(\d+)/(\d+)$'      # fraction
)


def parse_rate_amount(raw):
    """Parse a rate amount string to a float value in cents.
    Returns (numeric_value, is_roman) or (None, False) on failure.
    """
    if raw is None:
        return None, False

    s = str(raw).strip()
    if not s:
        return None, False

    # Roman numeral check
    if re.match(r'^[IVXLCDM]+$', s):
        val = roman_to_int(s)
        if val is not None:
            return float(val), True

    # Fractional: 12-1/2, 6-1/4, 1/2
    m = FRACTION_RE.match(s)
    if m:
        whole = int(m.group(1)) if m.group(1) else 0
        num = int(m.group(2))
        den = int(m.group(3))
        if den > 0:
            return float(whole) + num / den, False

    # Plain integer or decimal
    try:
        return float(s), False
    except ValueError:
        return None, False


# Self-tests
assert parse_rate_amount('5') == (5.0, False)
assert parse_rate_amount('25') == (25.0, False)
assert parse_rate_amount('12-1/2') == (12.5, False)
assert parse_rate_amount('6-1/4') == (6.25, False)
assert parse_rate_amount('1/2') == (0.5, False)
assert parse_rate_amount('V') == (5.0, True)
assert parse_rate_amount('X') == (10.0, True)
assert parse_rate_amount(None) == (None, False)
print('Rate amount parser self-tests passed')

Rate amount parser self-tests passed


## 9.2 Bracket Shape Resolution

Map bracket descriptors from rate tokens to Shape seed values. Compound
bracket content (e.g. `cogged circle`) is resolved to a base shape.


In [61]:
# --- Step 9.2: Bracket -> Shape resolution ---

# Direct bracket-to-shape mappings (case-insensitive)
BRACKET_SHAPE_MAP = {
    'c': 'C',
    'o': 'O',
    'box': 'BOX',
    'arc': 'ARC',
    'octagon': 'Octagon',
    'sl': 'SL',
    'rectangle': 'BOX',
    'oval': 'O',
    'circle': 'C',
}

# Pattern to extract optional dimensions from bracket content
BRACKET_DIM_RE = re.compile(r'(\d+\.?\d*)(?:\s*x\s*(\d+\.?\d*))?')


def resolve_bracket(bracket_text):
    """Resolve a bracket descriptor into shape/lettering/dimension components.
    Returns dict with keys: shape_name, lettering_name, width, height, qualifier.
    """
    if not bracket_text:
        return {'shape_name': None, 'lettering_name': None,
                'width': None, 'height': None, 'qualifier': None}

    text = bracket_text.strip()
    text_lower = text.lower()
    shape_name = None
    lettering_name = None
    width = None
    height = None
    qualifier = None

    # Direct shape match
    if shape_name is None:
        # Try the full text as a shape
        if text_lower in BRACKET_SHAPE_MAP:
            shape_name = BRACKET_SHAPE_MAP[text_lower]
        else:
            # Try each word against the shape map
            for word in text_lower.split():
                if word in BRACKET_SHAPE_MAP:
                    shape_name = BRACKET_SHAPE_MAP[word]
                    break

    # Lettering: check full text and individual words against lettering lookup
    if text_lower in lettering_lookup:
        lettering_name = text_lower
    else:
        for word in re.split(r'[\s,]+', text_lower):
            if word in lettering_lookup:
                lettering_name = word
                break

    # Extract dimensions from bracket content
    dim_m = BRACKET_DIM_RE.search(text)
    if dim_m:
        width = float(dim_m.group(1))
        if dim_m.group(2):
            height = float(dim_m.group(2))

    # Anything not recognized as shape/lettering/dimension is a qualifier
    if shape_name is None and lettering_name is None and width is None:
        qualifier = text

    return {
        'shape_name': shape_name,
        'lettering_name': lettering_name,
        'width': width,
        'height': height,
        'qualifier': qualifier,
    }


# Self-tests
assert resolve_bracket('C')['shape_name'] == 'C'
assert resolve_bracket('box')['shape_name'] == 'BOX'
assert resolve_bracket('arc')['shape_name'] == 'ARC'
assert resolve_bracket('octagon')['shape_name'] == 'Octagon'
r = resolve_bracket('cogged circle')
assert r['shape_name'] == 'C'
r = resolve_bracket('octagon 23')
assert r['shape_name'] == 'Octagon'
assert r['width'] == 23.0
r = resolve_bracket('30x23 rectangle')
assert r['shape_name'] == 'BOX'
assert r['width'] == 30.0
assert r['height'] == 23.0
assert resolve_bracket('hdstp rate')['qualifier'] == 'hdstp rate'
assert resolve_bracket(None)['shape_name'] is None
r = resolve_bracket('italics')
assert r['lettering_name'] == 'italics'
assert r['qualifier'] is None
r = resolve_bracket('cross hatched letters')
assert r['lettering_name'] is None  # no seed match
assert r['qualifier'] == 'cross hatched letters'
assert resolve_bracket('C')['lettering_name'] is None
print('Bracket resolver self-tests passed')

Bracket resolver self-tests passed


## 9.3 Token Classification & Entity Emission

Walk each listing's rate tokens and emit Ratemark, Auxmark, and
PostmarkRatemark rows. Classification logic:

1. Tokens with an amount produce a **Ratemark**.
2. Tokens with a keyword (and no amount) produce an **Auxmark** parented to the Postmark.
3. Tokens with both keyword and amount produce a **Ratemark** plus an **Auxmark**
   parented to that Ratemark.

Ratemarks (and their compound Auxmarks) are created once per fan-out postmark
so that each copy carries the correct color. Standalone Auxmarks (keyword only,
no amount) are likewise created once per fan-out postmark.

In [62]:
# --- Step 9.3: Token classification and entity emission ---

ratemark_rows = []
auxmark_rows = []
pm_rm_rows = []

next_rm_id = 1
next_aux_id = 1
next_pm_rm_id = 1

# Lookup tables for code generation
pm_code_lookup = postmarks_df.set_index('postmark_id')['code'].to_dict()
rm_counter_by_pm = {}      # {pm_id: next_rm_index}
aux_counter_by_parent = {} # {(parent_type, parent_id): next_aux_index}

# Group postmarks by source listing for fan-out handling
pm_by_listing = postmarks_df.groupby('source_listing_idx')['postmark_id'].apply(list).to_dict()

for listing_idx, row in listings.iterrows():
    postmark_ids = pm_by_listing.get(listing_idx, [])
    if not postmark_ids:
        continue

    # Flatten parsed_rates: list of lists -> flat list of tokens
    all_tokens = [tok for field_toks in row['parsed_rates'] for tok in field_toks]

    # Pre-fetch color for each postmark in this listing
    pm_color_map = {}
    for pm_id in postmark_ids:
        pm_color_map[pm_id] = postmarks_df.loc[
            postmarks_df['postmark_id'] == pm_id, 'color_id'
        ].iloc[0]

    for tok in all_tokens:
        kw = tok['rate_keyword']
        amt_raw = tok['rate_amount_raw']
        bracket = tok['rate_bracket']
        is_ms = tok['rate_is_manuscript']
        impression_override = tok.get('rate_impression')

        # Parse amount
        rate_value, is_roman = parse_rate_amount(amt_raw)
        has_amount = rate_value is not None

        # Resolve bracket -> shape/lettering
        br = resolve_bracket(bracket)
        bracket_shape_id = shape_lookup.get(br['shape_name'].upper()) if br['shape_name'] else None
        bracket_lettering_id = lettering_lookup.get(br['lettering_name']) if br['lettering_name'] else None

        # Determine impression
        if is_ms:
            mark_impression = None
        elif impression_override:
            mark_impression = impression_override
        else:
            mark_impression = 'Normal'

        # ── Tokens with an amount: emit Ratemark (+ compound Auxmark) per postmark ──
        if has_amount:
            for pm_id in postmark_ids:
                pm_color = pm_color_map[pm_id]

                # Inscription text
                if is_roman:
                    rm_inscription = amt_raw
                elif kw and amt_raw:
                    rm_inscription = amt_raw
                else:
                    rm_inscription = amt_raw or ''

                rm_id = next_rm_id
                pm_code = pm_code_lookup.get(pm_id)
                rm_idx = rm_counter_by_pm.get(pm_id, 0)
                rm_code = f'{pm_code}/RM{rm_idx}' if pm_code else None
                rm_counter_by_pm[pm_id] = rm_idx + 1
                ratemark_rows.append({
                    'ratemark_id': rm_id,
                    'inscription_text': rm_inscription,
                    'rate_value': rate_value,
                    'is_manuscript': is_ms,
                    'shape_id': None if is_ms else bracket_shape_id,
                    'lettering_id': None if is_ms else bracket_lettering_id,
                    'color_id': pm_color,
                    'width': br['width'] if not is_ms else None,
                    'height': br['height'] if not is_ms else None,
                    'is_irregular': None if is_ms else False,
                    'impression': mark_impression,
                    'source_listing_idx': listing_idx,
                    'rate_raw': tok['rate_raw'],
                    'bracket_qualifier': br.get('qualifier'),
                    'code': rm_code,
                })
                next_rm_id += 1

                # PostmarkRatemark junction
                pm_rm_rows.append({
                    'postmark_ratemark_id': next_pm_rm_id,
                    'postmark_id': pm_id,
                    'ratemark_id': rm_id,
                    'placement_type': None,
                })
                next_pm_rm_id += 1

                # Compound auxmark: keyword + amount -> parent to this ratemark
                if kw:
                    aux_inscription = kw
                    if kw in ('PM_FREE', 'PM_FRANK'):
                        aux_inscription = 'FREE' if kw == 'PM_FREE' else 'FRANK'

                    aux_parent_key = ('RATEMARK', rm_id)
                    aux_idx = aux_counter_by_parent.get(aux_parent_key, 0)
                    aux_code = f'{rm_code}/AM{aux_idx}' if rm_code else None
                    aux_counter_by_parent[aux_parent_key] = aux_idx + 1

                    auxmark_rows.append({
                        'auxmark_id': next_aux_id,
                        'inscription_text': aux_inscription,
                        'parent_mark_type': 'RATEMARK',
                        'parent_mark_id': rm_id,
                        'is_manuscript': False,
                        'shape_id': None if is_ms else bracket_shape_id,
                        'lettering_id': None if is_ms else bracket_lettering_id,
                        'color_id': pm_color,
                        'width': None,
                        'height': None,
                        'is_irregular': None if is_ms else False,
                        'impression': mark_impression,
                        'source_listing_idx': listing_idx,
                        'code': aux_code,
                    })
                    next_aux_id += 1

        # ── Standalone keyword (no amount): emit Auxmark per postmark ──
        elif kw:
            aux_inscription = kw
            if kw in ('PM_FREE', 'PM_FRANK'):
                aux_inscription = 'FREE' if kw == 'PM_FREE' else 'FRANK'

            for pm_id in postmark_ids:
                pm_color = pm_color_map[pm_id]

                pm_code_standalone = pm_code_lookup.get(pm_id)
                aux_parent_key_s = ('POSTMARK', pm_id)
                aux_idx_s = aux_counter_by_parent.get(aux_parent_key_s, 0)
                aux_code_s = f'{pm_code_standalone}/AM{aux_idx_s}' if pm_code_standalone else None
                aux_counter_by_parent[aux_parent_key_s] = aux_idx_s + 1

                auxmark_rows.append({
                    'auxmark_id': next_aux_id,
                    'inscription_text': aux_inscription,
                    'parent_mark_type': 'POSTMARK',
                    'parent_mark_id': pm_id,
                    'is_manuscript': False,
                    'shape_id': bracket_shape_id,
                    'lettering_id': bracket_lettering_id,
                    'color_id': pm_color,
                    'width': br['width'],
                    'height': br['height'],
                    'is_irregular': False,
                    'impression': mark_impression,
                    'source_listing_idx': listing_idx,
                    'code': aux_code_s,
                })

                next_aux_id += 1

print(f'Token classification complete')
print(f'  Ratemarks emitted: {len(ratemark_rows)}')
print(f'  Auxmarks emitted: {len(auxmark_rows)}')
print(f'  PostmarkRatemark junctions: {len(pm_rm_rows)}')

Token classification complete


  Ratemarks emitted: 663
  Auxmarks emitted: 586
  PostmarkRatemark junctions: 663


## 9.4 DataFrame Construction

Build DataFrames from the emitted rows.


In [63]:
# --- Step 9.4: DataFrame construction ---

ratemarks_df = pd.DataFrame(ratemark_rows) if ratemark_rows else pd.DataFrame(
    columns=['ratemark_id', 'inscription_text', 'rate_value', 'is_manuscript',
             'shape_id', 'lettering_id', 'color_id', 'width', 'height',
             'is_irregular', 'impression', 'source_listing_idx', 'rate_raw',
             'bracket_qualifier']
)

auxmarks_df = pd.DataFrame(auxmark_rows) if auxmark_rows else pd.DataFrame(
    columns=['auxmark_id', 'inscription_text', 'parent_mark_type',
             'parent_mark_id', 'is_manuscript', 'shape_id', 'lettering_id',
             'color_id', 'width', 'height', 'is_irregular', 'impression',
             'source_listing_idx']
)

postmark_ratemark_df = pd.DataFrame(pm_rm_rows) if pm_rm_rows else pd.DataFrame(
    columns=['postmark_ratemark_id', 'postmark_id', 'ratemark_id',
             'placement_type']
)

print('DataFrames constructed:')
print(f'  ratemarks_df:          {len(ratemarks_df)} rows')
print(f'  auxmarks_df:           {len(auxmarks_df)} rows')
print(f'  postmark_ratemark_df:  {len(postmark_ratemark_df)} rows')

DataFrames constructed:
  ratemarks_df:          663 rows
  auxmarks_df:           586 rows
  postmark_ratemark_df:  663 rows


## 9.5 Value Distributions

In [64]:
# --- Step 9.5: Value distributions ---

if len(ratemarks_df):
    print('Ratemark distributions:')
    print(f'  Total ratemarks: {len(ratemarks_df)}')
    print(f'  Manuscript ratemarks: {ratemarks_df["is_manuscript"].sum()}')
    print(f'  With shape: {ratemarks_df["shape_id"].notna().sum()}')
    print(f'  With bracket qualifier (unresolved): {ratemarks_df["bracket_qualifier"].notna().sum()}')
    print()

    # Rate value distribution
    rate_vals = ratemarks_df['rate_value'].dropna()
    if len(rate_vals):
        print(f'  Rate value range: {rate_vals.min():.1f} - {rate_vals.max():.1f} cents')
        print(f'  Most common values:')
        for val, count in rate_vals.value_counts().head(10).items():
            print(f'    {val:.1f}: {count}')
        print()

    # Shape distribution on ratemarks
    rm_shape_dist = ratemarks_df['shape_id'].value_counts(dropna=False)
    print('  Shape distribution:')
    for sid, count in rm_shape_dist.items():
        if pd.isna(sid):
            print(f'    (null): {count}')
        else:
            name = shapes_df.loc[shapes_df['shape_id'] == sid, 'name'].iloc[0]
            print(f'    {name}: {count}')
    print()

    # Lettering distribution on ratemarks
    rm_lettering_hits = ratemarks_df['lettering_id'].notna().sum()
    print(f'  Lettering assigned: {rm_lettering_hits}')
    if rm_lettering_hits:
        for lid, count in ratemarks_df['lettering_id'].value_counts(dropna=True).items():
            name = letterings_df.loc[letterings_df['lettering_id'] == lid, 'name'].iloc[0]
            print(f'    {name}: {count}')
    print()

if len(auxmarks_df):
    print('Auxmark distributions:')
    print(f'  Total auxmarks: {len(auxmarks_df)}')
    print(f'  Parented to Postmark: {(auxmarks_df["parent_mark_type"] == "Postmark").sum()}')
    print(f'  Parented to Ratemark: {(auxmarks_df["parent_mark_type"] == "Ratemark").sum()}')
    print()

    # Inscription text distribution
    print('  Inscription text distribution:')
    for text, count in auxmarks_df['inscription_text'].value_counts().items():
        print(f'    {text}: {count}')
    print()

    # Lettering distribution on auxmarks
    aux_lettering_hits = auxmarks_df['lettering_id'].notna().sum()
    print(f'  Lettering assigned: {aux_lettering_hits}')
    if aux_lettering_hits:
        for lid, count in auxmarks_df['lettering_id'].value_counts(dropna=True).items():
            name = letterings_df.loc[letterings_df['lettering_id'] == lid, 'name'].iloc[0]
            print(f'    {name}: {count}')
    print()

if len(postmark_ratemark_df):
    print('PostmarkRatemark junction:')
    print(f'  Total rows: {len(postmark_ratemark_df)}')
    print(f'  Distinct postmarks linked: {postmark_ratemark_df["postmark_id"].nunique()}')
    print(f'  Distinct ratemarks linked: {postmark_ratemark_df["ratemark_id"].nunique()}')
    rms_per_pm = postmark_ratemark_df.groupby('postmark_id').size()
    print(f'  Ratemarks per postmark: min={rms_per_pm.min()}, max={rms_per_pm.max()}, mean={rms_per_pm.mean():.1f}')

Ratemark distributions:
  Total ratemarks: 663
  Manuscript ratemarks: 0
  With shape: 10
  With bracket qualifier (unresolved): 20

  Rate value range: 1.0 - 50.0 cents
  Most common values:
    5.0: 275
    3.0: 212
    10.0: 122
    6.0: 12
    30.0: 7
    2.0: 7
    50.0: 5
    1.0: 5
    32.0: 4
    20.0: 4

  Shape distribution:
    (null): 653
    Box: 7
    Octagon: 3

  Lettering assigned: 3
    Outline: 3

Auxmark distributions:
  Total auxmarks: 586
  Parented to Postmark: 0
  Parented to Ratemark: 0

  Inscription text distribution:
    PAID: 453
    FREE: 118
    DUE: 12
    STEAM: 2
    FRANK: 1

  Lettering assigned: 0

PostmarkRatemark junction:
  Total rows: 663
  Distinct postmarks linked: 390
  Distinct ratemarks linked: 663
  Ratemarks per postmark: min=1, max=7, mean=1.7


## 9.6 FK Integrity Checks

In [65]:
# --- Step 9.6: FK integrity checks ---

print('Step 9 FK integrity checks:')

# Ratemark -> Shape
if len(ratemarks_df):
    orphan = ratemarks_df[
        ratemarks_df['shape_id'].notna() &
        ~ratemarks_df['shape_id'].isin(shapes_df['shape_id'])
    ]
    print(f'  Ratemarks with invalid shape_id: {len(orphan)}')

# Ratemark -> Color
if len(ratemarks_df):
    orphan = ratemarks_df[
        ratemarks_df['color_id'].notna() &
        ~ratemarks_df['color_id'].isin(colors_df['color_id'])
    ]
    print(f'  Ratemarks with invalid color_id: {len(orphan)}')

# Auxmark -> parent (Postmark or Ratemark)
if len(auxmarks_df):
    pm_auxmarks = auxmarks_df[auxmarks_df['parent_mark_type'] == 'Postmark']
    orphan_pm = pm_auxmarks[~pm_auxmarks['parent_mark_id'].isin(postmarks_df['postmark_id'])]
    print(f'  Auxmarks with invalid Postmark parent: {len(orphan_pm)}')

    rm_auxmarks = auxmarks_df[auxmarks_df['parent_mark_type'] == 'Ratemark']
    if len(ratemarks_df):
        orphan_rm = rm_auxmarks[~rm_auxmarks['parent_mark_id'].isin(ratemarks_df['ratemark_id'])]
    else:
        orphan_rm = rm_auxmarks
    print(f'  Auxmarks with invalid Ratemark parent: {len(orphan_rm)}')

# Auxmark -> Shape
if len(auxmarks_df):
    orphan = auxmarks_df[
        auxmarks_df['shape_id'].notna() &
        ~auxmarks_df['shape_id'].isin(shapes_df['shape_id'])
    ]
    print(f'  Auxmarks with invalid shape_id: {len(orphan)}')

# PostmarkRatemark -> Postmark
if len(postmark_ratemark_df):
    orphan = postmark_ratemark_df[
        ~postmark_ratemark_df['postmark_id'].isin(postmarks_df['postmark_id'])
    ]
    print(f'  PostmarkRatemark with invalid postmark_id: {len(orphan)}')

# PostmarkRatemark -> Ratemark
if len(postmark_ratemark_df) and len(ratemarks_df):
    orphan = postmark_ratemark_df[
        ~postmark_ratemark_df['ratemark_id'].isin(ratemarks_df['ratemark_id'])
    ]
    print(f'  PostmarkRatemark with invalid ratemark_id: {len(orphan)}')



Step 9 FK integrity checks:
  Ratemarks with invalid shape_id: 0
  Ratemarks with invalid color_id: 0
  Auxmarks with invalid Postmark parent: 0
  Auxmarks with invalid Ratemark parent: 0
  Auxmarks with invalid shape_id: 0
  PostmarkRatemark with invalid postmark_id: 0
  PostmarkRatemark with invalid ratemark_id: 0


## 9.7 Step 9 Summary

Step 9 transforms the parsed rate tokens from Step 6 into normalized Ratemark,
Auxmark, and PostmarkRatemark domain entities.

Ratemarks are created once per source listing and shared across color fan-out
postmarks via the PostmarkRatemark junction. Auxmarks parented to Ratemarks
are similarly created once. Standalone keyword auxmarks (e.g. PAID, FREE)
are duplicated per fan-out postmark since they directly reference a Postmark.

Key decisions:
- `WITH_ADHESIVE` tokens are not modeled as markings (they are Cover attributes)
- Bracket descriptors are resolved to Shape seed values where possible
- Color is inherited from the parent Postmark
- `placementType` on PostmarkRatemark is left null (not determinable from catalog text)

In [66]:
# --- Step 9.7: Summary ---

print('=' * 60)
print('Step 9: Ratemark & Auxmark Assembly -- Final Summary')
print('=' * 60)
print()
print(f'Input: {len(listings)} listings with {sum(len(t) for rlist in listings["parsed_rates"] for t in rlist)} rate tokens')
print()
print('New domain entity tables:')
print(f'  ratemarks_df:          {len(ratemarks_df)} rows')
print(f'  auxmarks_df:           {len(auxmarks_df)} rows')
print(f'  postmark_ratemark_df:  {len(postmark_ratemark_df)} rows')
print()

# Listings coverage
listings_with_rates = listings[listings['parsed_rates'].apply(
    lambda rlist: any(len(toks) > 0 for toks in rlist)
)].index
listings_with_ratemarks = set(ratemarks_df['source_listing_idx']) if len(ratemarks_df) else set()
listings_with_auxmarks = set(auxmarks_df['source_listing_idx']) if len(auxmarks_df) else set()
print(f'Listings with rate tokens: {len(listings_with_rates)}')
print(f'Listings producing ratemarks: {len(listings_with_ratemarks)}')
print(f'Listings producing auxmarks: {len(listings_with_auxmarks)}')
print(f'Listings with no rate/aux output: {len(listings) - len(listings_with_ratemarks | listings_with_auxmarks)}')

Step 9: Ratemark & Auxmark Assembly -- Final Summary

Input: 1537 listings with 641 rate tokens

New domain entity tables:
  ratemarks_df:          663 rows
  auxmarks_df:           586 rows
  postmark_ratemark_df:  663 rows

Listings with rate tokens: 366
Listings producing ratemarks: 268
Listings producing auxmarks: 299
Listings with no rate/aux output: 1173


## 9.8 Cover Assembly (Disabled) + DateSeen (Marking-Scoped)

The munger no longer auto-creates a Cover per listing. Users author Covers,
CoverMarkings, and CoverValuations by hand after the bundle is imported,
so `covers_df`, `cover_markings_df`, and `cover_valuations_df` ship empty
(header-only CSVs). The importer treats those three stems as optional.

DateSeen is still emitted -- the catalog's parsed dates are useful
observation data -- but now MARKING-scoped instead of COVER-scoped.
Anchoring: each parsed date on a listing produces one DateSeen row per
postmark of that listing. Ratemarks and auxmarks are not anchored
independently; in postal-history terms a ratemark's "date observed"
comes from the cover (or the parent postmark) it sits on, not from the
ratemark itself.

Output DataFrames:

- `covers_df`           -- empty (columns only)
- `cover_markings_df`   -- empty (columns only)
- `cover_valuations_df` -- empty (columns only)
- `dates_seen_df`       -- (marking_code, date, granularity); subject_type
                           is implicit 'MARKING' and is materialized in
                           Step 10 once marking ids are assigned.

In [67]:
# --- Step 9.8: Cover frames (empty) + DateSeen (MARKING-scoped) ---
# Policy change: the munger no longer auto-creates a Cover per listing.
# Users author Covers, CoverMarkings, and CoverValuations by hand after
# the bundle has been imported. Those three dataframes are kept as
# empty frames with the expected columns so the rest of the notebook
# (9.9 integrity, Step 10 CSV emit) still runs.
#
# DateSeen is still produced because the catalog's parsed_dates remain
# useful observation data; we just anchor each date to the listing's
# postmark(s) instead of to a synthetic Cover. dates_seen_df therefore
# now carries marking_code (not cover_code); Step 10 resolves it to the
# integer marking_id and stamps subject_type='MARKING'.
#
# Fanout rule: for each parsed_date on a listing, emit one DateSeen row
# per postmark of that listing. RANGE / DECADE granularities expand into
# one YEAR row per endpoint (start and end), matching the prior
# COVER-scoped logic.

from datetime import date as _date_cls


def _pm_codes_by_listing(frame):
    if frame is None or len(frame) == 0:
        return {}
    if "source_listing_idx" not in frame.columns or "code" not in frame.columns:
        return {}
    return frame.groupby("source_listing_idx")["code"].apply(list).to_dict()


_pm_codes_by_lst = _pm_codes_by_listing(postmarks_df)

ds_rows = []  # (marking_code, date, granularity)
for listing_idx, src in listings.iterrows():
    pm_codes = [c for c in _pm_codes_by_lst.get(listing_idx, []) if c]
    if not pm_codes:
        continue
    for d in (src.get("parsed_dates") or []):
        gran = d.get("date_granularity")
        obs_rows = []
        try:
            if gran == "DAY":
                obs = _date_cls(d["date_year_start"], d["date_month"], d["date_day"])
                obs_rows.append((str(obs), "DAY"))
            elif gran == "MONTH":
                obs = _date_cls(d["date_year_start"], d["date_month"], 1)
                obs_rows.append((str(obs), "MONTH"))
            elif gran == "YEAR":
                obs = _date_cls(d["date_year_start"], 1, 1)
                obs_rows.append((str(obs), "YEAR"))
            elif gran in ("RANGE", "DECADE"):
                for yr in (d["date_year_start"], d["date_year_end"]):
                    obs = _date_cls(int(yr), 1, 1)
                    obs_rows.append((str(obs), "YEAR"))
        except (ValueError, TypeError, KeyError):
            # Bad date components in source; Step 6 already reports parse errors.
            continue
        for obs_str, out_gran in obs_rows:
            for mc in pm_codes:
                ds_rows.append({
                    "marking_code": mc,
                    "date": obs_str,
                    "granularity": out_gran,
                })

covers_df = pd.DataFrame(
    columns=[
        "cover_code", "color_name", "type", "has_adhesive",
        "width", "height", "is_institutional",
    ]
)
cover_markings_df = pd.DataFrame(
    columns=["cover_code", "marking_code", "is_backstamp", "placement"]
)
cover_valuations_df = pd.DataFrame(
    columns=["cover_code", "amt", "appraisal_date"]
)
dates_seen_df = pd.DataFrame(ds_rows) if ds_rows else pd.DataFrame(
    columns=["marking_code", "date", "granularity"]
)

print("Cover auto-creation disabled. DateSeen anchored to postmarks (MARKING-scoped).")
print(f"  covers_df:           {len(covers_df)} rows")
print(f"  cover_markings_df:   {len(cover_markings_df)} rows")
print(f"  cover_valuations_df: {len(cover_valuations_df)} rows")
print(f"  dates_seen_df:       {len(dates_seen_df)} rows (subject_type=MARKING)")

Cover auto-creation disabled. DateSeen anchored to postmarks (MARKING-scoped).
  covers_df:           0 rows
  cover_markings_df:   0 rows
  cover_valuations_df: 0 rows
  dates_seen_df:       1460 rows (subject_type=MARKING)


## 9.9 Cover FK Integrity & Summary

In [68]:
# --- Step 9.9: Cover / DateSeen FK integrity checks ---
# After the cover-auto-creation policy change, cover_markings_df,
# covers_df, and cover_valuations_df are all empty by construction;
# their integrity is trivial. The interesting check is now DateSeen:
# every dates_seen_df row must reference a real marking code (postmark,
# ratemark, or auxmark) via marking_code.

# Build the universe of valid marking codes from the three marking frames.
all_marking_codes = set()
if postmarks_df is not None and len(postmarks_df) and "code" in postmarks_df.columns:
    all_marking_codes.update(c for c in postmarks_df["code"].dropna() if c)
if ratemarks_df is not None and len(ratemarks_df) and "code" in ratemarks_df.columns:
    all_marking_codes.update(c for c in ratemarks_df["code"].dropna() if c)
if auxmarks_df is not None and len(auxmarks_df) and "code" in auxmarks_df.columns:
    all_marking_codes.update(c for c in auxmarks_df["code"].dropna() if c)

# Cover-side frames must be empty under the new policy. Assert rather
# than warn: if any of these is non-empty something upstream has been
# resurrected accidentally.
assert len(covers_df) == 0, (
    f"covers_df should be empty (got {len(covers_df)} rows); cover "
    "auto-creation is disabled by policy."
)
assert len(cover_markings_df) == 0, (
    f"cover_markings_df should be empty (got {len(cover_markings_df)} rows)."
)
assert len(cover_valuations_df) == 0, (
    f"cover_valuations_df should be empty (got {len(cover_valuations_df)} rows)."
)

# DateSeen invariant: every MARKING-scoped row must reference a real
# marking_code.
if len(dates_seen_df):
    orphan_ds = dates_seen_df[~dates_seen_df["marking_code"].isin(all_marking_codes)]
    print(f"DateSeen with invalid marking_code: {len(orphan_ds)} (should be 0)")
    assert len(orphan_ds) == 0, (
        f"{len(orphan_ds)} DateSeen rows reference an unknown marking_code; "
        "first few: " + ", ".join(orphan_ds["marking_code"].head(5).tolist())
    )
else:
    print("DateSeen with invalid marking_code: 0 (frame empty)")

print()
print("Cover-assembly summary:")
print(f"  covers_df:           {len(covers_df)} rows (empty by policy)")
print(f"  cover_markings_df:   {len(cover_markings_df)} rows (empty by policy)")
print(f"  cover_valuations_df: {len(cover_valuations_df)} rows (empty by policy)")
print(f"  dates_seen_df:       {len(dates_seen_df)} rows (MARKING-scoped)")

DateSeen with invalid marking_code: 0 (should be 0)

Cover-assembly summary:
  covers_df:           0 rows (empty by policy)
  cover_markings_df:   0 rows (empty by policy)
  cover_valuations_df: 0 rows (empty by policy)
  dates_seen_df:       1460 rows (MARKING-scoped)


## 9.10 Sample Inspection

Spot-check assembled rate/aux entities and implied covers for representative catalog entries.

In [69]:
# --- Step 9.10: Sample inspection ---

def inspect_ratemark(rm_id):
    """Print full assembled state for a single ratemark."""
    rm = ratemarks_df[ratemarks_df['ratemark_id'] == rm_id].iloc[0]
    print(f'  === Ratemark {rm_id} ===')
    print(f'    inscription:   {rm["inscription_text"]}')
    print(f'    rate_value:    {rm["rate_value"]}')
    print(f'    is_manuscript: {rm["is_manuscript"]}')

    shape_name = '(null)'
    if pd.notna(rm.get('shape_id')):
        shape_name = shapes_df.loc[shapes_df['shape_id'] == rm['shape_id'], 'name'].iloc[0]
    print(f'    shape:         {shape_name}')

    color_name = '(null)'
    if pd.notna(rm.get('color_id')):
        color_name = colors_df.loc[colors_df['color_id'] == rm['color_id'], 'name'].iloc[0]
    print(f'    color:         {color_name}')
    print(f'    impression:    {rm["impression"]}')
    print(f'    raw:           {rm["rate_raw"]}')

    # Auxmarks parented to this ratemark
    child_aux = auxmarks_df[
        (auxmarks_df['parent_mark_type'] == 'RATEMARK') &
        (auxmarks_df['parent_mark_id'] == rm_id)
    ]
    if len(child_aux):
        for _, ax in child_aux.iterrows():
            aux_color = '(null)'
            if pd.notna(ax.get('color_id')):
                aux_color = colors_df.loc[colors_df['color_id'] == ax['color_id'], 'name'].iloc[0]
            print(f'    -> Auxmark {ax["auxmark_id"]}: {ax["inscription_text"]} [{aux_color}]')
    print()


def inspect_postmark_rates(pm_id):
    """Print all rate/aux associations for a postmark."""
    pm = postmarks_df[postmarks_df['postmark_id'] == pm_id].iloc[0]
    color_name = '(null)'
    if pd.notna(pm.get('color_id')):
        color_name = colors_df.loc[colors_df['color_id'] == pm['color_id'], 'name'].iloc[0]
    print(f'Postmark {pm_id}: {pm["catalog_text"][:100]}')
    print(f'  inscription: {pm["inscription_text"]}')
    print(f'  color:       {color_name}')
    print()

    # Linked ratemarks
    links = postmark_ratemark_df[postmark_ratemark_df['postmark_id'] == pm_id]
    if len(links):
        print(f'  Linked ratemarks ({len(links)}):')
        for _, lnk in links.iterrows():
            inspect_ratemark(lnk['ratemark_id'])
    else:
        print('  No linked ratemarks')

    # Direct auxmarks (parented to this postmark)
    direct_aux = auxmarks_df[
        (auxmarks_df['parent_mark_type'] == 'POSTMARK') &
        (auxmarks_df['parent_mark_id'] == pm_id)
    ]
    if len(direct_aux):
        print(f'  Direct auxmarks ({len(direct_aux)}):')
        for _, ax in direct_aux.iterrows():
            shape_name = '(null)'
            if pd.notna(ax.get('shape_id')):
                shape_name = shapes_df.loc[shapes_df['shape_id'] == ax['shape_id'], 'name'].iloc[0]
            aux_color = '(null)'
            if pd.notna(ax.get('color_id')):
                aux_color = colors_df.loc[colors_df['color_id'] == ax['color_id'], 'name'].iloc[0]
            print(f'    Auxmark {ax["auxmark_id"]}: {ax["inscription_text"]} [{shape_name}] [{aux_color}]')
    print()


# Find postmarks with rate tokens to inspect
# 1. A postmark with compound rate tokens (PAID + amount)
if len(postmark_ratemark_df):
    # Pick a postmark that has both ratemarks and direct auxmarks
    pms_with_rm = set(postmark_ratemark_df['postmark_id'])
    pms_with_aux = set(auxmarks_df.loc[
        auxmarks_df['parent_mark_type'] == 'POSTMARK', 'parent_mark_id'
    ]) if len(auxmarks_df) else set()
    both = pms_with_rm & pms_with_aux
    if both:
        sample_pm = sorted(both)[0]
        print('--- Sample: Postmark with both ratemarks and auxmarks ---')
        inspect_postmark_rates(sample_pm)

    # 2. A postmark with only auxmarks (keyword-only tokens)
    aux_only = pms_with_aux - pms_with_rm
    if aux_only:
        sample_pm = sorted(aux_only)[0]
        print('--- Sample: Postmark with auxmarks only ---')
        inspect_postmark_rates(sample_pm)

    # 3. A ratemark with compound bracket (if any exist)
    if len(ratemarks_df):
        bracket_rms = ratemarks_df[ratemarks_df['bracket_qualifier'].notna()]
        if len(bracket_rms):
            print('--- Sample: Ratemark with unresolved bracket qualifier ---')
            inspect_ratemark(bracket_rms['ratemark_id'].iloc[0])

# Implied-cover inspection removed: cover auto-creation is disabled (see 9.8).


--- Sample: Postmark with both ratemarks and auxmarks ---
Postmark 108: Same/Va.(1834-51;30;PAID,FREE,5,10;Red,Blue,Black) 20.00
  inscription: ABINGDON/Va.
  color:       RED

  Linked ratemarks (2):
  === Ratemark 8 ===
    inscription:   5
    rate_value:    5.0
    is_manuscript: False
    shape:         (null)
    color:         RED
    impression:    Normal
    raw:           5

  === Ratemark 11 ===
    inscription:   10
    rate_value:    10.0
    is_manuscript: False
    shape:         (null)
    color:         RED
    impression:    Normal
    raw:           10

  Direct auxmarks (2):
    Auxmark 5: PAID [(null)] [RED]
    Auxmark 8: FREE [(null)] [RED]

--- Sample: Postmark with auxmarks only ---
Postmark 99: WILLIAMS'B.G.(E)(July 2,1788;SL-34x2.5,MDD;PAID;Black) 750.00
  inscription: WILLIAMS'B.G.
  color:       BLACK

  No linked ratemarks
  Direct auxmarks (1):
    Auxmark 3: PAID [(null)] [BLACK]

--- Sample: Ratemark with unresolved bracket qualifier ---
  === Ratemark 

# Step 10: Output

Write all domain entity and seed tables to `./wip/out/` as CSV.
Intermediate/working DataFrames (`field_df`, `fanout_df`, `rate_mf_df`) are excluded.

In [70]:
# --- Step 10: Output (Django-shape CSVs for import_apmc_bundle) ---
# Emits twelve CSVs to OUT_DIR. Ten are generated from in-notebook frames
# in Django shape (explicit id, audit columns, integer FK columns).
# Two -- regions.csv and reference_works.csv -- are passed through verbatim
# from INPUT_DIR so the bundle is self-contained.
#
# Columns and types match the Resource declarations in
# backend/common/admin.py and the model fields in backend/common/models.py.
# After Phase 1 PK standardization, every Resource's import_id_fields is
# (or defaults to) ['id'], and every FK widget resolves the parent's id.
#
# DateSeen is polymorphic (subject_type, subject_id). The munger now
# emits MARKING-scoped rows (subject_type='MARKING'); subject_id is the
# postmark's resolved marking_id (see the dates_seen block below).

import os
import shutil

os.makedirs(OUT_DIR, exist_ok=True)

# ---- C1: Listing-order coverage assertion ---------------------------------
# Replaces the prior Postmark-Key-based assertion. The new check proves that
# every listing classified as a listing produced at least one marking row
# across postmarks_df / ratemarks_df / auxmarks_df.
_emitted_listing_idxs = set()
for _frame in (postmarks_df, ratemarks_df, auxmarks_df):
    if _frame is not None and len(_frame) and "source_listing_idx" in _frame.columns:
        _emitted_listing_idxs.update(_frame["source_listing_idx"].dropna().astype(int).tolist())

# Use the actual DataFrame index labels of listings, not range(len(listings)).
# Step 1 may classify some LISTING-type rows as non_entry (fragments, etc.),
# leaving gaps in the index. range(len) would include those gap values and
# incorrectly report them as un-emitted.
_expected = set(listings.index.tolist())
_missing = sorted(_expected - _emitted_listing_idxs)
if _missing:
    head = _missing[:20]
    raise AssertionError(
        f"Listing-coverage check failed: {len(_missing)} listing(s) produced no "
        f"marking rows. First {len(head)}: {head}"
        + (" ..." if len(_missing) > len(head) else "")
    )
print(f"Coverage check: all {len(listings)} listings emitted at least one marking.")

# ---- C2: Audit constants ---------------------------------------------------
AUDIT_USER_ID = 1
AUDIT_TS = pd.Timestamp.now(tz="UTC").isoformat(timespec="microseconds")

def _stamp(frame):
    out = frame.copy()
    out["created_date"]  = AUDIT_TS
    out["modified_date"] = AUDIT_TS
    out["created_by"]    = AUDIT_USER_ID
    out["modified_by"]   = AUDIT_USER_ID
    return out

# ---- C3: ID assignment -----------------------------------------------------
# Walk source listings in index order. For each listing, emit postmarks
# first, then ratemarks, then auxmarks, in their existing fan-out order.
# Each emitted row gets the next integer in the (single-region) marking
# serial. This produces marking codes of the form {RW_CODE}-{REGION_ABBREV}-N.

def _by_listing(frame, key):
    if frame is None or len(frame) == 0 or "source_listing_idx" not in frame.columns:
        return {}
    out = {}
    for _, r in frame.iterrows():
        out.setdefault(int(r["source_listing_idx"]), []).append(r[key])
    return out

pm_idx_by_listing = _by_listing(postmarks_df, "postmark_id") if (postmarks_df is not None and "postmark_id" in postmarks_df.columns) else {}
rm_idx_by_listing = _by_listing(ratemarks_df, "ratemark_id") if (ratemarks_df is not None and "ratemark_id" in ratemarks_df.columns) else {}
ax_idx_by_listing = _by_listing(auxmarks_df, "auxmark_id")  if (auxmarks_df  is not None and "auxmark_id"  in auxmarks_df.columns)  else {}

marking_id_by_pm = {}
marking_id_by_rm = {}
marking_id_by_ax = {}
emit_order = []  # list of ("PM"|"RM"|"AX", source_id, marking_id) in emission order

_next_marking_id = 1
for listing_idx in range(len(listings)):
    for pm_id in pm_idx_by_listing.get(listing_idx, []):
        marking_id_by_pm[pm_id] = _next_marking_id
        emit_order.append(("PM", pm_id, _next_marking_id))
        _next_marking_id += 1
    for rm_id in rm_idx_by_listing.get(listing_idx, []):
        marking_id_by_rm[rm_id] = _next_marking_id
        emit_order.append(("RM", rm_id, _next_marking_id))
        _next_marking_id += 1
    for ax_id in ax_idx_by_listing.get(listing_idx, []):
        marking_id_by_ax[ax_id] = _next_marking_id
        emit_order.append(("AX", ax_id, _next_marking_id))
        _next_marking_id += 1

print(f"Assigned marking ids: 1..{_next_marking_id - 1} ({len(emit_order)} markings)")

# Cover IDs: row order in covers_df.
_covers_src = covers_df.reset_index(drop=True).copy()
_covers_src.insert(0, "id", range(1, len(_covers_src) + 1))
cover_id_by_code = dict(zip(_covers_src["cover_code"], _covers_src["id"]))

# Post office IDs: row order in post_offices_df. The internal "post_office_id"
# column from the notebook becomes the lookup key for FK resolution downstream.
_po_src = post_offices_df.reset_index(drop=True).copy()
_po_src.insert(0, "id", range(1, len(_po_src) + 1))
po_id_by_internal = dict(zip(_po_src["post_office_id"], _po_src["id"]))

# Color / Lettering / Shape IDs: row order in their seed frames.
_colors_src = colors_df.reset_index(drop=True).copy()
_colors_src.insert(0, "id", range(1, len(_colors_src) + 1))
color_id_by_internal = dict(zip(_colors_src["color_id"], _colors_src["id"])) if "color_id" in _colors_src.columns else {}

_letterings_src = letterings_df.reset_index(drop=True).copy()
_letterings_src.insert(0, "id", range(1, len(_letterings_src) + 1))
lettering_id_by_internal = dict(zip(_letterings_src["lettering_id"], _letterings_src["id"])) if "lettering_id" in _letterings_src.columns else {}

_shapes_src = shapes_df.reset_index(drop=True).copy()
_shapes_src.insert(0, "id", range(1, len(_shapes_src) + 1))
shape_id_by_internal = dict(zip(_shapes_src["shape_id"], _shapes_src["id"])) if "shape_id" in _shapes_src.columns else {}

def _resolve_int_fk(lookup, internal_id):
    if internal_id is None or (isinstance(internal_id, float) and pd.isna(internal_id)):
        return None
    try:
        return lookup.get(internal_id) or lookup.get(int(internal_id))
    except (TypeError, ValueError):
        return None

# ---- C4: Build the 10 generated dataframes in Django shape -----------------

# colors
colors_out = pd.DataFrame({
    "id": _colors_src["id"],
    "name": _colors_src["name"],
    "hex_val": _colors_src["hex_val"] if "hex_val" in _colors_src.columns else None,
    "pantone_code": _colors_src["pantone_code"] if "pantone_code" in _colors_src.columns else None,
})
colors_out = _stamp(colors_out)

# letterings
letterings_out = pd.DataFrame({
    "id": _letterings_src["id"],
    "name": _letterings_src["name"],
})
letterings_out = _stamp(letterings_out)

# shapes (Phase 1 added the `code` column; emit empty string for now)
shapes_out = pd.DataFrame({
    "id": _shapes_src["id"],
    "name": _shapes_src["name"],
    "code": pd.NA,
})
shapes_out = _stamp(shapes_out)

# post_offices: PO rows no longer carry region inline; the link is
# emitted as post_office_regions below.
post_offices_out = pd.DataFrame({
    "id": _po_src["id"],
    "name": _po_src["name"],
})
post_offices_out = _stamp(post_offices_out)

# post_office_regions: one junction row per PostOffice, all pointing
# at the bundle's single REGION_ID (notebook is one-bundle-per-region).
# The exported "post_office" FK must reference the SEQUENTIAL id assigned
# above for post_offices_out, not the internal post_office_id from
# Step 8.8 -- map it through po_id_by_internal.
_por_src = post_office_regions_df.copy()
_por_src["post_office_export_id"] = _por_src["post_office_id"].map(po_id_by_internal)
_missing_por = _por_src["post_office_export_id"].isna().sum()
if _missing_por:
    raise ValueError(
        f"{_missing_por} post_office_regions rows reference an unknown "
        f"post_office_id; check Step 8.8 fan-out vs. Step 10 id assignment."
    )
post_office_regions_out = pd.DataFrame({
    "id": _por_src["post_office_region_id"].astype(int).values,
    "post_office": _por_src["post_office_export_id"].astype(int).values,
    "region": _por_src["region_id"].astype(int).values,
})
post_office_regions_out = _stamp(post_office_regions_out)

# markings: walk the same emit_order so id assignment matches the maps.
def _src_row_by(frame, key, internal_id):
    if frame is None or len(frame) == 0 or key not in frame.columns:
        return None
    sel = frame[frame[key] == internal_id]
    if len(sel) == 0:
        return None
    return sel.iloc[0]

marking_rows = []
for kind, src_id, mk_id in emit_order:
    if kind == "PM":
        r = _src_row_by(postmarks_df, "postmark_id", src_id)
        type_label = "TOWNMARK"
        rate_val = None
        catalog_txt = r.get("catalog_text") if r is not None else None
        date_fmt = r.get("date_format") if r is not None else None
    elif kind == "RM":
        r = _src_row_by(ratemarks_df, "ratemark_id", src_id)
        type_label = "RATEMARK"
        rate_val = r.get("rate_value") if r is not None else None
        catalog_txt = None
        date_fmt = None
    else:
        r = _src_row_by(auxmarks_df, "auxmark_id", src_id)
        type_label = "AUXMARK"
        rate_val = None
        catalog_txt = None
        date_fmt = None
    if r is None:
        continue
    src_idx = r.get("source_listing_idx")
    po_internal = r.get("post_office_id")
    if po_internal is None or (isinstance(po_internal, float) and pd.isna(po_internal)):
        # Fall back to the listing's postmark post_office (same convention as
        # the previous notebook). Pull from postmarks_df keyed by listing.
        if postmarks_df is not None and "post_office_id" in postmarks_df.columns:
            sel = postmarks_df[postmarks_df["source_listing_idx"] == src_idx]
            if len(sel):
                po_internal = sel.iloc[0]["post_office_id"]
    is_ms = bool(r.get("is_manuscript"))
    shape_int = r.get("shape_id")
    shape_id = _resolve_int_fk(shape_id_by_internal, shape_int)
    # Marking invariant: when is_manuscript is False, shape is required.
    # If no shape resolved, fall back to "SL - Straight Line" (default
    # handstamp form for text-only inscriptions without a bracket cue).
    if not is_ms and shape_id is None:
        _sl = _shapes_src[_shapes_src["name"] == "SL - Straight Line"]
        if len(_sl):
            shape_id = int(_sl.iloc[0]["id"])
    is_irreg_val = r.get("is_irregular")
    if not is_ms and (is_irreg_val is None or (isinstance(is_irreg_val, float) and pd.isna(is_irreg_val))):
        is_irreg_val = False
    marking_rows.append({
        "id": mk_id,
        "code": f"{RW_CODE}-{REGION_ABBREV}-{mk_id}",
        "type": type_label,
        "catalog_txt": catalog_txt,
        "inscription_txt": r.get("inscription_text"),
        "desc": None,
        "is_manuscript": is_ms,
        "shape": shape_id,
        "lettering": _resolve_int_fk(lettering_id_by_internal, r.get("lettering_id")),
        "color": _resolve_int_fk(color_id_by_internal, r.get("color_id")),
        "is_irreg": is_irreg_val,
        "width": r.get("width"),
        "height": r.get("height"),
        "date_fmt": date_fmt,
        "impression": r.get("impression"),
        "rate_val": rate_val,
        "post_office": _resolve_int_fk(po_id_by_internal, po_internal),
    })

markings_out = pd.DataFrame(marking_rows) if marking_rows else pd.DataFrame(columns=[
    "id", "code", "type", "catalog_txt", "inscription_txt", "desc", "is_manuscript",
    "shape", "lettering", "color", "is_irreg", "width", "height", "date_fmt",
    "impression", "rate_val", "post_office",
])
markings_out = _stamp(markings_out)

# covers
covers_out = pd.DataFrame({
    "id": _covers_src["id"],
    "code": _covers_src["cover_code"],
    "color": None,            # cover.color FK; the notebook does not currently resolve a per-cover color
    "type": _covers_src["type"] if "type" in _covers_src.columns else None,
    "has_adhesive": _covers_src["has_adhesive"] if "has_adhesive" in _covers_src.columns else False,
    "height": _covers_src["height"] if "height" in _covers_src.columns else None,
    "is_institutional": _covers_src["is_institutional"] if "is_institutional" in _covers_src.columns else None,
    "width": _covers_src["width"] if "width" in _covers_src.columns else None,
})
covers_out = _stamp(covers_out)

# dates_seen (polymorphic; MARKING-scoped under the no-auto-cover policy).
# Source frame carries marking_code (postmark code); we resolve to the
# final integer marking_id via marking_id_by_pm and stamp subject_type.
# The cell-121 fanout only ever emits postmark codes, so the lookup is
# postmark-only here.
_pm_code_to_mid = {}
if postmarks_df is not None and len(postmarks_df) and "postmark_id" in postmarks_df.columns and "code" in postmarks_df.columns:
    for _, _r in postmarks_df.iterrows():
        _mid = marking_id_by_pm.get(_r["postmark_id"])
        if _mid is not None:
            _pm_code_to_mid[_r["code"]] = _mid

_ds = dates_seen_df.copy() if dates_seen_df is not None else pd.DataFrame(columns=["marking_code", "date", "granularity"])
if len(_ds) and "marking_code" in _ds.columns:
    _ds["subject_id"] = _ds["marking_code"].map(_pm_code_to_mid)
    _ds = _ds[_ds["subject_id"].notna()].reset_index(drop=True)
    _ds["subject_id"] = _ds["subject_id"].astype(int)
    dates_seen_out = pd.DataFrame({
        "id": range(1, len(_ds) + 1),
        "subject_type": "MARKING",
        "subject_id": _ds["subject_id"],
        "date": _ds["date"],
        "granularity": _ds["granularity"],
    })
else:
    dates_seen_out = pd.DataFrame(columns=["id", "subject_type", "subject_id", "date", "granularity"])
dates_seen_out = _stamp(dates_seen_out)

# cover_valuations
_cv = cover_valuations_df.copy() if cover_valuations_df is not None else pd.DataFrame(columns=["cover_code", "amt", "appraisal_date"])
if len(_cv):
    _cv["cover"] = _cv["cover_code"].map(cover_id_by_code)
    _cv = _cv[_cv["cover"].notna()].reset_index(drop=True)
    _cv["cover"] = _cv["cover"].astype(int)
    cover_valuations_out = pd.DataFrame({
        "id": range(1, len(_cv) + 1),
        "cover": _cv["cover"],
        "amt": _cv["amt"],
        "appraisal_date": _cv["appraisal_date"],
    })
else:
    cover_valuations_out = pd.DataFrame(columns=["id", "cover", "amt", "appraisal_date"])
cover_valuations_out = _stamp(cover_valuations_out)

# cover_markings: marking_code -> marking_id via the per-type lookup dicts.
# Source frames carried marking_code in the {RW_CODE}-VA-N.0 / .1 / /RM / /AM
# legacy form; the new ids are looked up by the underlying postmark/rate/aux
# id captured in those source frames. Easier: rebuild from emit_order.
# For each cover, the prior notebook attached every marking from the listing,
# so we re-derive the (cover_id, marking_id) pairs directly.

# Map listing_idx -> cover_id by replaying cell 121's logic against the
# Django-shape covers we just built. cover_code already encodes the listing.
listing_to_cover_id = {}
# The notebook emits one cover per listing-with-valuations. Re-derive from
# covers_df rows: each cover_code has the cover.id we just assigned.
# We don't have listing_idx on covers_df directly, but cover_markings_df
# carries (cover_code, marking_code). We rebuild via the per-listing lookups.

cm_rows = []
_cm_id = 1
for listing_idx in range(len(listings)):
    # Find this listing's cover (if any) by walking covers_df: the prior
    # cell does not store listing_idx on covers_df, so we use the per-listing
    # marking sets to identify covers. Equivalent: cover exists iff at least
    # one cover_markings row references a marking from this listing.
    pass

# Cleaner: walk cover_markings_df directly. It already carries (cover_code,
# marking_code) where marking_code is the legacy code. The emission order in
# cell 121 attaches every postmark/ratemark/auxmark code from the listing, so
# we resolve marking_code -> marking_id by matching against the per-type code
# sets we can rebuild here.

legacy_pm_code_to_mid = {}
if postmarks_df is not None and len(postmarks_df) and "postmark_id" in postmarks_df.columns and "code" in postmarks_df.columns:
    for _, r in postmarks_df.iterrows():
        legacy_pm_code_to_mid[r["code"]] = marking_id_by_pm.get(r["postmark_id"])
legacy_rm_code_to_mid = {}
if ratemarks_df is not None and len(ratemarks_df) and "ratemark_id" in ratemarks_df.columns and "code" in ratemarks_df.columns:
    for _, r in ratemarks_df.iterrows():
        legacy_rm_code_to_mid[r["code"]] = marking_id_by_rm.get(r["ratemark_id"])
legacy_ax_code_to_mid = {}
if auxmarks_df is not None and len(auxmarks_df) and "auxmark_id" in auxmarks_df.columns and "code" in auxmarks_df.columns:
    for _, r in auxmarks_df.iterrows():
        legacy_ax_code_to_mid[r["code"]] = marking_id_by_ax.get(r["auxmark_id"])

def _resolve_marking_id(legacy_code):
    return (
        legacy_pm_code_to_mid.get(legacy_code)
        or legacy_rm_code_to_mid.get(legacy_code)
        or legacy_ax_code_to_mid.get(legacy_code)
    )

cm_out_rows = []
if cover_markings_df is not None and len(cover_markings_df):
    for _, r in cover_markings_df.iterrows():
        cover_id = cover_id_by_code.get(r["cover_code"])
        marking_id = _resolve_marking_id(r["marking_code"])
        if cover_id is None or marking_id is None:
            continue
        cm_out_rows.append({
            "cover": int(cover_id),
            "marking": int(marking_id),
            "is_backstamp": bool(r.get("is_backstamp", False)),
            "placement": r.get("placement"),
        })

cover_markings_out = pd.DataFrame(cm_out_rows) if cm_out_rows else pd.DataFrame(
    columns=["cover", "marking", "is_backstamp", "placement"]
)
cover_markings_out.insert(0, "id", range(1, len(cover_markings_out) + 1))
cover_markings_out = _stamp(cover_markings_out)

# citations: one row per emitted marking, in marking-id order.
cit_rows = []
for kind, src_id, mk_id in emit_order:
    if kind == "PM":
        r = _src_row_by(postmarks_df, "postmark_id", src_id)
    elif kind == "RM":
        r = _src_row_by(ratemarks_df, "ratemark_id", src_id)
    else:
        r = _src_row_by(auxmarks_df, "auxmark_id", src_id)
    if r is None:
        continue
    src_idx = r.get("source_listing_idx")
    page = listings.loc[int(src_idx), "Page"] if src_idx is not None else None
    if page is None or (isinstance(page, float) and pd.isna(page)):
        page_str = ""
    else:
        try:
            page_str = str(int(page))
        except (TypeError, ValueError):
            page_str = str(page).strip()
    cit_rows.append({
        "reference_work": RW_ID,
        "subject_type": "MARKING",
        "subject_id": mk_id,
        "citation_detail": page_str,
    })

citations_out = pd.DataFrame(cit_rows) if cit_rows else pd.DataFrame(
    columns=["reference_work", "subject_type", "subject_id", "citation_detail"]
)
citations_out.insert(0, "id", range(1, len(citations_out) + 1))
citations_out = _stamp(citations_out)

# ---- C5: Pass-through + emit ----------------------------------------------
# Order matches ASCC_LOAD_ORDER in import_ascc_bundle: dates_seen is loaded
# after cover_valuations and before cover_markings.
GENERATED = [
    ("colors",           colors_out,           ["id", "name", "hex_val", "pantone_code"]),
    ("letterings",       letterings_out,       ["id", "name"]),
    ("shapes",           shapes_out,           ["id", "name", "code"]),
    ("post_offices",         post_offices_out,         ["id", "name"]),
    ("post_office_regions",  post_office_regions_out,  ["id", "post_office", "region"]),
    ("markings",         markings_out,         [
                            "id", "code", "type", "catalog_txt", "inscription_txt",
                            "desc", "is_manuscript", "shape", "lettering", "color",
                            "is_irreg", "width", "height", "date_fmt", "impression",
                            "rate_val", "post_office",
                         ]),
    ("covers",           covers_out,           [
                            "id", "code", "color", "type", "has_adhesive",
                            "height", "is_institutional", "width",
                         ]),
    ("cover_valuations", cover_valuations_out, ["id", "cover", "amt", "appraisal_date"]),
    ("dates_seen",       dates_seen_out,       ["id", "subject_type", "subject_id", "date", "granularity"]),
    ("cover_markings",   cover_markings_out,   ["id", "cover", "marking", "is_backstamp", "placement"]),
    ("citations",        citations_out,        [
                            "id", "reference_work", "subject_type", "subject_id",
                            "citation_detail",
                         ]),
]

AUDIT_TAIL = ["created_date", "modified_date", "created_by", "modified_by"]

# Integer columns (id + FK columns + audit user FKs) per stem. Cast to
# pandas nullable Int64 so the CSV serializes "1" instead of "1.0" -- a
# pandas float column will be produced any time the source data has even
# one NaN, even when the column is conceptually an integer FK.
INT_COLS = {
    "colors":           ["id", "created_by", "modified_by"],
    "letterings":       ["id", "created_by", "modified_by"],
    "shapes":           ["id", "created_by", "modified_by"],
    "post_offices":         ["id", "created_by", "modified_by"],
    "post_office_regions":  ["id", "post_office", "region", "created_by", "modified_by"],
    "markings":         ["id", "shape", "lettering", "color", "post_office",
                         "created_by", "modified_by"],
    "covers":           ["id", "color", "created_by", "modified_by"],
    "cover_valuations": ["id", "cover", "created_by", "modified_by"],
    "dates_seen":       ["id", "subject_id", "created_by", "modified_by"],
    "cover_markings":   ["id", "cover", "marking", "created_by", "modified_by"],
    "citations":        ["id", "reference_work", "subject_id",
                         "created_by", "modified_by"],
}

def _cast_int_columns(frame, int_cols):
    """Cast each named column to pandas nullable Int64 in-place on a copy."""
    out = frame.copy()
    for c in int_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
    return out

for stem, frame, base_cols in GENERATED:
    cols = base_cols + AUDIT_TAIL
    out = frame[cols] if len(frame) else pd.DataFrame(columns=cols)
    out = _cast_int_columns(out, INT_COLS.get(stem, []))
    path = os.path.join(OUT_DIR, f"{stem}.csv")
    out.to_csv(path, index=False)
    print(f"  {stem + '.csv':<22s} {len(out):>5d} rows  ->  {path}")

# Pass-through CSVs (regions + reference_works) -- no transformation.
for stem in ("regions", "reference_works"):
    src = os.path.join(INPUT_DIR, f"{stem}.csv")
    dst = os.path.join(OUT_DIR, f"{stem}.csv")
    shutil.copyfile(src, dst)
    _row_count = sum(1 for _ in open(dst, "r", encoding="utf-8")) - 1
    print(f"  {stem + '.csv':<22s} {_row_count:>5d} rows  ->  {dst}  (passthrough)")

print(f"Wrote {len(GENERATED) + 3} tables to {OUT_DIR}")
print("Load via: python manage.py import_ascc_bundle " + OUT_DIR)

Coverage check: all 1537 listings emitted at least one marking.
Assigned marking ids: 1..3004 (3004 markings)


  colors.csv                16 rows  ->  ./wip/out/colors.csv
  letterings.csv             9 rows  ->  ./wip/out/letterings.csv
  shapes.csv                 9 rows  ->  ./wip/out/shapes.csv
  post_offices.csv        1206 rows  ->  ./wip/out/post_offices.csv
  post_office_regions.csv  1206 rows  ->  ./wip/out/post_office_regions.csv
  markings.csv            3004 rows  ->  ./wip/out/markings.csv
  covers.csv                 0 rows  ->  ./wip/out/covers.csv
  cover_valuations.csv       0 rows  ->  ./wip/out/cover_valuations.csv
  dates_seen.csv          1460 rows  ->  ./wip/out/dates_seen.csv
  cover_markings.csv         0 rows  ->  ./wip/out/cover_markings.csv
  citations.csv           3004 rows  ->  ./wip/out/citations.csv
  regions.csv               58 rows  ->  ./wip/out/regions.csv  (passthrough)
  reference_works.csv        1 rows  ->  ./wip/out/reference_works.csv  (passthrough)
Wrote 14 tables to ./wip/out/
Load via: python manage.py import_ascc_bundle ./wip/out/


# Step 11: Images Table Assembly

For each TOWNMARK (postmark) with `images_above > 0`, emit one `Image` row per
extracted marking PNG. Files must be present in `backend/media/<state>/` before
running this cell.

Image file naming convention from `tools/ascc_image_extract.py`:
`<state>-<page>-<chunk>-<counter>.png` (counter starts at 1).

Every image is labeled as a tracing via the explicit `is_tracing=True`
boolean on the `Image` model (see `backend/common/models.py` Image and
migration `0063_image_is_tracing`). `image_view` is set to `FULL` since
the legacy `COMPARISON` view choice was retired alongside the boolean.


In [71]:
# --- Step 11: Images table assembly ---
# Requires: PIL (pip install Pillow), marking_id_by_pm from Step 10 C3,
#           AUDIT_USER_ID / _stamp / _cast_int_columns / OUT_DIR from Step 10.
# Run AFTER cell 127 so postmark final marking ids are assigned.

import hashlib
import mimetypes
from pathlib import Path

from PIL import Image as PILImage

# Path to Django MEDIA_ROOT relative to the tools/ working directory.
MEDIA_ROOT = Path('../backend/media')
# Subdirectory under MEDIA_ROOT where extracted images were moved.
# Convention: lowercase state abbrev (matches the filename prefix).
IMAGES_SUBDIR = REGION_ABBREV.lower()  # e.g. 'va'

# pm_to_final_id: postmark_id -> final integer marking_id (assigned in Step 10 C3).
# marking_id_by_pm is built by the C3 block above.
pm_to_final_id = marking_id_by_pm

image_rows = []
next_image_id = 1

# For multi-color fan-out listings, multiple postmark rows share the same
# source (page, chunk). Only the first postmark per (page, chunk) receives
# image records to avoid duplicate Image rows for the same file.
seen_chunk_keys = set()

for _, pm in postmarks_df.sort_values('postmark_id').iterrows():
    n = int(pm.get('images_above') or 0)
    if n == 0:
        continue
    page = pm.get('page')
    chunk = pm.get('chunk')
    if page is None or chunk is None:
        print(f'WARNING: postmark_id={pm["postmark_id"]} has images_above={n} but '
              f'page={page!r} chunk={chunk!r}; skipping.')
        continue
    page = int(page)
    chunk = int(chunk)
    chunk_key = (page, chunk)
    if chunk_key in seen_chunk_keys:
        # Fan-out duplicate: images already emitted for this chunk.
        continue
    seen_chunk_keys.add(chunk_key)

    final_marking_id = pm_to_final_id.get(pm['postmark_id'])
    if final_marking_id is None:
        print(f'WARNING: no marking_id for postmark_id={pm["postmark_id"]}; skipping.')
        continue

    for counter in range(1, n + 1):
        fname = f'{IMAGES_SUBDIR}-{page}-{chunk}-{counter}.png'
        disk_path = MEDIA_ROOT / IMAGES_SUBDIR / fname
        if not disk_path.exists():
            raise FileNotFoundError(
                f'Missing image: {disk_path}  '
                f'(postmark_id={pm["postmark_id"]}, page={page}, chunk={chunk})'
            )
        data = disk_path.read_bytes()
        with PILImage.open(disk_path) as im:
            img_w, img_h = im.size
        image_rows.append({
            'image_id': next_image_id,
            'subject_type': 'MARKING',
            'subject_id': int(final_marking_id),
            'original_filename': fname,
            'storage_filename': f'{IMAGES_SUBDIR}/{fname}',
            'file_checksum': hashlib.sha256(data).hexdigest(),
            'mime_type': mimetypes.guess_type(fname)[0] or 'image/png',
            'image_width': img_w,
            'image_height': img_h,
            'file_size_bytes': len(data),
            # is_tracing is the canonical boolean on Image (see
            # backend/common/models.py and migration 0063_image_is_tracing).
            # The munger only ever emits trace/diagram extracts, so every
            # row is_tracing=True; image_view defaults to FULL since
            # COMPARISON was dropped from the choices in 0063.
            'image_view': 'FULL',
            'image_description': '',
            'is_tracing': True,
            'display_order': counter,
            'uploaded_by': AUDIT_USER_ID,
        })
        next_image_id += 1

_img_cols = [
    'image_id', 'subject_type', 'subject_id', 'original_filename',
    'storage_filename', 'file_checksum', 'mime_type', 'image_width',
    'image_height', 'file_size_bytes', 'image_view', 'image_description',
    'is_tracing', 'display_order', 'uploaded_by',
]
images_out = (
    pd.DataFrame(image_rows, columns=_img_cols)
    if image_rows
    else pd.DataFrame(columns=_img_cols)
)
images_out = _stamp(images_out)

# Coverage check: every (page, chunk) with images_above > 0 must have emitted
# exactly images_above image rows for that postmark's marking_id.
_img_counts = images_out.groupby('subject_id')['image_id'].count()
_check_pm = postmarks_df[
    postmarks_df['images_above'].notna() & (postmarks_df['images_above'] > 0)
]
_checked = set()
for _, pm in _check_pm.iterrows():
    chunk_key = (pm.get('page'), pm.get('chunk'))
    if chunk_key in _checked:
        continue
    _checked.add(chunk_key)
    mid = pm_to_final_id.get(pm['postmark_id'])
    if mid is None:
        continue
    expected = int(pm['images_above'])
    actual = int(_img_counts.get(mid, 0))
    if expected != actual:
        raise AssertionError(
            f'Image count mismatch for marking_id={mid} '
            f'(page={pm.get("page")}, chunk={pm.get("chunk")}): '
            f'expected {expected}, emitted {actual}'
        )

# Cast integer columns so the CSV serializes '1' instead of '1.0'.
_img_int_cols = [
    'image_id', 'subject_id', 'image_width', 'image_height',
    'file_size_bytes', 'display_order', 'uploaded_by',
    'created_by', 'modified_by',
]
images_out = _cast_int_columns(images_out, _img_int_cols)

_img_path = os.path.join(OUT_DIR, 'images.csv')
_img_emit_cols = _img_cols + ['created_date', 'modified_date', 'created_by', 'modified_by']
images_out[_img_emit_cols].to_csv(_img_path, index=False)
print(f'  {"images.csv":<22s} {len(images_out):>5d} rows  ->  {_img_path}')
print(f'  (tracing images: all, is_tracing=True, image_view=FULL)')


  images.csv               205 rows  ->  ./wip/out/images.csv
  (tracing images: all, is_tracing=True, image_view=FULL)
